# Giga Meter EDA Explorer

**Purpose:** country-parameterized EDA for Giga Meter measurement data — deployment, data
quality, connectivity performance, and **IQB-Edu educational-service-readiness verdicts** —
plus a full analytical appendix. The base for bespoke country analytics.

**How to use:** set the country in the Country cell, ensure the Trino tunnel is up
(`kubectl port-forward svc/trino 8080:8080 -n ictd-ooi-trino-prd`), then Run All.

---
## Layout
- **Part 0 — Setup & data** (imports, country, filters, loaders, field prep)
- **Part A — Core EDA** (exec summary + 7 core vectors; IQB-Edu readiness is the headline)
- **Part B — Appendix** (full deep-dive analysis)
- **Part C — Outputs** (exports, data dictionary)

---
## Part 0 — Setup & Data

In [ ]:
# =============================================================================
# IMPORTS
# =============================================================================

import os
import sys
import json
from pathlib import Path
from datetime import date, timedelta

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import pytz

from IPython.display import display

# Optional connectivity libs (only needed when USE_CACHED_DATA = False)
try:
    import delta_sharing
    DELTA_SHARING_AVAILABLE = True
except ImportError:
    DELTA_SHARING_AVAILABLE = False
    print("⚠️ delta_sharing not available - will use cached data only")

try:
    import trino
    from trino.dbapi import connect
    TRINO_AVAILABLE = True
except ImportError:
    TRINO_AVAILABLE = False
    print("⚠️ trino not available - will use cached data only")

# -----------------------------------------------------------------------------
# Data-loading helpers (bundled in ./helpers)
#   load_master            - school master via Delta Sharing / Trino, CSV-cached
#   load_measurements      - country measurements via Trino, parquet-cached + incremental
#   format_measurements    - query builder + light post-processing for the
#                            consolidated table default.all_gigameter_measurement_data
#   get_trino_cursor/engine - PRD Trino over kubectl port-forward (auto-started)
# -----------------------------------------------------------------------------
set_up_dir = Path.cwd() / "helpers"
if str(set_up_dir) not in sys.path:
    sys.path.insert(0, str(set_up_dir))

try:
    import format_measurements
    from load_master import load_master, load_master_trino
    from load_measurements import (
        load_measurements,
        load_registration,
        get_trino_cursor,
        get_trino_engine,
    )
    HELPERS_AVAILABLE = True
except ImportError as e:
    HELPERS_AVAILABLE = False
    print(f"⚠️ data-loading helpers not importable from {set_up_dir}: {e}")

# -----------------------------------------------------------------------------
# Analysis helpers + Giga chart style (moved out of the notebook)
#   eda_helpers      - education inference, ISP canonicalisation, IQB-Edu engine,
#                      legacy service-tier scaffolding
#   giga_chart_style - fonts + palette + rcParams (applied on import)
# -----------------------------------------------------------------------------
from eda_helpers import (resolve_country, infer_edlevel_from_name, clean_isp, build_isp_canon,
                         IQB_CONFIG, IQB_USE_CASES, IQB_PERCENTILES, IQB_BENCHMARK,
                         MIN_MEASUREMENTS_FOR_IQB, calculate_iqb_score,
                         _config_for_use_case,
                         classify_service_level, tier_order,
                         TIER_THRESHOLD_1, TIER_THRESHOLD_2, TIER_THRESHOLD_3,
                         paired_shift_test, two_group_shift_test,
                         format_shift_result, bootstrap_ci,
                         wilson_ci, fmt_pct_ci,
                         kruskal_omnibus, pairwise_shift_tests)
from giga_chart_style import (GIGA_PRIMARY, GIGA_GREY, GIGA_BLUE, GIGA_GOOD,
                              GIGA_MODERATE, GIGA_BAD, GIGA_TIER_RAMP, GIGA_CYCLE,
                              GIGA_SUPTITLE)

print("\u2713 Imports complete \u00b7 Giga chart style applied (Open Sans / Manrope, Giga palette)")


In [ ]:
# =============================================================================
# COUNTRY — set ONE code; iso2 / name / timezone resolve automatically
# =============================================================================
COUNTRY = "FJI"   # ISO3 code or country name (registry: helpers/country_reference.json)

_c = resolve_country(COUNTRY)   # single-timezone countries pick their zone via pytz;
                                # multi-zone countries use the curated registry default.
COUNTRY_ISO3, COUNTRY_ISO2, COUNTRY_NAME, TIMEZONE = _c["iso3"], _c["iso2"], _c["name"], _c["timezone"]
# Override for multi-zone countries if needed: resolve_country(COUNTRY, timezone="Asia/Samarkand")

# ── Data loading ─────────────────────────────────────────────────────────────
USE_CACHED_DATA = True        # True: load parquet caches; False: pull/refresh from Trino
MEASUREMENT_SOURCE = None     # rt_source filter; None = all sources (safe default)

# Scale knobs (large countries, e.g. UZB ~4.7M rows). The parquet on disk ALWAYS
# keeps full history and all columns; these only scope what gets LOADED.
ROWLEVEL_WINDOW_DAYS = None   # e.g. 365 -> only load the trailing year of row-level data
LOAD_COLUMNS = None           # e.g. a column list -> prune columns at read

CACHE_DIR = f"./cache/{COUNTRY_NAME}"   # parquet/CSV caches land here (gitignored)
from pathlib import Path as _P; _P(CACHE_DIR).mkdir(parents=True, exist_ok=True)
# Master data caches as {ISO3}_master_datapull.csv in this directory

print(f"\u2713 {COUNTRY_NAME} ({COUNTRY_ISO3}/{COUNTRY_ISO2}) \u00b7 timezone {TIMEZONE}"
      + (f"  [country spans {len(_c['timezones'])} zones]" if len(_c['timezones']) > 1 else ""))

In [ ]:
# =============================================================================
# NOTEBOOK-LEVEL FILTERS & ANALYSIS PARAMETERS (independent of country loading)
# =============================================================================
ADMIN1_FILTER = None              # e.g. "Eastern Cape" scopes the whole notebook to one region
SCHOOL_HOURS_START = 8            # school-hours window (local time, 24h)
SCHOOL_HOURS_END = 16
MIN_WEEKDAYS_MEASURED = 10        # min weekdays with data for detailed per-school analysis
USE_EDUCATION_INFERENCE = False   # infer education_level from school names if govt field incomplete

# NOTE: the latency outlier threshold is NOT set here — it is picked at the
# preprocessing stage, informed by this country's latency distribution.

# Output options
EXPORT_RESULTS = False            # True -> export summary tables to CSV
OUTPUT_DIR = "./output"

print(f"  Admin1 filter: {ADMIN1_FILTER if ADMIN1_FILTER else 'None (all regions)'} · "
      f"school hours {SCHOOL_HOURS_START}-{SCHOOL_HOURS_END} · "
      f"min weekdays {MIN_WEEKDAYS_MEASURED}")

In [ ]:
# Education-level normalization now lives in helpers (eda_helpers.infer_edlevel_from_name),
# imported in the IMPORTS cell below.
print("\u2713 Education level normalization: eda_helpers.infer_edlevel_from_name")

In [ ]:
# =============================================================================
# DATABASE CONNECTION (PRD Trino via kubectl port-forward)
# =============================================================================
# When USE_CACHED_DATA = False we query production Trino. The helpers below
# auto-start the tunnel if port 8080 isn't already open; you can also run it
# manually in a separate terminal and leave it open:
#
#     kubectl port-forward svc/trino 8080:8080 -n ictd-ooi-trino-prd
#
# Prereqs (one-time): az login -> az aks get-credentials --name uni-ooi-giga-aks-prd
#   --resource-group RS-UNI-GIGA-AKS-PRD -> kubelogin convert-kubeconfig -l azurecli
# If the tunnel fails to start, your az token has probably expired - re-run `az login`.
# =============================================================================
USE_CACHED_DATA = False

cur = None        # Trino DB-API cursor  -> used by load_measurements / refresh
engine = None     # SQLAlchemy engine    -> used by pd.read_sql for ad-hoc queries

if not USE_CACHED_DATA:
    cur = get_trino_cursor()      # auto-starts port-forward
    engine = get_trino_engine()   # auto-starts port-forward
    if cur is not None:
        print("✓ Trino PRD connection ready (catalog=delta_lake, schema=default)")
    else:
        print("⚠️ No Trino cursor - set USE_CACHED_DATA=True or fix the port-forward")
else:
    print("✓ Using cached data mode (no live Trino connection)")


In [ ]:
# =============================================================================
# LOAD MASTER DATA (load_master helper - Delta Sharing, CSV-cached)
# =============================================================================
# use_cached=True  -> read {ISO3}_master_datapull.csv from CACHE_DIR
# use_cached=False -> pull fresh from Delta Sharing and overwrite the cache
#                     (no port-forward needed - Delta Sharing is independent of Trino)

master_cache_csv = Path(CACHE_DIR) / f"{COUNTRY_ISO3}_master_datapull.csv"

master = load_master(COUNTRY_ISO3, master_cache_csv, use_cached=USE_CACHED_DATA)
master.head(3)


In [ ]:
# =============================================================================
# EDUCATION LEVEL NORMALIZATION
# =============================================================================

# Define inference function
def infer_edlevel_from_name(school_name):
    """Infer education level from school name patterns."""
    if pd.isna(school_name):
        return 'Unknown'
    
    name = str(school_name).upper()
    
    # Check for post-secondary
    if any(x in name for x in ['UNIVERSITY', 'COLLEGE', 'POLYTECHNIC', 'INSTITUTE']):
        return 'Post-secondary'
    
    # Check for secondary
    if any(x in name for x in ['SECONDARY', 'HIGH', 'TECHNICAL', 'VOCATIONAL']):
        return 'Secondary'
    
    # Check for primary and secondary combined
    if any(x in name for x in ['PRIMARY AND SECONDARY', 'PRIM. & SEC.', 'PRI. & SEC.']):
        return 'Primary and Secondary'
    
    # Check for primary
    if any(x in name for x in ['PRIMARY', 'BASIC', 'PRIM.', 'PRI.']):
        return 'Primary'
    
    # Check for pre-primary
    if any(x in name for x in ['PRE-PRIMARY', 'PRE-SCHOOL', 'ECE', 'NURSERY']):
        return 'Pre-primary'
    
    return 'Unknown'

print(f"\n{'='*80}")
print("EDUCATION LEVEL NORMALIZATION")
print(f"{'='*80}")

# Apply education level handling based on configuration
if USE_EDUCATION_INFERENCE:
    print(f"\nMode: Using inference from school names (education_level_inferred)")
    # Infer education level from school names
    master['education_level_inferred'] = master['school_name'].apply(infer_edlevel_from_name)
    
    # Prefer govt data where available, fall back to inferred
    master['education_level_normalized'] = master['education_level_govt'].fillna(master['education_level_inferred'])
    
    # Fill remaining nulls with 'Unknown'
    master['education_level_normalized'] = master['education_level_normalized'].fillna('Unknown')
    
    inferred_count = (master['education_level_normalized'] == master['education_level_inferred']).sum()
    govt_count = master['education_level_govt'].notna().sum()
    print(f"  Schools from govt data: {govt_count}")
    print(f"  Schools inferred from names: {inferred_count}")
    print(f"  Total schools: {len(master)}")
else:
    print(f"\nMode: Using government education level data only")
    # Just use govt data, fill nulls with Unknown
    master['education_level_normalized'] = master['education_level_govt'].fillna('Unknown')
    
    govt_count = master['education_level_govt'].notna().sum()
    unknown_count = (master['education_level_normalized'] == 'Unknown').sum()
    print(f"  Schools with govt data: {govt_count}")
    print(f"  Schools with unknown: {unknown_count}")
    print(f"  Total schools: {len(master)}")

# Display distribution
print(f"\nEducation Level Distribution:")
print(master['education_level_normalized'].value_counts().sort_index().to_string())


In [ ]:
# =============================================================================
# LOAD MEASUREMENTS (load_measurements helper - Trino, parquet-cached)
# =============================================================================
# Pulls from the consolidated table default.all_gigameter_measurement_data.
#   USE_CACHED_DATA = True   -> read the parquet cache (set in the CONFIG cell)
#   USE_CACHED_DATA = False  -> incremental refresh from Trino (delta since max cached date)
#   FORCE_REFRESH   = True   -> delete the parquet and pull EVERYTHING fresh from Trino
FORCE_REFRESH = False   # flip to True to wipe the cache and re-pull from scratch

if FORCE_REFRESH is True: 
    cur = None        # Trino DB-API cursor  -> used by load_measurements / refresh
    engine = None     # SQLAlchemy engine    -> used by pd.read_sql for ad-hoc queries

    cur = get_trino_cursor()      # auto-starts port-forward
    engine = get_trino_engine()   # auto-starts port-forward
    if cur is not None:
        print("✓ Trino PRD connection ready (catalog=delta_lake, schema=default)")
    else:
        print("⚠️ No Trino cursor - set USE_CACHED_DATA=True or fix the port-forward")
    
measurements_cache = Path(CACHE_DIR) / f"{COUNTRY_NAME.lower().replace(' ', '')}_measurements.parquet"

if FORCE_REFRESH and measurements_cache.exists():
    measurements_cache.unlink()
    print(f"\u2717 Deleted cache for full refresh: {measurements_cache.name}")

m = load_measurements(
    COUNTRY_NAME,
    measurements_cache,
    cur,
    use_cached=(USE_CACHED_DATA and not FORCE_REFRESH),   # FORCE_REFRESH always re-pulls
    source=MEASUREMENT_SOURCE,   # see CONFIG cell; None = all sources
    columns=globals().get('LOAD_COLUMNS'),            # scale knobs (CONFIG cell)
    window_days=globals().get('ROWLEVEL_WINDOW_DAYS'),
)
if globals().get('ROWLEVEL_WINDOW_DAYS'):
    print(f"NOTE: row-level data windowed to last {ROWLEVEL_WINDOW_DAYS} days - "
          f"full-history stats in this notebook reflect that window only.")

# Schema compatibility - the consolidated table's column set varies by source.
# GigaMeter rows use created_timestamp / isp_name / packet_loss_rate (no raw JSON);
# legacy DailyCheckApp rows also carry timestamp / detected_isp / results JSON.
# Alias the consolidated names to the legacy names the rest of the notebook expects.
if 'timestamp' not in m.columns and 'created_timestamp' in m.columns:
    m['timestamp'] = m['created_timestamp']
if 'detected_isp' not in m.columns and 'isp_name' in m.columns:
    m['detected_isp'] = m['isp_name']
if 'detected_isp_asn' not in m.columns and 'isp_asn' in m.columns:
    m['detected_isp_asn'] = m['isp_asn']

# Local-timezone conversion (downstream cells expect a tz-aware `timestamplocal`).
m['date'] = pd.to_datetime(m['date'])
m['timestamp'] = pd.to_datetime(m['timestamp'], utc=True)
m['timestamplocal'] = m['timestamp'].dt.tz_convert(TIMEZONE)

FILTER_LOG = {"loaded": len(m)}   # row-drop funnel, summarised at end of preprocessing

print(f"  Source(s): {m['rt_source'].value_counts().to_dict()}")
print(f"  Date range: {m['date'].min().date()} to {m['date'].max().date()}")
print(f"  Unique schools: {m['school_id_giga'].nunique()}")


In [ ]:
m.admin2.value_counts()

In [ ]:
# =============================================================================
# LOAD REGISTRATION DATA - direct SQL + parquet (notebook level)
# =============================================================================
# One row per school from delta_lake.default.all_gigameter_registered_schools
# (already joined with admin/geo metadata + funnel: registered_gigameter,
# sending_gigameter_data, install_status, first/last_measurement_date, device counts).

registration_cache = Path(CACHE_DIR) / f"{COUNTRY_NAME.lower().replace(' ', '')}_registered.parquet"

registered_query = f"""
SELECT *
FROM default.all_gigameter_registered_schools
WHERE iso3_code = '{COUNTRY_ISO3.upper()}'
"""

if USE_CACHED_DATA and registration_cache.exists():
    r = pd.read_parquet(registration_cache)
    print(f"\u2713 Registration loaded from cache: {r.shape[0]:,} rows  ({registration_cache.name})")
else:
    # run the SQL directly
    cur.execute(registered_query)
    r = pd.DataFrame(cur.fetchall(), columns=[d[0] for d in cur.description])
    print(f"\u2713 Registration queried: {r.shape[0]:,} rows")
    # save to parquet
    registration_cache.parent.mkdir(parents=True, exist_ok=True)
    r.to_parquet(registration_cache, index=False)
    print(f"\u2713 Cached to {registration_cache.name}")

# r already carries admin1/admin2 - no master merge needed.
if ADMIN1_FILTER:
    print("\n  Schools by admin1 (before filter):")
    print(r['admin1'].value_counts().head())
    r = r[r['admin1'] == ADMIN1_FILTER]
    print(f"\n\u2713 Filtered to {ADMIN1_FILTER}: {r.shape[0]} schools")

print(f"  registry lists {len(r):,} schools  "
      f"(status labels install_status/sending_gigameter_data NOT used — "
      f"canonical 'sent data'/'live' is derived from measurements)")


In [ ]:
# =============================================================================
# MEASUREMENT FIELD PREP (physical table is already unpacked - no JSON parsing)
# =============================================================================
# all_gigameter_measurement_data delivers WiFi, server, ISP and loss fields
# pre-extracted. We only coerce numeric dtypes and expose loss_rate.

# WiFi metrics - some arrive as object/string; coerce to numeric.
for c in ['detected_wifi_quality', 'detected_wifi_signal', 'detected_wifi_tx_rate',
          'detected_wifi_channel', 'detected_wifi_frequency']:
    if c in m.columns:
        m[c] = pd.to_numeric(m[c], errors='coerce')

m['loss_rate'] = pd.to_numeric(m['packet_loss_rate'], errors='coerce')

_wifi_n = m['detected_wifi_ssid'].notna().sum() if 'detected_wifi_ssid' in m.columns else 0
print(f"✓ WiFi info: {_wifi_n:,} measurements")
if 'detected_server' in m.columns:
    print(f"✓ Server (detected_server): {m['detected_server'].notna().sum():,} measurements")
print(f"✓ Packet loss: {m['loss_rate'].notna().sum():,} measurements "
      f"({100 * m['loss_rate'].notna().mean():.0f}% coverage, median {m['loss_rate'].median():.4%})")

In [ ]:
# =============================================================================
# ISP NAME NORMALISATION  ->  isp_mapped   (logic in eda_helpers.build_isp_canon)
# =============================================================================
# clean_isp() collapses near-duplicate strings (quotes, legal suffixes, spacing);
# per-country overrides come from isp_mappings.json ({ISO3: {canonical: [patterns]}})
# overlaid on the base dict. Extend the json per country as needed.
ISP_CANON, canon_isp, _isp_map_src = build_isp_canon(COUNTRY_ISO3)
if _isp_map_src:
    print(f"\u2713 Loaded per-country ISP mappings for {COUNTRY_ISO3} from {_isp_map_src}")

_isp_src = "detected_isp" if "detected_isp" in m.columns else "isp_name"
m["isp_mapped"] = m[_isp_src].map(canon_isp)
print(f"\u2713 isp_mapped: {m['isp_mapped'].nunique()} canonical ISPs (from {m[_isp_src].nunique()} raw variants)")
print(m["isp_mapped"].value_counts().head(10).to_string())

# Filter out ISPs with anomaly measurements

In [ ]:
# =============================================================================
# SERVER SELECTION — keep measurements against the MAIN test server only
# =============================================================================
# M-Lab routing can send tests to different servers over time; mixing servers
# mixes baseline latency (server distance), so downstream latency comparisons
# use the dominant server only. Rows with no server info are kept.
# NOTE: detected_server comes from a stale mlab-ns endpoint and is imperfect —
# treat it as a routing hint, not ground truth. Set SERVER_FILTER = False to keep all.
SERVER_FILTER = True

_srv = m['detected_server'].value_counts()
print("Measurements per detected server (median latency):")
for _s, _n in _srv.head(8).items():
    _lat = m.loc[m['detected_server'] == _s, 'latency'].median()
    print(f"  {str(_s)[:28]:28} {_n:>8,}   {_lat:5.0f} ms")
_no_srv = m['detected_server'].isna().sum()
print(f"  {'(no server info)':28} {_no_srv:>8,}")

MAIN_SERVER = _srv.idxmax() if len(_srv) else None
if SERVER_FILTER and MAIN_SERVER is not None:
    _b = len(m)
    m = m[(m['detected_server'] == MAIN_SERVER) | m['detected_server'].isna()]
    FILTER_LOG['other_servers'] = _b - len(m)
    print(f"\n✓ MAIN_SERVER = {MAIN_SERVER}: kept {len(m):,} rows "
          f"({_b - len(m):,} removed; no-server rows kept)")
else:
    FILTER_LOG['other_servers'] = 0
    print("\n(server filter off — all servers kept)")

In [ ]:
# =============================================================================
# MERGE MASTER METADATA INTO MEASUREMENTS
# =============================================================================
# The consolidated table already carries admin1/admin2/connectivity_type_govt/
# latitude/longitude/education_level. Only merge the master-derived columns that
# aren't already on `m`, to avoid _x/_y suffix collisions on re-merge. This works
# whether `m` came from the new 80+ col table or an older cached parquet.

_wanted = ['education_level', 'education_level_govt', 'connectivity_provider',
           'admin1', 'admin2', 'connectivity_type_govt', 'latitude', 'longitude']
_merge_cols = ['school_id_giga'] + [c for c in _wanted if c in master.columns and c not in m.columns]
m = m.merge(master[_merge_cols], on='school_id_giga', how='left')
print(f"✓ Merged master columns into m: {[c for c in _merge_cols if c != 'school_id_giga']}")


In [ ]:
# =============================================================================
# FILTER INVALID MEASUREMENTS (future-dated & other quality issues)
# =============================================================================
_n_before_invalid = len(m)

# Remove future-dated measurements (data quality issue - year 2247 etc)
tomorrow = (pd.Timestamp.now(tz='UTC') + pd.Timedelta(days=1)).normalize()
future_count = (m['timestamplocal'].dt.tz_convert('UTC') >= tomorrow).sum()

if future_count > 0:
    print(f"⚠️   Removing {future_count:,} future-dated measurements (data corruption):")
    future_examples = m[m['timestamplocal'].dt.tz_convert('UTC') >= tomorrow]['timestamplocal'].head(3)
    for ts in future_examples:
        print(f"      {ts}")
    m = m[m['timestamplocal'].dt.tz_convert('UTC') < tomorrow]
    print(f"✓ Kept {len(m):,} valid measurements")
else:
    print(f"✓ No future-dated measurements found")

# -----------------------------------------------------------------------------
# PHYSICAL-VALIDITY CLEANING (added 2026-07-30)
# Negative speeds/latency are impossible; latency around 4,294,967 ms
# (= 2^32 microseconds) is a uint32-overflow sentinel, not a measurement.
for _c in ['download_speed', 'upload_speed', 'latency']:
    if _c in m.columns:
        _v = pd.to_numeric(m[_c], errors='coerce')
        _bad = _v < 0
        if _c == 'latency':
            _bad |= _v >= 4_294_967
        if int(_bad.sum()):
            print(f"\u2713 Cleaned {_c}: {int(_bad.sum()):,} invalid values -> NaN")
        m[_c] = _v.mask(_bad)

FILTER_LOG['future_dated'] = _n_before_invalid - len(m)
# (physical-validity cleaning nulls invalid VALUES; it does not drop rows)


In [ ]:
# =============================================================================
# PRESERVE ORIGINAL MEASUREMENT DATA (clean, unfiltered)
# =============================================================================

# Create copy for drop-off/time-based analysis (after removing invalid timestamps, before time/admin filtering)
m_original = m.copy()

### Latency cutoff method comparison
Runs on the **unfiltered** distribution, before the threshold below is applied — IQR, modified z-score (MAD), percentiles, and per-connectivity-type variants.

In [ ]:
# =============================================================================
# LATENCY CUTOFF ANALYSIS - COMPARING METHODS
# =============================================================================

print("\n" + "="*80)
print("METHOD 1: INTERQUARTILE RANGE (IQR)")
print("="*80)

latency_q1 = m['latency'].quantile(0.25)
latency_q3 = m['latency'].quantile(0.75)
latency_iqr = latency_q3 - latency_q1
latency_iqr_threshold = latency_q3 + 1.5 * latency_iqr

iqr_outliers = (m['latency'] > latency_iqr_threshold).sum()
iqr_pct = iqr_outliers / len(m) * 100

print(f"Q1 (25th percentile): {latency_q1:>8.0f}ms")
print(f"Q3 (75th percentile): {latency_q3:>8.0f}ms")
print(f"IQR (Q3 - Q1):        {latency_iqr:>8.0f}ms")
print(f"\nThreshold (Q3 + 1.5×IQR): {latency_iqr_threshold:>8.0f}ms")
print(f"Outliers flagged: {iqr_outliers:,} ({iqr_pct:.2f}% of data)")

print("\n" + "="*80)
print("METHOD 2: MODIFIED Z-SCORE (MAD - Median Absolute Deviation)")
print("="*80)

latency_median = m['latency'].median()
latency_mad = (m['latency'] - latency_median).abs().median()   # NaN-skipping (runs pre-filter)
latency_modified_z_threshold = latency_median + 3.5 * latency_mad / 0.6745

mz_outliers = (m['latency'] > latency_modified_z_threshold).sum()
mz_pct = mz_outliers / len(m) * 100

print(f"Median latency:       {latency_median:>8.0f}ms")
print(f"MAD:                  {latency_mad:>8.0f}ms")
print(f"\nThreshold (median + 3.5×MAD/0.6745): {latency_modified_z_threshold:>8.0f}ms")
print(f"Outliers flagged: {mz_outliers:,} ({mz_pct:.2f}% of data)")

print("\n" + "="*80)
print("METHOD 3: PERCENTILE-BASED")
print("="*80)

p95_threshold = m['latency'].quantile(0.95)
p99_threshold = m['latency'].quantile(0.99)
p999_threshold = m['latency'].quantile(0.999)

p95_outliers = (m['latency'] > p95_threshold).sum()
p99_outliers = (m['latency'] > p99_threshold).sum()
p999_outliers = (m['latency'] > p999_threshold).sum()

print(f"95th percentile: {p95_threshold:>8.0f}ms → {p95_outliers:,} outliers ({p95_outliers/len(m)*100:.2f}%)")
print(f"99th percentile: {p99_threshold:>8.0f}ms → {p99_outliers:,} outliers ({p99_outliers/len(m)*100:.2f}%)")
print(f"99.9th percentile: {p999_threshold:>8.0f}ms → {p999_outliers:,} outliers ({p999_outliers/len(m)*100:.2f}%)")

print("\n" + "="*80)
print("METHOD 4: BY CONNECTIVITY TYPE (Modified Z-score per type)")
print("="*80)

type_thresholds = {}
for conn_type in m['connectivity_type_govt'].dropna().unique():
    subset = m[m['connectivity_type_govt'] == conn_type]['latency'].dropna()
    if len(subset) > 0:
        med = subset.median()
        mad = np.median(np.abs(subset - med))
        threshold = med + 3.5 * mad / 0.6745
        outliers = (subset > threshold).sum()
        pct = outliers / len(subset) * 100
        type_thresholds[conn_type] = threshold

        print(f"\n{conn_type} (n={len(subset):,}):")
        print(f"  Median: {med:>8.0f}ms")
        print(f"  Threshold: {threshold:>8.0f}ms")
        print(f"  Outliers: {outliers:,} ({pct:.2f}%)")

print("\n" + "="*80)
print("SUMMARY")
print("="*80)

summary_data = {
    'IQR (1.5×)': (latency_iqr_threshold, iqr_pct),
    'Modified Z-score (3.5×MAD)': (latency_modified_z_threshold, mz_pct),
    '95th percentile': (p95_threshold, p95_outliers/len(m)*100),
    '99th percentile': (p99_threshold, p99_outliers/len(m)*100),
}

print("\nComparison of methods (all data):")
for method, (threshold, pct) in summary_data.items():
    print(f"  {method:40} → {threshold:>8.0f}ms (filters {pct:>6.2f}%)")

# Visualize all thresholds
fig, ax = plt.subplots(figsize=(14, 6))

# Histogram of valid latency data (0-1000ms)
valid_latency = m[(m['latency'] > 0) & (m['latency'] < 1000)]['latency']
ax.hist(valid_latency.dropna(), bins=100, alpha=0.6, color='#0050e6', edgecolor='black', label='Valid latency data')

# Add threshold lines
thresholds_to_plot = [
    (latency_iqr_threshold, 'IQR (1.5×)', '#989898'),
    (latency_modified_z_threshold, 'Modified Z', '#002d9c'),
    (p99_threshold, '99th percentile', '#7eb0ff'),
]

for threshold, label, color in thresholds_to_plot:
    if threshold < 1000:  # Only plot if in visible range
        ax.axvline(threshold, color=color, linestyle='--', linewidth=2, label=f'{label}: {threshold:.0f}ms')

ax.set_xlabel('Latency (ms)', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_title('Latency Distribution with Candidate Outlier Thresholds', fontsize=14, fontweight='bold')
ax.set_xlim([0, 1000])
ax.legend(fontsize=11, loc='upper right')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# LATENCY OUTLIER THRESHOLD — inspect the distribution, then pick
# =============================================================================
# The preprocessing cell below applies this threshold globally. Default = p99
# for THIS country's distribution; review the histogram and candidates, then
# override the assignment at the bottom if a different cutoff fits better.
_lat = m['latency'].dropna()
_candidates = {'p95': _lat.quantile(0.95), 'p99': _lat.quantile(0.99),
               'p99.5': _lat.quantile(0.995),
               'IQR (Q3+1.5×IQR)': latency_iqr_threshold,        # from the method comparison above
               'mod-z (3.5×MAD)': latency_modified_z_threshold,  # from the method comparison above
               'fixed 400': 400, 'fixed 1000': 1000}

fig, ax = plt.subplots(figsize=(11, 3.5))
_plot = _lat[_lat <= _lat.quantile(0.999)]
ax.hist(_plot, bins=120, color=GIGA_PRIMARY[600], alpha=0.85)
for (_name, _v), _c in zip(_candidates.items(),
                           [GIGA_PRIMARY[300], GIGA_GREY[700], GIGA_PRIMARY[800],
                            GIGA_GOOD, GIGA_CYCLE[3], GIGA_MODERATE, GIGA_BAD]):
    if _v <= _plot.max():
        ax.axvline(_v, color=_c, linestyle='--', linewidth=1.3, label=f'{_name}: {_v:.0f} ms')
ax.set_yscale('log')
ax.set_xlabel('Latency (ms; display clipped at p99.9)')
ax.set_ylabel('measurements (log)')
ax.set_title(f'Latency distribution — pick the outlier cutoff — {COUNTRY_NAME}')
ax.legend(fontsize=8, ncol=3)
plt.tight_layout(); plt.show()

print('Candidate cutoffs and the share of measurements each would exclude:')
for _name, _v in _candidates.items():
    print(f"  {_name:>10}: {_v:7.0f} ms -> excludes {100 * (_lat > _v).mean():5.2f}%")

LATENCY_OUTLIER_THRESHOLD = round(float(_lat.quantile(0.99)))   # default: p99 — OVERRIDE after review
# LATENCY_OUTLIER_THRESHOLD = 

print(f"\nLATENCY_OUTLIER_THRESHOLD = {LATENCY_OUTLIER_THRESHOLD} ms (default = p99; edit this line to override)")

In [ ]:
# =============================================================================
# DATA PREPROCESSING
# =============================================================================
# Add measurement date and weekday columns
m['measurement_date'] = pd.to_datetime(m['timestamplocal']).dt.date
m['measurement_weekday'] = pd.to_datetime(m['timestamplocal']).dt.weekday  # 0=Monday

# Apply admin filter if specified
_n_before_admin = len(m)
if ADMIN1_FILTER:
    m = m[m.admin1 == ADMIN1_FILTER]
    print(f"✓ Measurements filtered to {ADMIN1_FILTER}: {m.shape[0]} records")
FILTER_LOG['admin1_filter'] = _n_before_admin - len(m)

# Filter latency outliers (threshold chosen above, from the distribution).
# Rows with MISSING latency are kept — they still carry valid speed tests.
_before = len(m)
m = m[(m['latency'] < LATENCY_OUTLIER_THRESHOLD) | m['latency'].isna()]
FILTER_LOG['latency_outliers'] = _before - len(m)
print(f"✓ Removed {_before - len(m):,} latency outliers (>={LATENCY_OUTLIER_THRESHOLD} ms; NaN latency kept)")

# Add time window classification
def classify_time_window(hour):
    if SCHOOL_HOURS_START <= hour <= SCHOOL_HOURS_END:
        return 'school_hours'
    return 'off_hours'

m['measurement_time_window'] = m['timestamplocal'].dt.hour.apply(classify_time_window)

# Canonical school-hours frame used by the exec summary and Part A/B analyses
# (previously defined in the removed SCHOOL ACTIVITY cell)
m_school = m[m['measurement_time_window'] == 'school_hours'].copy()
print(f"school-hours frame: {len(m_school):,} measurements")


In [ ]:
# =============================================================================
# TIME-OF-DAY PROFILE — inspect, then set the school-hours window (per-country)
# =============================================================================
# The preprocessing cell below applies SCHOOL_HOURS_START/END (from the filters
# cell) to classify school_hours vs off_hours — the cut behind per-school IQB
# and every "school hours" analysis (~79% of measurements in Fiji). Review the
# histogram and override the window HERE if this country's school day differs.
_hr = m['timestamplocal'].dt.hour

SCHOOL_HOURS_START, SCHOOL_HOURS_END = 7, 16   # <- override, then run on

fig, ax = plt.subplots(figsize=(11, 3.8))
ax.hist(_hr, bins=range(25), color=GIGA_PRIMARY[600], alpha=0.85, edgecolor='white')
ax.axvspan(SCHOOL_HOURS_START, SCHOOL_HOURS_END, color=GIGA_GOOD, alpha=0.12)
ax.axvline(SCHOOL_HOURS_START, color=GIGA_GREY[700], ls='--', lw=1.2)
ax.axvline(SCHOOL_HOURS_END, color=GIGA_GREY[700], ls='--', lw=1.2)
ax.set_xticks(range(0, 25, 2)); ax.set_xlabel('local hour'); ax.set_ylabel('measurements')
ax.set_title(f'Measurements by local time of day — {COUNTRY_NAME} '
             f'(shaded = school hours {SCHOOL_HOURS_START}-{SCHOOL_HOURS_END})')
plt.tight_layout(); plt.show()

_n_sch = _hr.between(SCHOOL_HOURS_START, SCHOOL_HOURS_END).sum()
print(f"Window {SCHOOL_HOURS_START}-{SCHOOL_HOURS_END} captures {_n_sch:,} of {len(m):,} "
      f"measurements ({100 * _n_sch / len(m):.0f}%) — applied by the preprocessing cell below.")

In [ ]:
# =============================================================================
# PREPROCESSING FUNNEL — what was filtered out of the analysis
# =============================================================================
_final = len(m)
_stages = [(k, v) for k, v in FILTER_LOG.items() if k != 'loaded']
print("=" * 64)
print("MEASUREMENTS FILTERED OUT OF THE ANALYSIS")
print("=" * 64)
print(f"  {'Loaded from cache/Trino':32s} {FILTER_LOG['loaded']:>10,}")
for _k, _v in _stages:
    print(f"  − {_k.replace('_', ' '):30s} {_v:>10,}   ({100 * _v / FILTER_LOG['loaded']:.2f}%)")
_dropped = FILTER_LOG['loaded'] - _final
print("-" * 64)
print(f"  {'ANALYSED':32s} {_final:>10,}   ({100 * _final / FILTER_LOG['loaded']:.1f}% of loaded; "
      f"{_dropped:,} rows removed)")
if 'measurement_time_window' in m.columns:
    _off = int((m['measurement_time_window'] != 'school_hours').sum())
    print(f"\n  Of the analysed rows, {_off:,} are OFF-HOURS ({100 * _off / _final:.0f}%) — kept in m,")
    print("  but excluded from every school-hours analysis (per-school IQB, performance")
    print(f"  distributions): those run on {_final - _off:,} school-hours measurements.")

print("\n  Note: physical-validity cleaning (negative speeds, latency overflow")
print("  sentinel) nulls VALUES without dropping rows, so it is not listed here.")

---
## IQB-Edu scoring engine

Educational-service readiness via the IQB-Edu method (ports `iqb.calculator` + `iqb.config` from the
IQB-Edu repo). For each use case (web browsing, video/audio streaming, video conferencing, online
backup, gaming), the network requirements (download / upload / latency / packet-loss) are weighted
and thresholded into a **continuous 0–1 score**. We compute the **overall IQB score and a per-use-case
score at p50 (typical) and p95 (best-case)** for every school.

Methodology (matches the IQB-Edu seed SQL): **school-hours measurements only**, percentiles over
**complete cases** (all metrics present), and the **polarity convention** — a label p{X} is the
*optimistic* case, so for latency & loss `p95` uses the raw 5th percentile (best), while throughput
`p95` uses the 95th (best). "Ready" for a use case = score ≥ `IQB_BENCHMARK` (0.7).


In [ ]:
# =============================================================================
# IQB-EDU SCORING ENGINE  (lives in eda_helpers; faithful port of the `iqb` package)
# =============================================================================
# IQB_CONFIG + calculate_iqb_score imported in the IMPORTS cell. Score is
# continuous 0..1 per use case, computed at multiple percentiles (p50, p95).
print(f"IQB engine ready: {len(IQB_USE_CASES)} use cases \u00b7 percentiles {IQB_PERCENTILES} \u00b7 benchmark {IQB_BENCHMARK}")

In [ ]:
# =============================================================================
# PER-SCHOOL IQB-EDU SCORES (overall + per use case, at p50 and p95)
# =============================================================================
# Methodology matches the IQB-Edu seed (iqbedu/queries/iqb_seed_by_school.sql):
#   - school-hours measurements only
#   - percentiles over COMPLETE cases (all IQB metrics present)
#   - POLARITY convention: label p{X} = optimistic. Throughput -> quantile X/100
#     (high = good); latency & loss -> quantile (100-X)/100 (low = good), so e.g.
#     latency_p95 = 5th percentile (best-case latency).
IQB_SCHOOL_HOURS_ONLY = True

_mi = m
if IQB_SCHOOL_HOURS_ONLY and "measurement_time_window" in m.columns:
    _mi = _mi[_mi["measurement_time_window"] == "school_hours"]
_need = ["download_speed", "upload_speed", "latency", "loss_rate"]
_mi = _mi.dropna(subset=_need)

_g = _mi.groupby("school_id_giga")
school_iqb = pd.DataFrame({"n": _g.size()}).reset_index()
school_iqb = school_iqb[school_iqb["n"] >= MIN_MEASUREMENTS_FOR_IQB].copy()

# (metric, source column, polarity) — polarity sets the quantile direction
_specs = [("download", "download_speed", "high_good"),
          ("upload",   "upload_speed",   "high_good"),
          ("latency",  "latency",        "low_good"),
          ("loss",     "loss_rate",      "low_good")]
def _qval(p, polarity):
    return p / 100.0 if polarity == "high_good" else 1 - p / 100.0
for p in IQB_PERCENTILES:
    cols = {f"{name}_p{p}": _g[col].quantile(_qval(p, pol)) for name, col, pol in _specs}
    school_iqb = school_iqb.merge(pd.DataFrame(cols).reset_index(), on="school_id_giga", how="left")

def _iqb_data(row, p):
    return {"m-lab": {
        "download_throughput_mbps": row[f"download_p{p}"],
        "upload_throughput_mbps":   row[f"upload_p{p}"],
        "latency_ms":               row[f"latency_p{p}"],
        "packet_loss":              row[f"loss_p{p}"]}}

_uc_cfg = {uc: _config_for_use_case(uc) for uc in IQB_USE_CASES}
for p in IQB_PERCENTILES:
    school_iqb[f"iqb_p{p}"] = school_iqb.apply(lambda r: calculate_iqb_score(_iqb_data(r, p)), axis=1)
    for uc in IQB_USE_CASES:
        col = f"iqb_{uc.replace(' ', '_')}_p{p}"
        school_iqb[col] = school_iqb.apply(lambda r: calculate_iqb_score(_iqb_data(r, p), _uc_cfg[uc]), axis=1)

_meta_cols = [c for c in ["admin1", "admin2", "school_area_type", "education_level", "connectivity_type_govt"] if c in m.columns]
school_iqb = school_iqb.merge(m[["school_id_giga"] + _meta_cols].drop_duplicates("school_id_giga"),
                              on="school_id_giga", how="left")

print(f"✓ IQB-Edu scored {len(school_iqb)} schools "
      f"(school-hours{' ' if IQB_SCHOOL_HOURS_ONLY else ' + off-hours '}only, >= {MIN_MEASUREMENTS_FOR_IQB} measurements)")
print(f"  overall median IQB: p50 {school_iqb['iqb_p50'].median():.2f}  |  p95 {school_iqb['iqb_p95'].median():.2f}")
display(school_iqb.head())


---
# Part A — Core EDA

## 1. Executive Summary

In [ ]:
# =============================================================================
# EXECUTIVE SUMMARY (computed headline metrics)
# =============================================================================
# Headline IQB percentile(s): 50 = typical school-day, 95 = best-case tail.
# Set to [50], [95], or [50, 95] to show one or both.
IQB_DISPLAY_PCTS = [50, 95]
assert all(p in IQB_PERCENTILES for p in IQB_DISPLAY_PCTS), f"choose from {IQB_PERCENTILES}"

# Deployment counts (self-contained — the former SCHOOL ACTIVITY cell was removed)
n_sent = int(r['first_measurement_date'].notna().sum()) if 'first_measurement_date' in r.columns \
         else m['school_id_giga'].nunique()
n_live = m.loc[m['date'].dt.to_period('M') == m['date'].max().to_period('M'), 'school_id_giga'].nunique()

print("=" * 80)
print(f"GIGA METER EDA  —  {COUNTRY_NAME} ({COUNTRY_ISO3})")
print("=" * 80)
print(f"  Window: {m['date'].min().date()} → {m['date'].max().date()}   |   measurements: {len(m):,}")
print()
print("  DEPLOYMENT")
print(f"    Sent data (historically) : {n_sent:,} schools")
print(f"    Sent data (this month)   : {n_live:,} schools  ({n_live/n_sent*100:.0f}% of senders)")
print()
print("  CONNECTIVITY (median, school hours)")
_loss_txt = f"{m_school['loss_rate'].median():.2%}" if m_school['loss_rate'].notna().any() else "pending"
print(f"    Download {m_school['download_speed'].median():.1f} Mbps   Upload {m_school['upload_speed'].median():.1f} Mbps   "
      f"Latency {m_school['latency'].median():.0f} ms   Loss {_loss_txt}")
print()
print(f"  IQB-EDU READINESS ({len(school_iqb)} schools)   score / % ready (>= {IQB_BENCHMARK}), by percentile")
for uc in IQB_USE_CASES:
    parts = []
    for p in IQB_DISPLAY_PCTS:
        col = f"iqb_{uc.replace(' ', '_')}_p{p}"
        parts.append(f"p{p} {school_iqb[col].mean():.2f}/{(school_iqb[col] >= IQB_BENCHMARK).mean()*100:.0f}%")
    print(f"    {uc:20s}  " + "   ".join(parts))
print(f"\n    Overall IQB    " + "   ".join(f"p{p} {school_iqb[f'iqb_p{p}'].mean():.2f}" for p in IQB_DISPLAY_PCTS))


## 2. Deployment & Adoption

In [ ]:
# =============================================================================
# CUMULATIVE SCHOOLS ONLINE OVER TIME
# =============================================================================
# The consolidated registration table has no app-install timestamp, so we use
# first_measurement_date (when a school first sent Giga Meter data) as the
# onboarding date. This tracks cumulative schools coming online.

r['first_measurement_date'] = pd.to_datetime(r['first_measurement_date'])
onboarded = r.dropna(subset=['first_measurement_date']).copy()
onboarded['onboard_day'] = onboarded['first_measurement_date'].dt.normalize()

cumulative_installs = (
    onboarded.drop_duplicates(['school_id_giga'])
     .sort_values('onboard_day')
     .groupby('onboard_day')['school_id_giga']
     .nunique()
     .cumsum()
     .reset_index(name='cumulative_schools')
)

plt.figure(figsize=(12, 5))
plt.fill_between(cumulative_installs['onboard_day'], cumulative_installs['cumulative_schools'],
                 color='#a9caff', alpha=0.8)
plt.plot(cumulative_installs['onboard_day'], cumulative_installs['cumulative_schools'], color='#277aff')

title = f"Schools with Giga Meter installed - {COUNTRY_NAME}"
if ADMIN1_FILTER:
    title = f"Schools with Giga Meter installed - {ADMIN1_FILTER}, {COUNTRY_NAME}"
plt.title(title)
plt.xlabel("Date of first measurement")
plt.ylabel("Cumulative schools online")
plt.grid(alpha=0.15)

# Current total annotation
latest_value = cumulative_installs['cumulative_schools'].iloc[-1]
plt.gca().text(0.05, 0.91, f"{latest_value}", fontsize=48, color='#161616', weight='bold',
               va='top', ha='left',
               bbox=dict(boxstyle="round,pad=0.3", fc="#eaf2ff", ec="#277aff", lw=2, alpha=0.85),
               transform=plt.gca().transAxes)

plt.tight_layout()
plt.show()

n_registered = int((r['registered_gigameter'] == 'Yes').sum()) if 'registered_gigameter' in r.columns else None
print(f"Schools that have sent Giga Meter data: {latest_value}")
if n_registered is not None:
    print(f"Schools registered for Giga Meter: {n_registered}")


In [ ]:
# =============================================================================
# MEASUREMENT FUNNEL — install -> measurement recency tiers (Giga Maps style)
# =============================================================================
# One row per MAPPED school in `r`; recency from days_since_last_measurement,
# as of the data pull date. The consolidated table lags ~1 day, so the freshest
# tier is "measured yesterday", not "today".
_d = pd.to_numeric(r['days_since_last_measurement'], errors='coerce')
_devs = pd.to_numeric(r.get('num_devices_registered'), errors='coerce').fillna(0)
funnel_stages = [
    ('All installed schools',     int((_devs > 0).sum())),
    ('Ever measured',             int(_d.notna().sum())),
    ('Measured in last year',     int((_d <= 365).sum())),
    ('Measured in last 6 months', int((_d <= 182).sum())),
    ('Measured in last 2 months',    int((_d <= 60).sum())),
    ('Measured in last month',    int((_d <= 30).sum())),
    ('Measured in last week',     int((_d <= 7).sum())),
    ('Measured yesterday',        int((_d <= 1).sum())),
]
labels = [l for l, _ in funnel_stages][::-1]
vals = [v for _, v in funnel_stages][::-1]
fig, ax = plt.subplots(figsize=(10, 4.8))
bars = ax.barh(labels, vals, color='#00d661', edgecolor='white')
for b, v in zip(bars, vals):
    ax.text(v + max(vals) * 0.012, b.get_y() + b.get_height() / 2, f'{v:,}',
            va='center', fontweight='bold')
ax.set_xlim(0, max(vals) * 1.12)
title = f'Measurement funnel — {COUNTRY_NAME}'
if ADMIN1_FILTER:
    title += f' — {ADMIN1_FILTER}'
ax.set_title(title, fontweight='bold')
plt.tight_layout(); plt.show()

print(f"Mapped schools in master (context): {r['school_id_giga'].nunique():,}")
if 'install_status' in r.columns:
    _st = r['install_status'].value_counts()
    print('install_status: ' + ' | '.join(f'{k} {v:,}' for k, v in _st.items())
          + "  ('Unknown' = never installed)")


In [ ]:
# =============================================================================
# SCHOOLS MEASURING PER MONTH — stacked by education level
# =============================================================================
_mm = m.dropna(subset=['school_id_giga']).copy()
_mm['month'] = _mm['date'].dt.to_period('M').dt.to_timestamp()
_mm['edu'] = _mm['education_level'].fillna('Unknown') if 'education_level' in _mm.columns else 'Unknown'
_tbl = _mm.groupby(['month', 'edu'])['school_id_giga'].nunique().unstack(fill_value=0)
_order = [c for c in ['Pre-Primary', 'Primary', 'Secondary'] if c in _tbl.columns] + \
         sorted(c for c in _tbl.columns if c not in ('Pre-Primary', 'Primary', 'Secondary'))
_tbl = _tbl[_order]

_palette = [GIGA_PRIMARY[300], GIGA_PRIMARY[600], GIGA_PRIMARY[900],
            GIGA_GREY[400], GIGA_GREY[600], GIGA_MODERATE, GIGA_GOOD] + GIGA_CYCLE
fig, ax = plt.subplots(figsize=(13, 5))
_tbl.plot(kind='bar', stacked=True, ax=ax, width=0.85, color=_palette[:len(_tbl.columns)])
ax.set_xticklabels([t.strftime('%Y-%m') for t in _tbl.index], rotation=60, ha='right', fontsize=8)
ax.set_ylabel('schools measuring')
ax.set_title(f'Schools measuring per month by education level — {COUNTRY_NAME}')
ax.legend(title=None, fontsize=9, ncol=2)
plt.tight_layout(); plt.show()

# Richest month — most schools AND most measurement days per school.
# school-days = sum over schools of distinct days measured (breadth x depth).
_mm['d'] = _mm['date'].dt.date
_msum = (_mm.drop_duplicates(['school_id_giga', 'd']).groupby('month')
            .agg(schools=('school_id_giga', 'nunique'), school_days=('school_id_giga', 'size')))
_msum['days_per_school'] = _msum['school_days'] / _msum['schools']
_b_sch, _b_vol, _b_dep = _msum['schools'].idxmax(), _msum['school_days'].idxmax(), _msum['days_per_school'].idxmax()
print(f"Most schools measuring:        {_b_sch:%Y-%m}  ({_msum.loc[_b_sch,'schools']:.0f} schools, "
      f"{_msum.loc[_b_sch,'days_per_school']:.1f} days/school)")
print(f"Richest month (school-days):   {_b_vol:%Y-%m}  ({_msum.loc[_b_vol,'school_days']:,.0f} school-days = "
      f"{_msum.loc[_b_vol,'schools']:.0f} schools x {_msum.loc[_b_vol,'days_per_school']:.1f} days each)")
print(f"Deepest per-school coverage:   {_b_dep:%Y-%m}  ({_msum.loc[_b_dep,'days_per_school']:.1f} days/school "
      f"across {_msum.loc[_b_dep,'schools']:.0f} schools)")

# Time series: days/school/month, each point annotated with n = # schools
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(_msum.index, _msum['days_per_school'], marker='o', ms=5, lw=1.8, color=GIGA_BLUE)
for _mo, _r in _msum.iterrows():
    ax.annotate(f"{_r['schools']:.0f}", (_mo, _r['days_per_school']),
                textcoords='offset points', xytext=(0, 7), ha='center',
                fontsize=7, color=GIGA_GREY[700])
ax.set_ylabel('days / school / month')
ax.set_ylim(0, _msum['days_per_school'].max() * 1.2)
ax.set_title(f'Measurement days per school per month — {COUNTRY_NAME} (labels = schools measuring)')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.show()

# Measurement type per day (notes field: daily / startup / manual / first ...)
_nt = m.dropna(subset=['notes']).copy()
_nt['notes_clean'] = _nt['notes'].replace('', 'unlabelled')
_dtbl = _nt.groupby([_nt['date'].dt.date, 'notes_clean']).size().unstack(fill_value=0)
_dtbl = _dtbl[_dtbl.sum().sort_values(ascending=False).index]   # biggest type at the bottom
fig, ax = plt.subplots(figsize=(13, 4))
_x = np.arange(len(_dtbl)); _bottom = np.zeros(len(_dtbl))
for _col, _c in zip(_dtbl.columns, _palette):
    ax.bar(_x, _dtbl[_col].values, bottom=_bottom, width=1.0, label=_col, color=_c, linewidth=0)
    _bottom += _dtbl[_col].values
_ticks = [i for i, d in enumerate(_dtbl.index) if d.day == 1][::2]
ax.set_xticks(_ticks)
ax.set_xticklabels([_dtbl.index[i].strftime('%Y-%m') for i in _ticks], rotation=60, ha='right', fontsize=8)
ax.set_ylabel('measurements / day')
ax.set_title(f'Measurements per day by type (notes) — {COUNTRY_NAME}')
ax.legend(fontsize=9, ncol=2)
plt.tight_layout(); plt.show()

# Average measurements/day by type — past year vs past month
_ref = m['date'].max()
for _label, _days in (('past year', 365), ('past month', 30)):
    _win = _nt[_nt['date'] >= _ref - pd.Timedelta(days=_days)]
    _ndays = _win['date'].dt.date.nunique()
    if _ndays == 0:
        print(f"\n({_label}: no labelled measurements)"); continue
    _avg = (_win.groupby('notes_clean').size() / _ndays).sort_values(ascending=False)
    print(f"\nAvg measurements/day by type — {_label} ({_ndays} active days):")
    for _t, _v in _avg.items():
        print(f"  {_t:12s} {_v:7.1f}/day  ({100 * _v / _avg.sum():.0f}%)")

## 3. Data Quality

In [ ]:
# =============================================================================
# FAIL REASONS DEEP-DIVE — do invalid tests concentrate in schools or ISPs?
# =============================================================================
# pass_fail_overall is a DATA-VALIDITY flag (NDT7 test reliability), not a
# quality verdict. If failures cluster in specific schools/ISPs, those places
# are systematically under-measured wherever failed tests get excluded.
_dq = m.copy()
_dq['pf'] = _dq['pass_fail_overall'].astype('string').str.lower()
_known = _dq[_dq['pf'].isin(['pass', 'fail'])].copy()
_known['is_fail'] = _known['pf'] == 'fail'
_overall_fr = _known['is_fail'].mean()
print(f"Validity flag: pass {(_dq['pf'] == 'pass').sum():,} | fail {(_dq['pf'] == 'fail').sum():,} "
      f"| not computed {(~_dq['pf'].isin(['pass', 'fail'])).sum():,}")
print(f"Overall fail rate (where computed): {100 * _overall_fr:.1f}%")

# 1. Reasons (a failed test can carry several)
_reasons = (_known.loc[_known['is_fail'], 'reasons_failed_overall'].dropna()
            .str.split(', ').explode().str.strip())
print("\nFail reasons:")
for _r, _n in _reasons.value_counts().head(8).items():
    print(f"  {_r[:58]:58s} {_n:>7,}")

# 2. Schools — volume concentration + highest rates
_sch = (_known.groupby(['school_id_giga', 'school_name'])
        .agg(n=('is_fail', 'size'), fails=('is_fail', 'sum'),
             primary_isp=('isp_mapped', lambda s: s.mode().iloc[0] if s.notna().any() else '?'))
        .reset_index())
_sch['fail_perc'] = 100 * _sch['fails'] / _sch['n']
_top10_share = _sch.nlargest(10, 'fails')['fails'].sum() / max(_sch['fails'].sum(), 1)
print(f"\nSchool concentration: the top 10 schools by fail volume hold {100 * _top10_share:.0f}% "
      f"of all {_sch['fails'].sum():,} fails ({(_sch['fails'] > 0).sum()} schools have any)")
print("Highest fail-RATE schools (n >= 50), with their primary ISP:")
display(_sch[_sch['n'] >= 50].sort_values('fail_perc', ascending=False)
        .head(10)[['school_name', 'primary_isp', 'n', 'fails', 'fail_perc']].round({'fail_perc': 1}))

# 3. ISPs — fail rate vs the overall rate
_isp = (_known.dropna(subset=['isp_mapped']).groupby('isp_mapped')
        .agg(n=('is_fail', 'size'), schools=('school_id_giga', 'nunique'), fails=('is_fail', 'sum'))
        .reset_index())
_isp['fail_perc'] = 100 * _isp['fails'] / _isp['n']
_isp['vs_overall_pp'] = _isp['fail_perc'] - 100 * _overall_fr
print(f"\nFail rate by ISP (n >= 500), overall = {100 * _overall_fr:.1f}%:")
display(_isp[_isp['n'] >= 500].sort_values('fail_perc', ascending=False)
        .round({'fail_perc': 1, 'vs_overall_pp': 1}))

# 4. Triangulation — school effect vs ISP effect
_hi = _sch[(_sch['n'] >= 50) & (_sch['fail_perc'] >= 200 * _overall_fr)]
print(f"\nTriangulation: {len(_hi)} schools fail at >= 2x the overall rate (n >= 50).")
print("Their primary-ISP mix vs all schools:")
_mix = _hi['primary_isp'].value_counts(normalize=True).round(2)
_all_mix = _sch['primary_isp'].value_counts(normalize=True).round(2)
for _i in _mix.index[:6]:
    print(f"  {str(_i)[:24]:24s} {100 * _mix[_i]:4.0f}% of high-fail schools vs {100 * _all_mix.get(_i, 0):4.0f}% of all schools")
print("-> If the mixes match, failures are school-local (device/setup); a large"
      "\n   over-representation points at the ISP's network instead.")

# 5. Fail reasons by ISP — does the failure MODE differ by provider?
_fr = _known[_known['is_fail']].dropna(subset=['isp_mapped']).copy()
_fr = _fr.assign(reason=_fr['reasons_failed_overall'].str.split(', ')).explode('reason')
_fr['reason'] = _fr['reason'].str.strip()
_big = set(_isp.loc[_isp['n'] >= 500, 'isp_mapped'])
_ct = (pd.crosstab(_fr['isp_mapped'], _fr['reason'], normalize='index') * 100)
_ct = _ct.loc[_ct.index.isin(_big)]
_ct = _ct[_ct.mean().sort_values(ascending=False).index].round(0).astype(int)
print("\nFail-reason mix by ISP (% of the ISP's reason mentions):")
display(_ct)

## 4. Connectivity Performance

In [ ]:
# =============================================================================
# CONNECTIVITY PERFORMANCE DISTRIBUTIONS (school hours)
# =============================================================================
_src = m_school   # school-hours subset, consistent with IQB scoring
_metrics = [("download_speed", "Download (Mbps)", GIGA_PRIMARY[600]),
            ("upload_speed",   "Upload (Mbps)",   GIGA_PRIMARY[700]),
            ("latency",        "Latency (ms)",    GIGA_GREY[600])]
if _src["loss_rate"].notna().any():
    _metrics.append(("loss_rate", "Packet loss", GIGA_BAD))

n = len(_metrics)
ncols = int(np.ceil(n / 2))
fig, axes = plt.subplots(2, ncols, figsize=(6 * ncols, 8))
axes = np.array(axes).reshape(-1)
for ax, (col, label, color) in zip(axes, _metrics):
    vals = _src[col].dropna()
    if col == "latency":
        vals = vals[(vals > 0) & (vals < 1000)]
    ax.hist(vals, bins=50, color=color, alpha=0.8, edgecolor="white")
    ax.axvline(vals.median(), color=GIGA_GREY[700], linestyle="--", linewidth=1.5,
               label=f"median {vals.median():.2f}")
    ax.axvline(vals.mean(), color=GIGA_GREY[900], linestyle=":", linewidth=1.5,
               label=f"mean {vals.mean():.2f}")
    ax.set_title(label); ax.set_ylabel("measurements"); ax.legend()
for ax in axes[n:]:           # hide any unused panel (e.g. when loss is absent)
    ax.set_visible(False)
fig.suptitle(f"Connectivity performance (school hours) — {COUNTRY_NAME}")
plt.tight_layout(); plt.show()
print(f"(school-hours measurements: {len(_src):,} of {len(m):,} total)")


In [ ]:
# =============================================================================
# TIME-OF-DAY BOTTLENECK SEVERITY — Identify Peak Hour Congestion
# =============================================================================

bottleneck = m.copy()
bottleneck['hour'] = pd.to_datetime(bottleneck['timestamplocal']).dt.hour
bottleneck['dayofweek'] = pd.to_datetime(bottleneck['timestamplocal']).dt.dayofweek  # 0=Mon, 6=Sun

# Hourly aggregates — PER-SCHOOL first (each school's median for that hour),
# then the median across schools, so heavy-testing schools don't dominate.
_sch_hour = (bottleneck.groupby(['school_id_giga', 'hour'])
             .agg(dl=('download_speed', 'median'), ul=('upload_speed', 'median'),
                  lat=('latency', 'median'), loss=('loss_rate', 'median'),
                  n=('download_speed', 'size')).reset_index())
hourly_perf = (_sch_hour.groupby('hour')
               .agg(DL_Median=('dl', 'median'), DL_Mean=('dl', 'mean'), DL_Std=('dl', 'std'),
                    Schools=('school_id_giga', 'nunique'), Measurements=('n', 'sum'),
                    UL_Median=('ul', 'median'), UL_Mean=('ul', 'mean'),
                    Lat_Median=('lat', 'median'), Lat_Mean=('lat', 'mean'),
                    Loss_Median=('loss', 'median'), Loss_Mean=('loss', 'mean')).round(4))

# Identify peak and off-peak
overall_median_dl = bottleneck.groupby('school_id_giga')['download_speed'].median().median()  # median of school medians
hourly_perf['DL_Diff'] = hourly_perf['DL_Median'] - overall_median_dl
hourly_perf['DL_Pct_Change'] = (hourly_perf['DL_Diff'] / overall_median_dl * 100).round(1)

peak_hour = hourly_perf['DL_Median'].idxmin()
offpeak_hour = hourly_perf['DL_Median'].idxmax()

print(f"\n{'='*80}")
print("TIME-OF-DAY BOTTLENECK SEVERITY")
print(f"{'='*80}")

print(f"\nOverall median download (median of school medians): {overall_median_dl:.2f} Mbps")
print(f"\nSlowest Hour: {peak_hour}:00 ({hourly_perf.loc[peak_hour, 'DL_Median']:.2f} Mbps, {hourly_perf.loc[peak_hour, 'DL_Pct_Change']:.1f}% vs median)")
print(f"Fastest Hour: {offpeak_hour}:00 ({hourly_perf.loc[offpeak_hour, 'DL_Median']:.2f} Mbps, {hourly_perf.loc[offpeak_hour, 'DL_Pct_Change']:.1f}% vs median)")

# School hours performance
school_hours_data = bottleneck[(bottleneck['hour'] >= SCHOOL_HOURS_START) & (bottleneck['hour'] < SCHOOL_HOURS_END)]
offhours_data = bottleneck[(bottleneck['hour'] < SCHOOL_HOURS_START) | (bottleneck['hour'] >= SCHOOL_HOURS_END)]

def _sch_med(df, col):
    return df.groupby('school_id_giga')[col].median().median()

_d = {}
for _lbl, _c, _hg in [('download', 'download_speed', True), ('upload', 'upload_speed', True),
                      ('latency', 'latency', False)]:
    _a, _b = _sch_med(school_hours_data, _c), _sch_med(offhours_data, _c)
    _d[_lbl] = (_a, _b, 100 * (_a - _b) / _b, _hg)
_la = 100 * school_hours_data.groupby('school_id_giga')['loss_rate'].mean().mean()
_lb = 100 * offhours_data.groupby('school_id_giga')['loss_rate'].mean().mean()
_worse = [k for k, (_a, _b, _pc, _hg) in _d.items() if (_pc < -10 if _hg else _pc > 10)]
_verdict = ("school-hours performance is materially WORSE than off-hours on: " + ", ".join(_worse)) if _worse \
           else "school-hours performance is broadly comparable to off-hours (within ±10% on all metrics)"
print(f"\n\nSCHOOL vs OFF-HOURS ({SCHOOL_HOURS_START}:00-{SCHOOL_HOURS_END}:00, per-school medians): "
      + "; ".join(f"{k} {_a:.1f} vs {_b:.1f} ({_pc:+.0f}%)" for k, (_a, _b, _pc, _hg) in _d.items())
      + f"; loss {_la:.2f}% vs {_lb:.2f}% (school vs off)"
      + f"\n-> {_verdict}.")

# Bottleneck severity classification
speed_drop = (1 - hourly_perf['DL_Median'].min() / hourly_perf['DL_Median'].max()) * 100
print(f"\n\nBottleneck Severity:")
print(f"  Speed drop from best to worst hour: {speed_drop:.1f}%")
if speed_drop > 50:
    print(f"  ⚠️  SEVERE: Major congestion peaks")
elif speed_drop > 30:
    print(f"  ⚠️  MODERATE: Notable time-of-day variation")
else:
    print(f"  ✓ MILD: Relatively consistent throughout day")

# Hourly table
print(f"\n\nHourly Performance Breakdown:")
print(hourly_perf[['DL_Median', 'DL_Std', 'DL_Pct_Change', 'UL_Median', 'Lat_Median', 'Loss_Mean', 'Measurements']].to_string())

# Visualization — one row per metric: level by hour (left), % change vs overall (right)
_overall_base = {
    'DL_Median':  bottleneck.groupby('school_id_giga')['download_speed'].median().median(),
    'UL_Median':  bottleneck.groupby('school_id_giga')['upload_speed'].median().median(),
    'Lat_Median': bottleneck.groupby('school_id_giga')['latency'].median().median(),
    'Loss_Mean':  100 * bottleneck.groupby('school_id_giga')['loss_rate'].mean().mean(),
}
_rows = [('DL_Median', 'Download (Mbps)', True, 1),
         ('UL_Median', 'Upload (Mbps)', True, 1),
         ('Lat_Median', 'Latency (ms)', False, 1),
         ('Loss_Mean', 'Packet loss (%)', False, 100)]
fig, axes = plt.subplots(4, 2, figsize=(14, 18))
for _r, (_col, _lab, _higher_good, _scale) in enumerate(_rows):
    _v = hourly_perf[_col] * _scale
    _base = _overall_base[_col]
    axL, axR = axes[_r, 0], axes[_r, 1]
    axL.plot(hourly_perf.index, _v, marker='o', linewidth=2, markersize=6, color='#0050e6')
    axL.axvspan(SCHOOL_HOURS_START, SCHOOL_HOURS_END, alpha=0.1, color='#277aff')
    axL.axhline(_base, color='#525252', linestyle='--', linewidth=1, alpha=0.6)
    axL.set_ylabel(_lab, fontweight='bold')
    axL.set_title(f"{_lab.split(' (')[0]} by Hour", fontweight='bold')
    axL.grid(alpha=0.3); axL.set_xticks(range(0, 24, 2))
    _pct = (_v - _base) / _base * 100
    _good = (_pct >= 0) if _higher_good else (_pct <= 0)
    axR.bar(hourly_perf.index, _pct, color=np.where(_good, '#00d661', '#ed1c24'),
            alpha=0.85, edgecolor='black')
    axR.axhline(0, color='black', linewidth=1)
    axR.axvspan(SCHOOL_HOURS_START, SCHOOL_HOURS_END, alpha=0.1, color='#277aff')
    axR.set_ylabel('% change vs overall', fontweight='bold')
    axR.set_title(f"{_lab.split(' (')[0]} Change by Hour (green = better)", fontweight='bold')
    axR.grid(axis='y', alpha=0.3); axR.set_xticks(range(0, 24, 2))
axes[3, 0].set_xlabel('Hour of Day', fontweight='bold')
axes[3, 1].set_xlabel('Hour of Day', fontweight='bold')
plt.tight_layout()
plt.show()

### 4.1. Learning Readiness (IQB-Edu)

In [ ]:
# =============================================================================
# IQB-EDU USE-CASE READINESS — mean score per use case (p50 typical, p95 best tail)
# =============================================================================
import numpy as np
ucs = IQB_USE_CASES

# ── Panel 1: % of SCHOOLS ready per use case (score >= benchmark), p50 & p95 ──
# (chart shows readiness shares — a mean IQB score blends pass/fail into an
#  unactionable number, so means are no longer charted)
ready_p50 = [100 * (school_iqb[f"iqb_{u.replace(' ', '_')}_p50"] >= IQB_BENCHMARK).mean() for u in ucs]
ready_p95 = [100 * (school_iqb[f"iqb_{u.replace(' ', '_')}_p95"] >= IQB_BENCHMARK).mean() for u in ucs]

# ── Panel 2: % of SCHOOL-DAYS ready per use case (day-grain pass rate) ────────
# A school-day passes a use case when the day's p50 inputs meet every threshold
# (score = 1). Days need >= MIN_TESTS_DAY school-hours complete-case tests.
MIN_TESTS_DAY = 3
_day = m[m['measurement_time_window'] == 'school_hours'].dropna(
    subset=['download_speed', 'upload_speed', 'latency', 'loss_rate']).copy()
_day['d'] = _day['timestamplocal'].dt.date
_dg = (_day.groupby(['school_id_giga', 'd'])
       .agg(n=('download_speed', 'size'), dl=('download_speed', 'median'),
            ul=('upload_speed', 'median'), lat=('latency', 'median'), loss=('loss_rate', 'median')))
_dg = _dg[_dg['n'] >= MIN_TESTS_DAY]
day_ready = {}
for _uc in ucs:
    _r = IQB_CONFIG['use cases'][_uc]['network requirements']
    _ok = ((_dg['dl'] >= _r['download_throughput_mbps']['threshold min']) &
           (_dg['ul'] >= _r['upload_throughput_mbps']['threshold min']) &
           (_dg['lat'] <= _r['latency_ms']['threshold min']) &
           ((_dg['loss'] <= _r['packet_loss']['threshold min']) | _dg['loss'].isna()))
    day_ready[_uc] = 100 * _ok.mean()

y = np.arange(len(ucs)); h = 0.38
fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
ax = axes[0]
ax.barh(y + h/2, ready_p50, height=h, color=GIGA_PRIMARY[600], edgecolor="white", label="p50 (typical)")
ax.barh(y - h/2, ready_p95, height=h, color=GIGA_PRIMARY[300], edgecolor="white", label="p95 (best tail)")
ax.set_yticks(y); ax.set_yticklabels(ucs); ax.invert_yaxis()
ax.set_xlim(0, 100); ax.set_xlabel(f"% of schools ready (score >= {IQB_BENCHMARK})")
ax.set_title(f"Schools ready per use case ({len(school_iqb)} schools)")
ax.legend(loc="lower right")
for yi, v in zip(y + h/2, ready_p50):
    ax.text(v + 1, yi, f"{v:.0f}%", va="center", fontsize=9)
for yi, v in zip(y - h/2, ready_p95):
    ax.text(v + 1, yi, f"{v:.0f}%", va="center", fontsize=9)

ax = axes[1]
_dv = [day_ready[u] for u in ucs]
ax.barh(y, _dv, height=0.6, color=GIGA_PRIMARY[800], edgecolor="white")
ax.set_xlim(0, 100); ax.set_xlabel("% of school-days ready (day p50 meets all thresholds)")
ax.set_title(f"School-days ready per use case ({len(_dg):,} school-days, >= {MIN_TESTS_DAY} tests/day)")
for yi, v in zip(y, _dv):
    ax.text(v + 1, yi, f"{v:.0f}%", va="center", fontsize=9)
fig.suptitle(f"IQB-Edu educational service readiness — {COUNTRY_NAME}")
plt.tight_layout(); plt.show()

# Readiness is a PROPORTION of schools, so each share carries a Wilson 95% CI.
# Schools with a NaN score for a use case are excluded from that use case's n.
def _ready_share(col):
    _valid = school_iqb[col].notna()
    _k = int((school_iqb.loc[_valid, col] >= IQB_BENCHMARK).sum())
    return _k, int(_valid.sum())

print("% of schools READY (p50 score >= benchmark) per use case  [Wilson 95% CI]:")
for u in ucs:
    col = f"iqb_{u.replace(' ', '_')}_p50"
    print(f"  {u:20s}: {fmt_pct_ci(*_ready_share(col))}")
if not m["loss_rate"].notna().any():
    print("\n⚠️ packet loss pending — scores exclude the loss requirement for now.")

print("\n% of schools READY (p95 score >= benchmark) per use case  [Wilson 95% CI]:")
for u in ucs:
    col = f"iqb_{u.replace(' ', '_')}_p95"
    print(f"  {u:20s}: {fmt_pct_ci(*_ready_share(col))}")
if not m["loss_rate"].notna().any():
    print("\n⚠️ packet loss pending — scores exclude the loss requirement for now.")

# =============================================================================
# IQB-EDU READINESS BY SUBGROUP — % of schools ready, per use case and core set
# =============================================================================
# Score MEANS are not shown (a mean of a pass-blend ranks groups on an
# unactionable number). Heatmap = % of the group's schools ready per use case
# (p50); bars = % ready on the CORE set at p50 and p95.
from matplotlib.colors import LinearSegmentedColormap as _LSC
_RAG = _LSC.from_list('rag', [GIGA_BAD, GIGA_MODERATE, GIGA_GOOD])
IQB_CORE_USE_CASES = ['web browsing', 'video streaming', 'audio streaming']

_pisp_iqb = (m.dropna(subset=['isp_mapped']).groupby('school_id_giga')['isp_mapped']
             .agg(lambda s: s.mode().iloc[0]).rename('primary_isp'))
school_iqb2 = school_iqb.merge(_pisp_iqb, on='school_id_giga', how='left')
for _p in (50, 95):
    school_iqb2[f'core_ready_p{_p}'] = np.logical_and.reduce(
        [(school_iqb2[f"iqb_{u.replace(' ', '_')}_p{_p}"] >= IQB_BENCHMARK).values
         for u in IQB_CORE_USE_CASES])

def _subgroup_readiness(col, title, min_schools=3):
    if col not in school_iqb2.columns or school_iqb2[col].notna().sum() == 0:
        print(f"({col}: not populated for {COUNTRY_NAME} — skipped)"); return
    _df = school_iqb2.dropna(subset=[col]).copy()
    _n = _df.groupby(col).size()
    _keep = _n[_n >= min_schools].index.tolist()
    if not _keep:
        print(f"({col}: no group with >= {min_schools} schools — skipped)"); return
    _df = _df[_df[col].isin(_keep)]
    _mats = {}
    for _p in (50, 95):
        _rcols = {u: f"_ready_{u.replace(' ', '_')}_p{_p}" for u in ucs}
        for u, _rc in _rcols.items():
            _df = _df.assign(**{_rc: _df[f"iqb_{u.replace(' ', '_')}_p{_p}"] >= IQB_BENCHMARK})
        _m = 100 * _df.groupby(col)[list(_rcols.values())].mean()
        _m.columns = list(_rcols.keys())
        _mats[_p] = _m
    core = _df.groupby(col)[['core_ready_p50', 'core_ready_p95']].mean() * 100
    _order = core.sort_values('core_ready_p50', ascending=False).index
    core = core.loc[_order]

    # Sequential white -> brand blue: at low readiness a 0-100 RAG collapses to
    # all-red; a sequential ramp keeps low values distinguishable.
    from matplotlib.colors import LinearSegmentedColormap as _LSC2
    _SEQ = _LSC2.from_list('seq', ['#ffffff', GIGA_PRIMARY[500], GIGA_PRIMARY[900]])

    fig, axes = plt.subplots(1, 3, figsize=(18, max(3.2, 0.55 * len(_order))),
                             gridspec_kw={'width_ratios': [3, 3, 1.5]})
    for _ax, _p in zip(axes[:2], (50, 95)):
        _mm = _mats[_p].loc[_order]
        _ax.imshow(_mm.values, aspect='auto', cmap=_SEQ, vmin=0, vmax=100)
        _ax.set_xticks(range(len(ucs)))
        _ax.set_xticklabels([u.replace(' ', chr(10)) for u in ucs], fontsize=8)
        if _p == 50:
            _ax.set_yticks(range(len(_order)))
            _ax.set_yticklabels([f"{str(g)[:20]} (n={_n[g]})" for g in _order], fontsize=9)
        else:
            _ax.set_yticks(range(len(_order))); _ax.set_yticklabels([])
        for _r in range(_mm.shape[0]):
            for _c in range(_mm.shape[1]):
                _v = _mm.values[_r, _c]
                _ax.text(_c, _r, f"{_v:.0f}", ha='center', va='center', fontsize=8,
                         color='white' if _v > 55 else GIGA_GREY[900])
        _ax.set_title(f"% of schools ready (p{_p})")

    ax = axes[2]
    _yb = np.arange(len(_order)); _hb = 0.38
    ax.barh(_yb + _hb/2, core['core_ready_p50'], height=_hb, color=GIGA_PRIMARY[600], label='p50')
    ax.barh(_yb - _hb/2, core['core_ready_p95'], height=_hb, color=GIGA_PRIMARY[300], label='p95')
    ax.set_yticks(_yb); ax.set_yticklabels([]); ax.invert_yaxis()
    ax.set_xlim(0, 100); ax.set_xlabel('% ready on CORE set')
    ax.set_title('Core set (web / video / audio)')
    ax.legend(fontsize=8, loc='lower right')
    for _yi, _v in zip(_yb + _hb/2, core['core_ready_p50']):
        ax.text(_v + 1, _yi, f"{_v:.0f}%", va='center', fontsize=8)
    for _yi, _v in zip(_yb - _hb/2, core['core_ready_p95']):
        ax.text(_v + 1, _yi, f"{_v:.0f}%", va='center', fontsize=8)
    fig.suptitle(f"IQB-Edu readiness {title} — {COUNTRY_NAME}", y=1.02)
    plt.tight_layout(); plt.show()

_subgroup_readiness('admin1', 'by region (admin1)')
_subgroup_readiness('education_level', 'by education level')
_subgroup_readiness('primary_isp', 'by primary ISP')
_subgroup_readiness('school_area_type', 'by urban/rural')
_subgroup_readiness('connectivity_type_govt', 'by connectivity type')


In [ ]:
# =============================================================================
# IQB FAIL REASONS — which requirement blocks readiness? (per use case, per ISP)
# =============================================================================
# Mirrors the pass_fail deep-dive: for schools NOT ready on a use case (p50),
# check which requirement their p50 inputs miss (a school can miss several).
# NaN inputs are skipped, matching the scoring engine.
_req_specs = {'download': ('download_p50', 'download_throughput_mbps', 'ge'),
              'upload':   ('upload_p50',   'upload_throughput_mbps',   'ge'),
              'latency':  ('latency_p50',  'latency_ms',               'le'),
              'loss':     ('loss_p50',     'packet_loss',              'le')}

_long = []
for _u in ucs:
    _req = IQB_CONFIG['use cases'][_u]['network requirements']
    _k = _u.replace(' ', '_')
    _nr = school_iqb2[school_iqb2[f'iqb_{_k}_p50'] < IQB_BENCHMARK].copy()
    _row = {'use_case': _u, 'not_ready_schools': len(_nr)}
    for _m, (_cv, _rk, _op) in _req_specs.items():
        _thr = _req[_rk]['threshold min']
        _fail = (_nr[_cv] < _thr) if _op == 'ge' else (_nr[_cv] > _thr)
        _row[_m] = round(100 * _fail.fillna(False).mean(), 0)
        _nr[f'fail_{_m}'] = _fail.fillna(False)
    _long.append((_u, _nr))
    if _u == ucs[0]:
        uc_fail = pd.DataFrame([_row])
    else:
        uc_fail = pd.concat([uc_fail, pd.DataFrame([_row])], ignore_index=True)

print("Why schools are NOT ready — % of not-ready schools missing each requirement (p50):")
display(uc_fail.set_index('use_case').astype(int))

# By ISP, CORE use cases only (incl. gaming/VC would swamp every ISP with the
# universal latency/upload misses — core isolates the addressable gaps)
_core_long = pd.concat([_nr.assign(use_case=_u) for _u, _nr in _long if _u in IQB_CORE_USE_CASES])
_core_long = _core_long.dropna(subset=['primary_isp'])
_isp_n = _core_long.groupby('primary_isp')['school_id_giga'].nunique()
_big_isps = _isp_n[_isp_n >= 15].index
isp_fail = (100 * _core_long[_core_long['primary_isp'].isin(_big_isps)]
            .groupby('primary_isp')[[f'fail_{_m}' for _m in _req_specs]].mean()).round(0).astype(int)
isp_fail.columns = list(_req_specs)
isp_fail.insert(0, 'not_ready_pairs', _core_long[_core_long['primary_isp'].isin(_big_isps)]
                .groupby('primary_isp').size())
print(f"\nBy primary ISP — % of not-ready (school x core-use-case) pairs missing each requirement")
print(f"(core set = {', '.join(IQB_CORE_USE_CASES)}; ISPs with >= 15 not-ready schools):")
display(isp_fail.sort_values('not_ready_pairs', ascending=False))
print("-> a high 'latency' share means the ISP's schools are latency-blocked even when"
      "\n   throughput clears; high 'download' means raw capacity is the binding gap.")

In [ ]:
# =============================================================================
# WEEKLY IQB PASS CONSISTENCY — which schools pass the use cases most weeks?
# =============================================================================
# Grain: school x week (Mon-start), school-hours complete-case tests. A week is
# analysable with >= MIN_TESTS_WEEK tests; a school is eligible with
# >= MIN_WEEKS_ELIGIBLE analysable weeks. A week PASSES when every use case in
# the set scores >= IQB_BENCHMARK on that week's p50 inputs.
# Two sets: ALL six use cases (strict — empty wherever gaming/video-conferencing
# never pass) and a CORE set. Weekly p50 over few tests is noisier than the
# all-time verdicts — read consistency as a stability signal, not a re-ranking.
MIN_TESTS_WEEK = 5
MIN_WEEKS_ELIGIBLE = 8
IQB_CORE_USE_CASES = ['web browsing', 'video streaming', 'audio streaming']

_wk = m[m['measurement_time_window'] == 'school_hours'].dropna(
    subset=['download_speed', 'upload_speed', 'latency', 'loss_rate']).copy()
_wk['week'] = _wk['timestamplocal'].dt.tz_localize(None).dt.to_period('W-SUN').dt.start_time
wk_iqb = (_wk.groupby(['school_id_giga', 'week'])
          .agg(n=('download_speed', 'size'),
               download_p50=('download_speed', 'median'), upload_p50=('upload_speed', 'median'),
               latency_p50=('latency', 'median'), loss_p50=('loss_rate', 'median')).reset_index())
wk_iqb = wk_iqb[wk_iqb['n'] >= MIN_TESTS_WEEK].copy()

_uc_cfg_w = {uc: _config_for_use_case(uc) for uc in IQB_USE_CASES}
def _wk_data(r):
    return {"m-lab": {"download_throughput_mbps": r['download_p50'],
                      "upload_throughput_mbps": r['upload_p50'],
                      "latency_ms": r['latency_p50'], "packet_loss": r['loss_p50']}}
for uc in IQB_USE_CASES:
    wk_iqb['pass_' + uc.replace(' ', '_')] = wk_iqb.apply(
        lambda r: calculate_iqb_score(_wk_data(r), _uc_cfg_w[uc]) >= IQB_BENCHMARK, axis=1)
_all_cols = ['pass_' + u.replace(' ', '_') for u in IQB_USE_CASES]
_core_cols = ['pass_' + u.replace(' ', '_') for u in IQB_CORE_USE_CASES]
wk_iqb['pass_all'] = wk_iqb[_all_cols].all(axis=1)
wk_iqb['pass_core'] = wk_iqb[_core_cols].all(axis=1)

cons = (wk_iqb.groupby('school_id_giga')
        .agg(weeks=('week', 'size'), pass_all_weeks=('pass_all', 'sum'),
             pass_core_weeks=('pass_core', 'sum')).reset_index())
cons = cons[cons['weeks'] >= MIN_WEEKS_ELIGIBLE].copy()
cons['pass_all_perc'] = (100 * cons['pass_all_weeks'] / cons['weeks']).round(0)
cons['pass_core_perc'] = (100 * cons['pass_core_weeks'] / cons['weeks']).round(0)
_meta_w = m[['school_id_giga', 'school_name'] + (['admin1'] if 'admin1' in m.columns else [])
            ].drop_duplicates('school_id_giga')
_pisp_w = (m.dropna(subset=['isp_mapped']).groupby('school_id_giga')['isp_mapped']
           .agg(lambda s: s.mode().iloc[0]).rename('primary_isp'))
cons = cons.merge(_meta_w, on='school_id_giga', how='left').merge(_pisp_w, on='school_id_giga', how='left')

print(f"Eligible schools (>= {MIN_WEEKS_ELIGIBLE} analysable weeks of >= {MIN_TESTS_WEEK} tests): {len(cons)}")
print(f"Pass ALL {len(IQB_USE_CASES)} use cases in >= 50% of weeks: {(cons['pass_all_perc'] >= 50).sum()} schools")
print(f"Pass the CORE set ({', '.join(IQB_CORE_USE_CASES)}) in >= 50% of weeks: {(cons['pass_core_perc'] >= 50).sum()} schools")

_ccols = ['school_name', 'primary_isp'] + (['admin1'] if 'admin1' in cons.columns else []) + \
         ['weeks', 'pass_core_perc', 'pass_all_perc']
print("\nMost consistent (core set):")
display(cons.sort_values(['pass_core_perc', 'weeks'], ascending=False).head(15)[_ccols])
print("Least consistent (core set):")
display(cons.sort_values(['pass_core_perc', 'weeks'], ascending=[True, False]).head(15)[_ccols])

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.hist(cons['pass_core_perc'], bins=20, color=GIGA_PRIMARY[600], alpha=0.85, edgecolor='white')
ax.set_xlabel('% of analysable weeks passing the core use-case set')
ax.set_ylabel('schools')
ax.set_title(f'Weekly IQB pass consistency (core set) — {COUNTRY_NAME}')
plt.tight_layout(); plt.show()

if EXPORT_RESULTS:
    from pathlib import Path as _Pw
    _Pw(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
    cons.to_csv(f"{OUTPUT_DIR}/{COUNTRY_ISO3.lower()}_iqb_weekly_consistency.csv", index=False)
    print(f"exported {COUNTRY_ISO3.lower()}_iqb_weekly_consistency.csv")

### 4.1 Performance by ISP (summary)

In [ ]:
# =============================================================================
# ISP SUMMARY (top providers + median performance) -- deep dive in the appendix
# =============================================================================
if "isp_mapped" in m.columns and m["isp_mapped"].notna().any():
    _top = m["isp_mapped"].value_counts().head(8).index
    isp_summary = (m[m["isp_mapped"].isin(_top)]
                   .groupby("isp_mapped")
                   .agg(measurements=("isp_mapped", "size"),
                        schools=("school_id_giga", "nunique"),
                        dl_median=("download_speed", "median"),
                        ul_median=("upload_speed", "median"),
                        latency_median=("latency", "median"),
                        loss_median=("loss_rate", "median"))
                   .sort_values("measurements", ascending=False)
                   .round({"dl_median": 1, "ul_median": 1, "latency_median": 0, "loss_median": 4}))
    print("Top ISPs by measurement volume:")
    display(isp_summary)
else:
    print("(no detected_isp data)")

# NOTE: dl_median above is pooled across ALL measurements (volume-weighted —
# heavy-testing schools dominate). Below: the SCHOOL-level view — each school's
# own median, distribution across the ISP's schools.
_pair = (m.dropna(subset=["isp_mapped"])
         .groupby(["isp_mapped", "school_id_giga"])[["download_speed", "upload_speed", "latency"]]
         .median().rename(columns={"download_speed": "dl", "upload_speed": "ul", "latency": "lat"})
         .reset_index())
_pair = _pair[_pair["isp_mapped"].isin(_top)]
def _school_dist(df, by):
    return (df.groupby(by)
            .agg(schools=("dl", "size"),
                 dl_p5=("dl", lambda s: s.quantile(0.05)), dl_p50=("dl", "median"), dl_p95=("dl", lambda s: s.quantile(0.95)),
                 ul_p5=("ul", lambda s: s.quantile(0.05)), ul_p50=("ul", "median"), ul_p95=("ul", lambda s: s.quantile(0.95)),
                 lat_p5=("lat", lambda s: s.quantile(0.05)), lat_p50=("lat", "median"), lat_p95=("lat", lambda s: s.quantile(0.95))))
_dist = _school_dist(_pair, "isp_mapped").sort_values("dl_p50", ascending=False).round(1)
print("\nPer-school medians — distribution across each ISP's schools (dl/ul Mbps, lat ms):")
display(_dist)

_order = [i for i in _dist.index if _dist.loc[i, "schools"] >= 5]
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, (_col, _lab) in zip(axes, [("dl", "download (Mbps)"), ("ul", "upload (Mbps)"), ("lat", "latency (ms)")]):
    _bp = ax.boxplot([_pair.loc[_pair["isp_mapped"] == i, _col].dropna() for i in _order],
                     tick_labels=[str(i)[:14] for i in _order], showfliers=False, patch_artist=True)
    for _p in _bp["boxes"]:
        _p.set(facecolor=GIGA_PRIMARY[300], edgecolor=GIGA_PRIMARY[700])
    for _med in _bp["medians"]:
        _med.set(color=GIGA_PRIMARY[800], linewidth=1.8)
    ax.set_ylabel(f"school median {_lab}")
    ax.tick_params(axis="x", rotation=45)
fig.suptitle(f"Per-school medians by ISP (boxes = school distribution) — {COUNTRY_NAME}")
plt.tight_layout(); plt.show()

if "detected_server" in m.columns and m["detected_server"].notna().any():
    print("\nTop M-Lab server locations:")
    display(m["detected_server"].value_counts().head(5).rename("measurements").to_frame())

In [ ]:
# =============================================================================
# ISP PERFORMANCE — DISTRIBUTION ACROSS SCHOOL-ISP PAIRS
# (the summary above pools all measurements per ISP, so heavy-measuring schools
#  dominate dl_median; here each school-ISP pair contributes ONE point: its own median)
# =============================================================================
MIN_PAIR_N = 5  # min measurements for a school-ISP pair to count

if "isp_mapped" in m.columns and m["isp_mapped"].notna().any():
    _pairs = (m.dropna(subset=["isp_mapped"])
                .groupby(["school_id_giga", "isp_mapped"])
                .agg(n=("download_speed", "size"),
                     dl=("download_speed", "median"),
                     ul=("upload_speed", "median"),
                     lat=("latency", "median"))
                .reset_index())
    _pairs = _pairs[_pairs["n"] >= MIN_PAIR_N]

    _top = m["isp_mapped"].value_counts().head(8).index
    _pairs_top = _pairs[_pairs["isp_mapped"].isin(_top)]

    _pd_isp = _school_dist(_pairs_top, "isp_mapped").sort_values("schools", ascending=False)
    # contrast: the pooled (measurement-weighted) median from the summary above
    _pd_isp["dl_pooled"] = (m[m["isp_mapped"].isin(_top)]
                            .groupby("isp_mapped")["download_speed"].median())
    print(f"Distribution ACROSS school-ISP pairs (pairs with >= {MIN_PAIR_N} measurements):")
    print("dl_p50 = the median school; dl_pooled = all-measurements median (summary above)")
    display(_pd_isp.round(1))

    fig, ax = plt.subplots(figsize=(11, max(3, 0.5 * len(_pd_isp))))
    _order = _pd_isp.index.tolist()
    _bp = ax.boxplot([_pairs_top.loc[_pairs_top["isp_mapped"] == i, "dl"] for i in _order],
                     tick_labels=[f"{str(i)[:20]} (n={_pd_isp.loc[i, 'schools']})" for i in _order],
                     showfliers=True, patch_artist=True, vert=False)
    for _p in _bp["boxes"]:
        _p.set(facecolor=GIGA_PRIMARY[300], edgecolor=GIGA_PRIMARY[700])
    for _med in _bp["medians"]:
        _med.set(color=GIGA_PRIMARY[800], linewidth=1.8)
    ax.invert_yaxis()
    ax.set_xlabel("school median download (Mbps)")
    ax.set_title(f"Download distribution across school-ISP pairs — {COUNTRY_NAME}")
    plt.tight_layout(); plt.show()
else:
    print("(no detected_isp data)")

In [ ]:
# =============================================================================
# PER-SCHOOL DOWNLOAD DISTRIBUTIONS — by admin2, education level, connectivity type
# =============================================================================
# Same lens as the ISP boxes above: each school's own median download, then the
# distribution ACROSS schools within each group.
_sm = (m.groupby('school_id_giga')[['download_speed', 'upload_speed', 'latency']]
       .median().rename(columns={'download_speed': 'dl', 'upload_speed': 'ul', 'latency': 'lat'})
       .reset_index())
_grp_cols = ['admin2', 'education_level', 'connectivity_type_govt']
_meta = m[['school_id_giga'] + [c for c in _grp_cols if c in m.columns]].drop_duplicates('school_id_giga')
_sm = _sm.merge(_meta, on='school_id_giga', how='left')

def _dist_by(col, title, min_schools=5):
    if col not in _sm.columns or _sm[col].notna().sum() == 0:
        print(f"({col}: not populated for {COUNTRY_NAME} — skipped)"); return
    _d = _school_dist(_sm.dropna(subset=[col]), col)
    _d = _d[_d['schools'] >= min_schools].sort_values('dl_p50', ascending=False).round(1)
    if _d.empty:
        print(f"({col}: no group with >= {min_schools} schools — skipped)"); return
    print(f"\n{title} — per-school medians (dl/ul Mbps, lat ms):")
    display(_d)
    fig, ax = plt.subplots(figsize=(11, max(3, 0.45 * len(_d))))
    _vals = [_sm.loc[_sm[col] == g, 'dl'].dropna() for g in _d.index]
    _bp = ax.boxplot(_vals, tick_labels=[str(g)[:22] for g in _d.index], showfliers=False,
                     patch_artist=True, vert=False)
    for _p in _bp['boxes']:
        _p.set(facecolor=GIGA_PRIMARY[300], edgecolor=GIGA_PRIMARY[700])
    for _med in _bp['medians']:
        _med.set(color=GIGA_PRIMARY[800], linewidth=1.8)
    ax.invert_yaxis()
    ax.set_xlabel('school median download (Mbps)')
    ax.set_title(f'{title} — {COUNTRY_NAME}')
    plt.tight_layout(); plt.show()

_dist_by('admin2', 'By province (admin2)')
_dist_by('education_level', 'By education level')
_dist_by('connectivity_type_govt', 'By connectivity type (govt)')

# ── ISP x education level together (school's primary ISP) ────────────────────
_pisp = (m.dropna(subset=['isp_mapped']).groupby('school_id_giga')['isp_mapped']
         .agg(lambda s: s.mode().iloc[0]).rename('primary_isp'))
_sm2 = _sm.merge(_pisp, on='school_id_giga', how='left')
_ie = _school_dist(_sm2.dropna(subset=['primary_isp', 'education_level']),
                   ['primary_isp', 'education_level'])
_ie = _ie[_ie['schools'] >= 3].round(1)
print("\nISP x education level — per-school medians (pairs with >= 3 schools; dl/ul Mbps, lat ms):")
display(_ie)

_isps = (_sm2.dropna(subset=['primary_isp'])['primary_isp'].value_counts()
         .loc[lambda s: s >= 15].index.tolist())
_lvls = [l for l in ['Pre-Primary', 'Primary', 'Secondary'] if l in _sm2['education_level'].unique()]
_lvl_colors = dict(zip(_lvls, [GIGA_PRIMARY[200], GIGA_PRIMARY[500], GIGA_PRIMARY[800]]))
fig, ax = plt.subplots(figsize=(12, 5))
_w = 0.8 / max(len(_lvls), 1)
for _k, _lvl in enumerate(_lvls):
    _pos, _vals = [], []
    for _j, _i in enumerate(_isps):
        _v = _sm2.loc[(_sm2['primary_isp'] == _i) & (_sm2['education_level'] == _lvl), 'dl'].dropna()
        if len(_v) >= 3:
            _pos.append(_j + (_k - (len(_lvls) - 1) / 2) * _w); _vals.append(_v)
    if _vals:
        _bp = ax.boxplot(_vals, positions=_pos, widths=_w * 0.85, showfliers=False, patch_artist=True)
        for _p in _bp['boxes']:
            _p.set(facecolor=_lvl_colors[_lvl], edgecolor=GIGA_PRIMARY[900])
        for _med in _bp['medians']:
            _med.set(color=GIGA_PRIMARY[900], linewidth=1.6)
ax.set_xticks(range(len(_isps))); ax.set_xticklabels([str(i)[:14] for i in _isps])
ax.set_ylabel('school median download (Mbps)')
ax.set_title(f'Per-school median download — ISP x education level — {COUNTRY_NAME}')
from matplotlib.patches import Patch as _Patch
ax.legend(handles=[_Patch(facecolor=_lvl_colors[l], edgecolor=GIGA_PRIMARY[900], label=l) for l in _lvls],
          fontsize=9)
plt.tight_layout(); plt.show()

---
# Part B — Appendix (full analysis)

Deep-dive analysis carried over from the analytics template. These run on the same `m` / `r` /
`school_iqb` objects built above. Some cells are exploratory; review before citing in a report.


### Data distributions

In [ ]:
# =============================================================================
# APPENDIX PREREQUISITES (carried-over deep-dive cells depend on these)
# =============================================================================
# Canonical ISP label for grouping. The optional country ISP-mapping file isn't
# required — fall back to the cleaned ISP name from the physical table.
if "isp_mapped" not in m.columns:
    m["isp_mapped"] = m["detected_isp"] if "detected_isp" in m.columns else m.get("isp_name")

# Snapshot of the full measurement frame for admin/education tier breakdowns.
m_geo = m.copy()

# Legacy Mbps service-tier scaffolding used by several appendix cells.

# classify_service_level / tier_order / TIER_THRESHOLD_* now live in eda_helpers
# (imported in the IMPORTS cell).

print("\u2713 Appendix prerequisites ready (isp_mapped, m_geo, classify_service_level, tier scaffolding)")

In [ ]:
# =============================================================================
# DATA DISTRIBUTIONS - GUIDE FOR PARAMETER SELECTION
# =============================================================================

# Overview of all numeric columns
print("\n" + "="*70)
print("NUMERIC VARIABLE DISTRIBUTIONS")
print("="*70)
print(m[['download_speed', 'upload_speed', 'latency', 'loss_rate',
         'detected_wifi_quality', 'detected_wifi_signal', 'detected_wifi_tx_rate']].describe().round(4))

# Create histograms for key speed/latency metrics + measurement timing
fig, axes = plt.subplots(3, 4, figsize=(20, 14))
fig.suptitle('Distribution of Speed, Latency, WiFi, Timing & Measurement Types', 
             fontsize=14, fontweight='bold', y=0.995)

# Download speed
axes[0, 0].hist(m['download_speed'].dropna(), bins=50, color='#277aff', alpha=0.7, edgecolor='black')
axes[0, 0].set_title(f'Download Speed (Mbps)\nMedian: {m["download_speed"].median():.1f} Mbps')
axes[0, 0].set_xlabel('Speed (Mbps)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].legend(fontsize=9)

# Upload speed
axes[0, 1].hist(m['upload_speed'].dropna(), bins=50, color='#0050e6', alpha=0.7, edgecolor='black')
axes[0, 1].set_title(f'Upload Speed (Mbps)\nMedian: {m["upload_speed"].median():.1f} Mbps')
axes[0, 1].set_xlabel('Speed (Mbps)')
axes[0, 1].set_ylabel('Frequency')

# Latency (filter to 0-1000ms before plotting to get proper bin resolution)
latency_valid = m[(m['latency'] > 0) & (m['latency'] < 1000)]['latency'].dropna()
axes[0, 2].hist(latency_valid, bins=50, color='#989898', alpha=0.7, edgecolor='black')
axes[0, 2].set_title(f'Latency (ms, 0-1000ms range)\nMedian: {latency_valid.median():.1f}ms (n={len(latency_valid):,})')
axes[0, 2].set_xlabel('Latency (ms)')
axes[0, 2].set_ylabel('Frequency')
# Reference lines: the chosen preprocessing threshold + distribution percentiles
axes[0, 2].axvline(LATENCY_OUTLIER_THRESHOLD, color='#ed1c24', linestyle='-', linewidth=1.5,
                   label=f'chosen cutoff: {LATENCY_OUTLIER_THRESHOLD}ms')
for _q, _qc in ((0.95, '#7eb0ff'), (0.99, '#525252')):
    _qv = latency_valid.quantile(_q)
    axes[0, 2].axvline(_qv, color=_qc, linestyle='--', linewidth=1.5, label=f'p{int(_q*100)}: {_qv:.0f}ms')
axes[0, 2].legend(fontsize=9)
outliers_count = (m['latency'] > 1000).sum()
axes[0, 2].text(0.98, 0.97, f'Excluded: {outliers_count:,} outliers (>{1000}ms)', 
                transform=axes[0, 2].transAxes, ha='right', va='top', fontsize=8, 
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Loss rate histogram (BytesRetrans / BytesSent)
loss_valid = m['loss_rate'].dropna()
loss_pct = loss_valid * 100
axes[0, 3].hist(loss_pct.clip(upper=loss_pct.quantile(0.99)), bins=50,
                color='#989898', alpha=0.7, edgecolor='black')
axes[0, 3].set_title(f'Loss Rate (%)\nMedian: {loss_pct.median():.2f}%  (n={len(loss_valid):,})')
axes[0, 3].set_xlabel('Loss Rate (%)')
axes[0, 3].set_ylabel('Frequency')

# WiFi Quality
axes[1, 0].hist(m['detected_wifi_quality'].dropna(), bins=50, color='#7eb0ff', alpha=0.7, edgecolor='black')
axes[1, 0].set_title(f'WiFi Quality\nMedian: {m["detected_wifi_quality"].median():.1f}')
axes[1, 0].set_xlabel('Quality')
axes[1, 0].set_ylabel('Frequency')

# WiFi Signal
axes[1, 1].hist(m['detected_wifi_signal'].dropna(), bins=50, color='#002d9c', alpha=0.7, edgecolor='black')
axes[1, 1].set_title(f'WiFi Signal (dBm)\nMedian: {m["detected_wifi_signal"].median():.1f}')
axes[1, 1].set_xlabel('Signal (dBm)')
axes[1, 1].set_ylabel('Frequency')

# WiFi TX Rate
axes[1, 2].hist(m['detected_wifi_tx_rate'].dropna(), bins=50, color='#0050e6', alpha=0.7, edgecolor='black')
axes[1, 2].set_title(f'WiFi TX Rate (Mbps)\nMedian: {m["detected_wifi_tx_rate"].median():.1f}')
axes[1, 2].set_xlabel('TX Rate (Mbps)')
axes[1, 2].set_ylabel('Frequency')

# Measurement type from 'notes' column
if 'notes' in m.columns:
    notes_counts = m['notes'].value_counts().head(10)
    axes[1, 3].barh(range(len(notes_counts)), notes_counts.values, color='#393939', alpha=0.7, edgecolor='black')
    axes[1, 3].set_yticks(range(len(notes_counts)))
    axes[1, 3].set_yticklabels([str(note)[:30] for note in notes_counts.index], fontsize=9)
    axes[1, 3].set_xlabel('Count')
    axes[1, 3].set_title(f'Measurement Types (from notes)\nTotal: {m["notes"].notna().sum():,}')
    axes[1, 3].invert_yaxis()
else:
    axes[1, 3].text(0.5, 0.5, 'No notes column', ha='center', va='center')
    axes[1, 3].set_title('Measurement Types')

# Measurement time of day (extract hour from timestamplocal)
if 'timestamplocal' in m.columns:
    m['measurement_hour'] = pd.to_datetime(m['timestamplocal']).dt.hour
    axes[2, 0].hist(m['measurement_hour'].dropna(), bins=24, color='#0050e6', alpha=0.7, edgecolor='black')
    axes[2, 0].set_title(f'Measurement Time of Day\n(n={m["measurement_hour"].notna().sum():,})')
    axes[2, 0].set_xlabel('Hour (24h)')
    axes[2, 0].set_ylabel('Frequency')
    axes[2, 0].set_xticks(range(0, 24, 3))
    axes[2, 0].axvline(SCHOOL_HOURS_START, color='#277aff', linestyle='--', linewidth=2, label='School start')
    axes[2, 0].axvline(SCHOOL_HOURS_END, color='#525252', linestyle='--', linewidth=2, label='School end')
    axes[2, 0].legend(fontsize=9)

# Measurement day of week
if 'timestamplocal' in m.columns:
    m['measurement_dayofweek'] = pd.to_datetime(m['timestamplocal']).dt.day_name()
    day_counts = m['measurement_dayofweek'].value_counts().reindex(['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'])
    axes[2, 1].bar(range(len(day_counts)), day_counts.values, color='#002d9c', alpha=0.7, edgecolor='black')
    axes[2, 1].set_xticks(range(len(day_counts)))
    axes[2, 1].set_xticklabels(['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'], rotation=45)
    axes[2, 1].set_title(f'Measurements by Day of Week\n(n={m["measurement_dayofweek"].notna().sum():,})')
    axes[2, 1].set_ylabel('Frequency')

# Measurement date range
if 'timestamplocal' in m.columns:
    m['measurement_date_only'] = pd.to_datetime(m['timestamplocal']).dt.date
    date_counts = m['measurement_date_only'].value_counts().sort_index()
    axes[2, 2].plot(range(len(date_counts)), date_counts.values, color='#525252', linewidth=1.5, marker='o', markersize=3)
    axes[2, 2].fill_between(range(len(date_counts)), date_counts.values, alpha=0.3, color='#525252')
    axes[2, 2].set_title(f'Measurement Timeline\n({date_counts.index[0]} to {date_counts.index[-1]})')
    axes[2, 2].set_xlabel('Date (index)')
    axes[2, 2].set_ylabel('Measurements per day')
    axes[2, 2].grid(alpha=0.3)

# Connectivity type distribution
if 'connectivity_type_govt' in m.columns:
    conn_counts = m['connectivity_type_govt'].value_counts()
    colors_conn = ['#277aff', '#989898', '#0050e6', '#002d9c'][:len(conn_counts)]
    axes[2, 3].bar(range(len(conn_counts)), conn_counts.values, color=colors_conn, alpha=0.7, edgecolor='black')
    axes[2, 3].set_xticks(range(len(conn_counts)))
    axes[2, 3].set_xticklabels(conn_counts.index, rotation=45, ha='right')
    axes[2, 3].set_title(f'Measurements by Connectivity Type\n(n={conn_counts.sum():,})')
    axes[2, 3].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

print("\n✓ Distributions plotted")
print("  - Speed/latency thresholds overlaid where applicable")
print("  - Measurement timing and type distribution shown")
print("  - School hours (green/red lines) for context")


### Categorical value counts

In [ ]:
# =============================================================================
# VALUE COUNTS FOR CATEGORICAL VARIABLES
# =============================================================================

# WiFi SSIDs (top 15)
print("\n" + "="*70)
print("TOP WiFi SSIDs DETECTED")
print("="*70)
ssid_counts = m['detected_wifi_ssid'].value_counts().head(15)
for ssid, count in ssid_counts.items():
    pct = count / m['detected_wifi_ssid'].notna().sum() * 100
    print(f"  {str(ssid)[:40]:40} {count:6,} ({pct:5.1f}%)")

# WiFi Models (top 10)
print("\n" + "="*70)
print("TOP WiFi ROUTER MODELS")
print("="*70)
model_counts = m['detected_wifi_model'].value_counts().head(10)
for model, count in model_counts.items():
    pct = count / m['detected_wifi_model'].notna().sum() * 100
    print(f"  {str(model)[:40]:40} {count:6,} ({pct:5.1f}%)")

# WiFi Channels
print("\n" + "="*70)
print("WiFi CHANNELS IN USE")
print("="*70)
channel_counts = m['detected_wifi_channel'].value_counts().sort_index()
for channel, count in channel_counts.items():
    pct = count / m['detected_wifi_channel'].notna().sum() * 100
    print(f"  Channel {int(channel):2.0f}: {count:6,} ({pct:5.1f}%)")

# Speedtest server locations - detected_server is the M-Lab server city
# (the physical table has no separate server-country column).
print("\n" + "="*70)
print("SPEEDTEST SERVER LOCATIONS (Top 15)")
print("="*70)
if 'detected_server' in m.columns and m['detected_server'].notna().any():
    server_counts = m['detected_server'].value_counts().head(15)
    for loc, count in server_counts.items():
        pct = count / m['detected_server'].notna().sum() * 100
        print(f"  {str(loc)[:32]:32} {count:6,} ({pct:5.1f}%)")
else:
    print("  (No detected_server data for this pull)")

# ISPs detected
print("\n" + "="*70)
print("ISPs DETECTED IN MEASUREMENTS (Top 10)")
print("="*70)
if 'detected_isp' in m.columns:
    isp_counts = m['detected_isp'].value_counts().head(10)
    for isp, count in isp_counts.items():
        pct = count / m['detected_isp'].notna().sum() * 100
        print(f"  {str(isp)[:40]:40} {count:6,} ({pct:5.1f}%)")
else:
    print("  (No detected_isp column found)")

print("\n\u2713 Analysis complete")
print("  \u2022 detected_server is the M-Lab server city (no server-country field in the physical table)")
print("  \u2022 Use ISP data to calibrate ISP_PATTERNS for your country")


### Device drop-off & retention

In [ ]:
# =============================================================================
# DEVICE DROP-OFF ANALYSIS — Measurement Duration & Retention Patterns
# =============================================================================

# Analyze per-school measurement windows
dropoff_data = m_original.copy()
dropoff_data['timestamp'] = pd.to_datetime(dropoff_data['timestamplocal'])

# Per-school: first and last measurement
school_retention = dropoff_data.groupby('school_id_giga').agg({
    'timestamp': ['min', 'max', 'count'],
}).reset_index()

school_retention.columns = ['school_id_giga', 'first_measurement', 'last_measurement', 'n_measurements']

# DEBUG: Check date ranges
print(f"\nDEBUG - First/Last Measurement Dates:")
print(f"  Earliest first_measurement: {school_retention['first_measurement'].min()}")
print(f"  Latest last_measurement: {school_retention['last_measurement'].max()}")
print(f"  Overall span: {(school_retention['last_measurement'].max() - school_retention['first_measurement'].min()).days} days")
print(f"\nDEBUG - Installation Dates (if available):")
if 'created_timestamp' in dropoff_data.columns:
    install_min = pd.to_datetime(dropoff_data['created_timestamp']).min()
    install_max = pd.to_datetime(dropoff_data['created_timestamp']).max()
    print(f"  Earliest installation: {install_min}")
    print(f"  Latest installation: {install_max}")
    print(f"  Span: {(install_max - install_min).days} days")
else:
    print(f"  No created_timestamp column — using first measurement as installation proxy")
school_retention['measurement_duration_days'] = (school_retention['last_measurement'] - school_retention['first_measurement']).dt.days
school_retention['days_since_last'] = (pd.Timestamp.now(tz='UTC') - school_retention['last_measurement'].dt.tz_convert('UTC')).dt.days
school_retention['daily_measurement_rate'] = school_retention['n_measurements'] / (school_retention['measurement_duration_days'] + 1)

# Installation date (fixed 2026-07-30): `created_timestamp` in the measurements
# table is PER-MEASUREMENT, not an install date. Merging it duplicated each school
# once per measurement, so every downstream count (histogram, cohorts) counted
# school-x-measurement rows instead of schools. First measurement = install proxy.
school_retention['measurement_duration_since_install'] = school_retention['measurement_duration_days']
install_label = "First Measurement (install proxy)"

# Define active vs dropped off
DROPOFF_THRESHOLD_DAYS = 30  # Schools inactive for >30 days
school_retention['status'] = school_retention['days_since_last'].apply(
    lambda x: 'Active' if x <= DROPOFF_THRESHOLD_DAYS else 'Dropped Off'
)

active_schools = school_retention[school_retention['status'] == 'Active']
dropped_schools = school_retention[school_retention['status'] == 'Dropped Off']

print(f"\n{'='*80}")
print("DEVICE DROP-OFF ANALYSIS")
print(f"{'='*80}")

print(f"\nSchool Status (inactive threshold: {DROPOFF_THRESHOLD_DAYS} days):")
print(f"  Active schools: {len(active_schools)} ({len(active_schools)/len(school_retention)*100:.1f}%)")
print(f"  Dropped off: {len(dropped_schools)} ({len(dropped_schools)/len(school_retention)*100:.1f}%)")

print(f"\n\nMeasurement Duration Statistics:")
print(f"  Median duration: {school_retention['measurement_duration_days'].median():.0f} days")
print(f"  Mean duration: {school_retention['measurement_duration_days'].mean():.0f} days")
print(f"  Min/Max: {school_retention['measurement_duration_days'].min():.0f} / {school_retention['measurement_duration_days'].max():.0f} days")

print(f"\n\nDaily Measurement Rate:")
print(f"  Median: {school_retention['daily_measurement_rate'].median():.2f} measurements/day")
print(f"  Mean: {school_retention['daily_measurement_rate'].mean():.2f} measurements/day")

print(f"\n\nActive Schools Performance:")
print(f"  Median duration: {active_schools['measurement_duration_days'].median():.0f} days")
print(f"  Median daily rate: {active_schools['daily_measurement_rate'].median():.2f} measurements/day")
print(f"  Mean measurements: {active_schools['n_measurements'].mean():.0f}")

print(f"\n\nDropped-Off Schools Performance:")
print(f"  Median duration before drop-off: {dropped_schools['measurement_duration_days'].median():.0f} days")
print(f"  Median daily rate (while active): {dropped_schools['daily_measurement_rate'].median():.2f} measurements/day")
print(f"  Mean measurements: {dropped_schools['n_measurements'].mean():.0f}")
print(f"  Days since last measurement: {dropped_schools['days_since_last'].median():.0f} days (median)")

# Time to first measurement
if 'days_to_first_measurement' in school_retention.columns:
    print(f"\n\nTime to First Measurement (install → first test):")
    valid_ttfm = school_retention[school_retention['days_to_first_measurement'].notna()]
    print(f"  Median: {valid_ttfm['days_to_first_measurement'].median():.0f} days")
    print(f"  Schools tested within 7 days: {(valid_ttfm['days_to_first_measurement'] <= 7).sum()} ({(valid_ttfm['days_to_first_measurement'] <= 7).sum()/len(valid_ttfm)*100:.1f}%)")
    print(f"  Schools took >30 days: {(valid_ttfm['days_to_first_measurement'] > 30).sum()} ({(valid_ttfm['days_to_first_measurement'] > 30).sum()/len(valid_ttfm)*100:.1f}%)")

# Drop-off timing pattern
print(f"\n\nDrop-Off Timing Pattern (when did dropped schools stop measuring?):")
if 'measurement_duration_since_install' in school_retention.columns:
    valid_dropped = dropped_schools[dropped_schools['measurement_duration_since_install'].notna()]
    if len(valid_dropped) > 0:
        print(f"  Median days from {install_label} to last measurement: {valid_dropped['measurement_duration_since_install'].median():.0f} days")
        print(f"  Median days from {install_label} to drop-off: {(valid_dropped['measurement_duration_since_install'] + valid_dropped['days_since_last']).median():.0f} days")
        
        # Cohort analysis
        print(f"\n  Drop-off by cohort (weeks since {install_label}):")
        valid_dropped = valid_dropped.copy()
        valid_dropped['cohort'] = pd.cut(valid_dropped['measurement_duration_since_install'] / 7, 
                                         bins=[0, 1, 2, 4, 8, 16, 1000],
                                         labels=['Week 1', 'Week 2', 'Weeks 3-4', 'Weeks 5-8', 'Weeks 9-16', 'Month 5+'])
        cohort_counts = valid_dropped['cohort'].value_counts().sort_index()
        print(cohort_counts.to_string())

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Measurement duration distribution
axes[0, 0].hist(school_retention['measurement_duration_days'], bins=50, color='#0050e6', alpha=0.7, edgecolor='black')
axes[0, 0].axvline(school_retention['measurement_duration_days'].median(), color='#525252', linestyle='--', 
                   linewidth=2, label=f"Median: {school_retention['measurement_duration_days'].median():.0f}d")
axes[0, 0].set_xlabel('Measurement Duration (days)', fontweight='bold')
axes[0, 0].set_ylabel('Number of Schools', fontweight='bold')
axes[0, 0].set_title('How Long Schools Measure For', fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# Active vs Dropped off
status_counts = school_retention['status'].value_counts()
colors = ['#00d661', '#ed1c24']
axes[0, 1].bar(status_counts.index, status_counts.values, color=colors, alpha=0.8, edgecolor='black')
axes[0, 1].set_ylabel('Number of Schools', fontweight='bold')
axes[0, 1].set_title(f'Active vs Dropped Off (>{DROPOFF_THRESHOLD_DAYS} days inactive)', fontweight='bold')
axes[0, 1].grid(axis='y', alpha=0.3)

for i, (idx, v) in enumerate(status_counts.items()):
    axes[0, 1].text(i, v + 1, f'{v}\n({v/len(school_retention)*100:.1f}%)', ha='center', fontweight='bold')

# Daily measurement rate
axes[1, 0].hist(school_retention['daily_measurement_rate'], bins=50, color='#277aff', alpha=0.7, edgecolor='black')
axes[1, 0].axvline(school_retention['daily_measurement_rate'].median(), color='#525252', linestyle='--',
                   linewidth=2, label=f"Median: {school_retention['daily_measurement_rate'].median():.2f}/day")
axes[1, 0].set_xlabel('Measurements per Day', fontweight='bold')
axes[1, 0].set_ylabel('Number of Schools', fontweight='bold')
axes[1, 0].set_title('Daily Measurement Rate', fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

# Drop-off timing pattern
if 'measurement_duration_since_install' in school_retention.columns:
    valid_dropped = dropped_schools[dropped_schools['measurement_duration_since_install'].notna()]
    if len(valid_dropped) > 0:
        axes[1, 1].hist(valid_dropped['measurement_duration_since_install'] / 7, bins=30, 
                       color='#989898', alpha=0.7, edgecolor='black')
        axes[1, 1].set_xlabel(f'Weeks from {install_label} to Last Measurement', fontweight='bold')
        axes[1, 1].set_ylabel('Number of Dropped-Off Schools', fontweight='bold')
        axes[1, 1].set_title('When Dropped-Off Schools Stopped Measuring', fontweight='bold')
        axes[1, 1].grid(alpha=0.3)
    else:
        axes[1, 1].text(0.5, 0.5, 'No dropped-off schools', ha='center', va='center')
else:
    axes[1, 1].text(0.5, 0.5, 'No installation data available', ha='center', va='center')

plt.tight_layout()
plt.show()

# Retention summary
print(f"\n\nRETENTION SUMMARY:")
if len(active_schools) > len(dropped_schools):
    print(f"  Good retention: {len(active_schools)/len(school_retention)*100:.0f}% of schools still actively measuring")
else:
    print(f"  Retention concern: {len(dropped_schools)/len(school_retention)*100:.0f}% of schools have dropped off")

# Check for early drop-off pattern
if 'measurement_duration_since_install' in school_retention.columns:
    early_dropoff = dropped_schools[dropped_schools['measurement_duration_since_install'] <= 14]
    if len(early_dropoff) > len(dropped_schools) * 0.3:
        print(f"  Pattern detected: {len(early_dropoff)} schools ({len(early_dropoff)/len(dropped_schools)*100:.0f}%) dropped within 2 weeks of {install_label}")
        print(f"    Suggests potential setup/onboarding issues")
    else:
        print(f"  Drop-offs spread over time, no early-stage cliff detected")

In [ ]:
# =============================================================================
# DROP-OFF v2 — PING-INFORMED (added 2026-07-30)
# =============================================================================
# Giga Meter V3 also sends connectivity pings (heartbeats) besides speed tests.
# For Fiji, pings exist only since Nov 2025 and only for V3 schools, so the
# comparison runs within the ping-capable cohort.
# Cache: fji_ping_daily.parquet (school_id_giga x date x n_pings x n_connected),
# built from gigameter_production_db.public.connectivity_ping_checks with
# is_deleted excluded and future-dated rows (data corruption, e.g. year 2039)
# dropped:
#   SELECT p.giga_id_school AS school_id_giga, CAST(p.timestamp AS date) AS date,
#          count(*) AS n_pings, sum(CASE WHEN p.is_connected THEN 1 ELSE 0 END) AS n_connected
#   FROM gigameter_production_db.public.connectivity_ping_checks p
#   JOIN gigameter_production_db.public.school s ON p.giga_id_school = s.giga_id_school
#   JOIN gigameter_production_db.public.country c ON s.country_id = c.id
#   WHERE c.code = '<ISO2>' AND NOT p.is_deleted
#     AND CAST(p.timestamp AS date) BETWEEN DATE '2025-11-01' AND current_date
#   GROUP BY 1, 2

ping_path = Path(CACHE_DIR) / f'{COUNTRY_ISO3.lower()}_ping_daily.parquet'
if not ping_path.exists():
    print(f"Missing {ping_path} - re-pull with the SQL in this cell's header comment.")
else:
    ping = pd.read_parquet(ping_path)
    ping['date'] = pd.to_datetime(ping['date'])

    _msrc = m_original if 'm_original' in globals() else m
    _md = _msrc[['school_id_giga', 'timestamplocal']].dropna().copy()
    _ts = _md['timestamplocal']
    _md['date'] = (_ts.dt.tz_localize(None) if _ts.dt.tz is not None else _ts).dt.normalize()
    meas_days = _md.drop_duplicates(['school_id_giga', 'date'])[['school_id_giga', 'date']]

    REF = max(ping['date'].max(), meas_days['date'].max())
    W0, W1 = ping['date'].min(), meas_days['date'].max()   # common observation window

    p_win = ping[(ping['date'] >= W0) & (ping['date'] <= W1)]
    m_win = meas_days[(meas_days['date'] >= W0) & (meas_days['date'] <= W1)]
    pk = set(map(tuple, p_win[['school_id_giga', 'date']].itertuples(index=False)))
    mk = set(map(tuple, m_win[['school_id_giga', 'date']].itertuples(index=False)))
    ping_schools = set(p_win['school_id_giga'])
    mk_pc = {k for k in mk if k[0] in ping_schools}

    print(f"{'='*80}")
    print(f"PING vs SPEED-TEST ACTIVITY — window {W0.date()} .. {W1.date()}")
    print(f"{'='*80}")
    print(f"Ping-capable schools (V3): {len(ping_schools)} of {meas_days['school_id_giga'].nunique()} ever-measured")
    print(f"Ping school-days: {len(pk):,} | speed-test school-days (ping-capable cohort): {len(mk_pc):,}")
    print(f"Ping days WITHOUT a speed test: {len(pk - mk):,} ({100*len(pk - mk)/max(len(pk),1):.1f}% of ping days)")
    print(f"Speed-test days WITHOUT a ping (ping-capable cohort): {len(mk_pc - pk):,} ({100*len(mk_pc - pk)/max(len(mk_pc),1):.1f}%)")

    lp = ping.groupby('school_id_giga')['date'].max().rename('last_ping')
    lm = meas_days.groupby('school_id_giga')['date'].max().rename('last_meas')
    j = pd.concat([lp, lm], axis=1).loc[sorted(ping_schools)]
    j['days_since_ping'] = (REF - j['last_ping']).dt.days
    j['days_since_meas'] = (REF - j['last_meas']).dt.days

    zombie = ((j['days_since_meas'] > 30) & (j['days_since_ping'] <= 30)).sum()
    dead   = ((j['days_since_meas'] > 30) & (j['days_since_ping'] > 30)).sum()
    print(f"\nDropped off by measurement (>30d) but STILL pinging: {zombie} schools")
    print(f"Silent on both signals (>30d): {dead} schools")

    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
    ax = axes[0]
    ax.scatter(j['days_since_meas'], j['days_since_ping'], s=22, alpha=0.55,
               c='#277aff', edgecolor='black', linewidth=0.3)
    ax.axvline(30, color='#ed1c24', ls='--', lw=1); ax.axhline(30, color='#ed1c24', ls='--', lw=1)
    ax.set_xlabel('Days since last SPEED TEST', fontweight='bold')
    ax.set_ylabel('Days since last PING', fontweight='bold')
    ax.set_title('Last activity per school - ping vs speed test\n(red lines = 30-day drop-off threshold)',
                 fontweight='bold')
    ax.grid(alpha=0.3)

    ax = axes[1]
    windows = [7, 30]
    defs = {
        'by speed test': [(j['days_since_meas'] <= w).sum() for w in windows],
        'by ping':       [(j['days_since_ping'] <= w).sum() for w in windows],
        'by either':     [((j['days_since_meas'] <= w) | (j['days_since_ping'] <= w)).sum() for w in windows],
    }
    x = np.arange(len(windows)); width = 0.26
    for k, (name, vals) in enumerate(defs.items()):
        bars = ax.bar(x + (k - 1) * width, vals, width, label=name,
                      color=['#277aff', '#00d661', '#989898'][k], edgecolor='black', alpha=0.85)
        for b, v in zip(bars, vals):
            ax.text(b.get_x() + b.get_width()/2, v, str(v), ha='center', va='bottom', fontsize=9)
    ax.set_xticks(x); ax.set_xticklabels([f'active <= {w}d' for w in windows])
    ax.set_ylabel('Schools (ping-capable cohort)', fontweight='bold')
    ax.set_title('"Active schools" under different signals', fontweight='bold')
    ax.legend(); ax.grid(axis='y', alpha=0.3)
    plt.tight_layout(); plt.show()

    print(f"\nVERDICT:")
    print(f"  - The speed test IS the more persistent signal here: ping never outlives it")
    print(f"    ({zombie} schools test-silent but ping-alive), and {100*len(mk_pc - pk)/max(len(mk_pc),1):.0f}% of")
    print(f"    speed-test days have no ping vs {100*len(pk - mk)/max(len(pk),1):.0f}% of ping days without a test.")
    print(f"  - Defining 'active' by ping would UNDERCOUNT active schools "
          f"({(j['days_since_ping'] <= 30).sum()} vs {(j['days_since_meas'] <= 30).sum()} at 30d).")
    print(f"  - Ping covers only the V3 cohort since Nov 2025 - measurement-based drop-off")
    print(f"    metrics remain the primary definition; ping adds no rescued schools.")


### Measurement pattern heatmap

In [ ]:
# =============================================================================
# MEASUREMENT PATTERN HEATMAP — Normalized Timeline (School-Relative Weeks)
# =============================================================================

# Normalize timeline: each school starts at week 1
pattern_data = m_original.copy()
pattern_data['measurement_date'] = pd.to_datetime(pattern_data['timestamplocal']).dt.date

# For each school, calculate weeks since their first measurement
school_first_measurement = pattern_data.groupby('school_id_giga')['measurement_date'].min()

def calc_weeks_since_start(row):
    first_date = school_first_measurement[row['school_id_giga']]
    current_date = row['measurement_date']
    weeks_diff = (pd.Timestamp(current_date) - pd.Timestamp(first_date)).days // 7
    return weeks_diff + 1  # Week 1, not Week 0

pattern_data['weeks_since_first'] = pattern_data.apply(calc_weeks_since_start, axis=1)

# Create a binary indicator: has measurement in this week?
weekly_measurements = pattern_data.groupby(['school_id_giga', 'school_name', 'weeks_since_first']).size().reset_index(name='n')
weekly_measurements['has_measurement'] = 1  # Binary indicator

# Pivot: schools as rows, weeks as columns
pivot_data = weekly_measurements.pivot_table(
    index=['school_id_giga', 'school_name'],
    columns='weeks_since_first',
    values='has_measurement',
    fill_value=0
)

# Sort schools by measurement span (longest first) to show drop-off patterns
pivot_data['span'] = pivot_data.columns[pivot_data.iloc[:, :].idxmax(axis=1)].astype(int)
pivot_data['total_weeks'] = (pivot_data > 0).sum(axis=1)
pivot_data = pivot_data.sort_values('total_weeks', ascending=False)
pivot_data = pivot_data.drop('total_weeks', axis=1)
pivot_data = pivot_data.drop('span', axis=1)

# Compute today's week per school (used for heatmap marker and summary table)
today = pd.Timestamp.now().date()
today_week_by_school = {}
for school_id in pivot_data.index.get_level_values(0):
    first_date = school_first_measurement[school_id]
    weeks_to_today = (today - first_date).days // 7 + 1
    today_week_by_school[school_id] = weeks_to_today

print(f"\n{'='*80}")
print("MEASUREMENT PATTERN HEATMAP (Normalized to School-Relative Weeks)")
print(f"{'='*80}")
print(f"\nSchools: {len(pivot_data)}")
print(f"Max weeks measured: {pivot_data.columns.max()}")
print(f"\nSchools by measurement span (weeks):")

span_dist = (pivot_data.sum(axis=1)).value_counts().sort_index(ascending=False)
for weeks, count in span_dist.head(10).items():
    pct = count / len(pivot_data) * 100
    print(f"  {int(weeks):3d} weeks: {int(count):3d} schools ({pct:5.1f}%)")

# Visualization: heatmap
fig, ax = plt.subplots(figsize=(20, max(8, len(pivot_data) * 0.15)))

im = ax.imshow(pivot_data.values, aspect='auto', cmap='Blues', vmin=0, vmax=1, interpolation='nearest')

# Format axes
ax.set_xticks(range(0, len(pivot_data.columns), max(1, len(pivot_data.columns)//20)))
ax.set_xticklabels([str(int(c)) for c in pivot_data.columns[::max(1, len(pivot_data.columns)//20)]], rotation=0, fontsize=9)
ax.set_yticks(range(len(pivot_data)))
ax.set_yticklabels([name[:30] for _, name in pivot_data.index], fontsize=7)

ax.set_xlabel('Weeks Since First Measurement (Normalized)', fontweight='bold', fontsize=11)
ax.set_ylabel('School', fontweight='bold', fontsize=11)
ax.set_title('Measurement Pattern Heatmap — Drop-Off Visibility\n(Darker = week with measurements, Lighter = no measurements, sorted by duration)',
            fontweight='bold', fontsize=13, pad=20)

# --- Today marker: per-row red tick at each school's current week ---
cols = list(pivot_data.columns)
max_col_idx = len(cols) - 1

today_x = []
today_y = []
for i, (school_id, school_name) in enumerate(pivot_data.index):
    tw = today_week_by_school.get(school_id, 0)
    if tw >= cols[-1]:
        # Today is at or beyond the rightmost plotted week — pin to right edge
        col_idx = max_col_idx
    elif tw < cols[0]:
        continue  # Shouldn't happen
    else:
        # Find the column index whose week number is closest to today's week
        col_idx = min(range(len(cols)), key=lambda j: abs(cols[j] - tw))
    today_x.append(col_idx + 0.5)
    today_y.append(i)

ax.scatter(today_x, today_y, marker='|', color='#525252', s=120, zorder=5,
           linewidths=1.5, label='Today')
ax.legend(loc='lower right', fontsize=9, framealpha=0.8)

# Colorbar
cbar = plt.colorbar(im, ax=ax, label='Has Measurements')
cbar.set_ticks([0, 1])
cbar.set_ticklabels(['No Measurement', 'Measurement'])

plt.tight_layout()
plt.show()

# Identify drop-off patterns
print(f"\n\nDROP-OFF PATTERNS:")
print(f"\nSchools with continuous measurement (no gaps):")
continuous_schools = []
for idx, row in pivot_data.iterrows():
    values = row.values
    if 0 not in values or (values == 0).sum() == len(values):
        continuous_schools.append((idx[1], int(row.sum())))

if continuous_schools:
    continuous_schools.sort(key=lambda x: x[1], reverse=True)
    for name, weeks in continuous_schools[:5]:
        print(f"  {name:<40} {weeks:3d} weeks continuous")
else:
    print(f"  (None - all schools have measurement gaps)")

print(f"\nSchools with abrupt drop-off (stopped measuring):")
dropoff_schools = []
for idx, row in pivot_data.iterrows():
    values = row.values
    if 1 in values:
        nonzero_idx = np.where(values == 1)[0]
    last_measurement_week = nonzero_idx[-1] + 1 if len(nonzero_idx) > 0 else 0
    if last_measurement_week < len(values) * 0.9:
        dropoff_week = values[::-1].tolist().index(1) if 1 in values else len(values)
        dropoff_schools.append((idx[1], last_measurement_week, len(values)))

if dropoff_schools:
    dropoff_schools.sort(key=lambda x: x[1])
    for name, last_week, max_week in dropoff_schools[:5]:
        print(f"  {name:<40} last measurement week {int(last_week)} (vs max {int(max_week)} weeks)")
else:
    print(f"  (None detected)")


# =============================================================================
# SUMMARY TABLE: Current Week for Each School
# =============================================================================

print(f"\n\n{'='*110}")
print(f"SCHOOL MEASUREMENT STATUS — Current Week & Drop-Off Status")
print(f"{'='*110}")
print(f"{'School':<50} {'Weeks':<12} {'Today':<12} {'Status':<25}")
print(f"{'-'*110}")

for (school_id, school_name) in pivot_data.index:
    total_weeks = (pivot_data.loc[(school_id, school_name)] > 0).sum()
    current_week = today_week_by_school.get(school_id, 0)

    if total_weeks >= current_week - 1:
        status = "ACTIVE"
    else:
        status = f"DROPPED at W{int(total_weeks)}"

    print(f"{school_name:<50} {int(total_weeks):<12} W{int(current_week):<11} {status:<25}")

print(f"{'='*110}")

### Device activity summary

In [ ]:
# =============================================================================
# DEVICE ACTIVITY SUMMARY
# =============================================================================

max_date = m['date'].max()
weekly_active_start = max_date - pd.Timedelta(days=6)

# Daily active
daily_active_schools = m[m['date'] == max_date]['school_id_giga'].nunique()

# Weekly active
weekly_active_schools = m[m['date'] >= weekly_active_start]['school_id_giga'].nunique()

# Offline schools (no data in last 7 days)
active_last_7_days = m[m['date'] >= weekly_active_start]['school_id_giga'].unique()
all_schools = m['school_id_giga'].unique()
offline_schools = [s for s in all_schools if s not in active_last_7_days]

print("="*60)
print(f"DEVICE ACTIVITY SUMMARY (as of {max_date.date()})")
print("="*60)
print(f"Daily Active Schools:         {daily_active_schools}")
print(f"Weekly Active Schools:        {weekly_active_schools}")
print(f"Offline Schools (7+ days):    {len(offline_schools)}")
print(f"Total Schools with Data:      {len(all_schools)}")


### Speed distribution histograms

In [ ]:
# =============================================================================
# SPEED DISTRIBUTION HISTOGRAMS - WITH SERVICE TIER BOUNDARIES
# =============================================================================
# TIER 0 - Insufficient
#   ├─ Download:    < 1 Mbps
#   ├─ Upload:      < 0.5 Mbps
#   ├─ Latency:     any
#   └─ Use cases:   No interactive use viable. Static page loads unreliably. Supports form submission / text-only email. 

#   TIER 1 - Basic          
#   ├─ Download:    1–5 Mbps
#   ├─ Upload:      0.5–2 Mbps
#   ├─ Latency:     < 150 ms   (unloaded)
#   ├─ Use cases:
#   │  ✓ Web browsing (text, images)
#   │  ✓ Email with small attachments
#   │  ✓ Audio calls (WhatsApp voice, VoIP)
#   │  ~ 1-on-1 video calls (latency sits at edge of acceptable)
#   └─  

#   TIER 2 - Ready         
#   ├─ Download:    5-20 Mbps
#   ├─ Upload:      2–10 Mbps
#   ├─ Latency:     < 100 ms   (unloaded)
#   ├─ Use cases:
#   │  ✓ Web browsing + email (comfortably)
#   │  ✓ Audio + 1-on-1 video calls
#   │  ✓ Video conferencing 2–5 participants (camera-on, SD)
#   │  ✓ Streaming 480p (YouTube classroom, receive-only)
#   │  ✓ Document uploads/downloads
#   │  ~ Basic Zoom class: viable receive-only; camera-on for all is tight (*depends on how many users share the link)
#   └─ 

#   TIER 3 - Advanced          
#   ├─ Download:    20+ Mbps
#   ├─ Upload:      10+ Mbps
#   ├─ Latency:     < 50 ms    (unloaded)
#   ├─ Packet Loss: < 0.5%
#   ├─ Use cases:
#   │  ✓ Full camera-on video conferencing (class-sized)
#   │  ✓ HD / 4K streaming
#   │  ✓ Large simultaneous  file transfers
#   │  ✓ Simultaneous interactive sessions
#   │  ✓ Real-time collaborative tools (live coding, experiments)
#   └─ 
# Define tier thresholds
TIER_THRESHOLDS = {
    'TIER 0 - Insufficient': 1,
    'TIER 1 - Basic': 5,
    'TIER 2 - Ready': 20,
    'TIER 3 - Advanced': None  # No upper bound shown
}

TIER_COLORS = {
    'TIER 0 - Insufficient': '#ed1c24',
    'TIER 1 - Basic': '#ffc93d',
    'TIER 2 - Ready': '#33ff8f',
    'TIER 3 - Advanced': '#00d661'
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Speed Distribution by Service Tier', fontsize=14, fontweight='bold')

# Download speed
axes[0].hist(m['download_speed'].dropna(), bins=50, color='#277aff', edgecolor='white', alpha=0.7)
axes[0].set_title('Download Speed Distribution')
axes[0].set_xlabel('Download Speed (Mbps)')
axes[0].set_ylabel('Count')

# Add tier boundary lines
colors_tier = ['#ed1c24', '#ffc93d', '#33ff8f', '#00d661']
thresholds = [1, 5, 20]
tier_names = ['TIER 0', 'TIER 1', 'TIER 2', 'TIER 3']

for threshold, color, tier_name in zip(thresholds, colors_tier[:-1], tier_names[:-1]):
    axes[0].axvline(x=threshold, color=color, linestyle='--', linewidth=2.5, alpha=0.8, label=tier_name)
axes[0].legend(fontsize=9, loc='upper right')
axes[0].set_xlim([0, 50])

# Upload speed
axes[1].hist(m['upload_speed'].dropna(), bins=50, color='#0050e6', edgecolor='white', alpha=0.7)
axes[1].set_title('Upload Speed Distribution')
axes[1].set_xlabel('Upload Speed (Mbps)')
axes[1].set_ylabel('Count')

# Add upload tier lines (1/5 of download for reference)
upload_thresholds = [0.5, 2, 10]
for threshold, color, tier_name in zip(upload_thresholds, colors_tier[:-1], tier_names[:-1]):
    axes[1].axvline(x=threshold, color=color, linestyle='--', linewidth=2.5, alpha=0.8, label=f'{tier_name} ({threshold}M)')
axes[1].legend(fontsize=9, loc='upper right')
axes[1].set_xlim([0, 20])

# Latency
axes[2].hist(m['latency'].dropna(), bins=50, color='#6f6f6f', edgecolor='white', alpha=0.7)
axes[2].set_title('Latency Distribution')
axes[2].set_xlabel('Latency (ms)')
axes[2].set_ylabel('Count')

# Add latency tier lines
latency_thresholds = [150, 100, 50]  # Basic, Ready, Advanced
latency_labels = ['TIER 1 (150ms)', 'TIER 2 (100ms)', 'TIER 3 (50ms)']
for threshold, color, label in zip(latency_thresholds, colors_tier[1:], latency_labels):
    axes[2].axvline(x=threshold, color=color, linestyle='--', linewidth=2.5, alpha=0.8, label=label)
axes[2].legend(fontsize=9, loc='upper right')
axes[2].set_xlim([0, 300])

plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("SERVICE TIER THRESHOLDS VISUALIZED")
print("="*70)
print("\nDownload Speed Boundaries:")
print("  TIER 0 - Insufficient: < 1 Mbps")
print("  TIER 1 - Basic: 1-5 Mbps")
print("  TIER 2 - Ready: 5-20 Mbps")
print("  TIER 3 - Advanced: 20+ Mbps")
print("\nUpload Speed Boundaries:")
print("  TIER 0 - Insufficient: < 0.5 Mbps")
print("  TIER 1 - Basic: 0.5-2 Mbps")
print("  TIER 2 - Ready: 2-10 Mbps")
print("  TIER 3 - Advanced: 10+ Mbps")
print("\nLatency Boundaries:")
print("  TIER 1 - Basic: < 150 ms")
print("  TIER 2 - Ready: < 100 ms")
print("  TIER 3 - Advanced: < 50 ms")


### Per-school speed/latency boxplots

In [ ]:
# Boxplots of download, upload, and latency per school (top 10 and bottom 10 by median *of each* value)
import numpy as np

# Compute stats for school ranking (at least 5 valid speed measurements)
school_speed_stats = (
    m.groupby(['school_id_giga', 'school_name'], dropna=False)
     .agg(
        download_speed_median=('download_speed', 'median'),
        upload_speed_median=('upload_speed', 'median'),
        latency_median=('latency', 'median'),
        num_measurements=('download_speed', 'count')
     )
     .reset_index()
)

# Only keep schools with 5+ measurements and non-null id/name
school_speed_stats = school_speed_stats[
    (school_speed_stats['num_measurements'] >= 5) &
    (school_speed_stats['school_id_giga'].notnull()) &
    (school_speed_stats['school_name'].notnull())
]

blue_fill = '#a9caff'
blue_line = '#277aff'
green_fill = '#a9caff'
green_line = '#0050e6'

### --- Top 10 and Bottom 10 by MEDIAN DOWNLOAD speed ---

# Download — top & bottom by median download
top10_download = school_speed_stats.nlargest(10, 'download_speed_median')
top10_download_sorted = top10_download.sort_values('download_speed_median', ascending=False)
top10_download_ids = top10_download_sorted['school_id_giga'].tolist()
top10_download_names = top10_download_sorted['school_name'].tolist()
top10_download_data = [
    m.loc[m['school_id_giga'] == sid, 'download_speed'].dropna().values
    for sid in top10_download_ids
]

fig, ax = plt.subplots(figsize=(14, 6))
bp = ax.boxplot(
    top10_download_data,
    vert=False,
    patch_artist=True,
    showfliers=True,
    widths=0.7,
    whis=[0, 100],
    medianprops=dict(color=blue_line),
    boxprops=dict(facecolor=blue_fill, color=blue_line, alpha=0.7)
)
ax.set_yticks(np.arange(1, len(top10_download_names)+1))
ax.set_yticklabels(top10_download_names)
plt.title('Top 10 Schools by Median Download Speed (min-max box)')
plt.xlabel('Download Speed (Mbps)')
plt.ylabel('School')
plt.tight_layout()
plt.show()

bottom10_download = school_speed_stats.nsmallest(10, 'download_speed_median')
bottom10_download_sorted = bottom10_download.sort_values('download_speed_median', ascending=True)
bottom10_download_ids = bottom10_download_sorted['school_id_giga'].tolist()
bottom10_download_names = bottom10_download_sorted['school_name'].tolist()
bottom10_download_data = [
    m.loc[m['school_id_giga'] == sid, 'download_speed'].dropna().values
    for sid in bottom10_download_ids
]

fig, ax = plt.subplots(figsize=(14, 6))
bp = ax.boxplot(
    bottom10_download_data,
    vert=False,
    patch_artist=True,
    showfliers=True,
    widths=0.7,
    whis=[0, 100],
    medianprops=dict(color=blue_line),
    boxprops=dict(facecolor=blue_fill, color=blue_line, alpha=0.7)
)
ax.set_yticks(np.arange(1, len(bottom10_download_names)+1))
ax.set_yticklabels(bottom10_download_names)
plt.title('Bottom 10 Schools by Median Download Speed (min-max box)')
plt.xlabel('Download Speed (Mbps)')
plt.ylabel('School')
plt.tight_layout()
plt.show()

### --- Top 10 and Bottom 10 by MEDIAN UPLOAD speed ---

top10_upload = school_speed_stats.nlargest(10, 'upload_speed_median')
top10_upload_sorted = top10_upload.sort_values('upload_speed_median', ascending=False)
top10_upload_ids = top10_upload_sorted['school_id_giga'].tolist()
top10_upload_names = top10_upload_sorted['school_name'].tolist()
top10_upload_data = [
    m.loc[m['school_id_giga'] == sid, 'upload_speed'].dropna().values
    for sid in top10_upload_ids
]

fig, ax = plt.subplots(figsize=(14, 6))
bp = ax.boxplot(
    top10_upload_data,
    vert=False,
    patch_artist=True,
    showfliers=True,
    widths=0.7,
    whis=[0, 100],
    medianprops=dict(color=green_line),
    boxprops=dict(facecolor=green_fill, color=green_line, alpha=0.7)
)
ax.set_yticks(np.arange(1, len(top10_upload_names)+1))
ax.set_yticklabels(top10_upload_names)
plt.title('Top 10 Schools by Median Upload Speed (min-max box)')
plt.xlabel('Upload Speed (Mbps)')
plt.ylabel('School')
plt.tight_layout()
plt.show()

bottom10_upload = school_speed_stats.nsmallest(10, 'upload_speed_median')
bottom10_upload_sorted = bottom10_upload.sort_values('upload_speed_median', ascending=True)
bottom10_upload_ids = bottom10_upload_sorted['school_id_giga'].tolist()
bottom10_upload_names = bottom10_upload_sorted['school_name'].tolist()
bottom10_upload_data = [
    m.loc[m['school_id_giga'] == sid, 'upload_speed'].dropna().values
    for sid in bottom10_upload_ids
]

fig, ax = plt.subplots(figsize=(14, 6))
bp = ax.boxplot(
    bottom10_upload_data,
    vert=False,
    patch_artist=True,
    showfliers=True,
    widths=0.7,
    whis=[0, 100],
    medianprops=dict(color=green_line),
    boxprops=dict(facecolor=green_fill, color=green_line, alpha=0.7)
)
ax.set_yticks(np.arange(1, len(bottom10_upload_names)+1))
ax.set_yticklabels(bottom10_upload_names)
plt.title('Bottom 10 Schools by Median Upload Speed (min-max box)')
plt.xlabel('Upload Speed (Mbps)')
plt.ylabel('School')
plt.tight_layout()
plt.show()

### --- Top 10 and Bottom 10 by MEDIAN LATENCY ---

top10_latency = school_speed_stats.nlargest(10, 'latency_median')
top10_latency_sorted = top10_latency.sort_values('latency_median', ascending=False)
top10_latency_ids = top10_latency_sorted['school_id_giga'].tolist()
top10_latency_names = top10_latency_sorted['school_name'].tolist()
top10_latency_data = [
    m.loc[m['school_id_giga'] == sid, 'latency'].dropna().values
    for sid in top10_latency_ids
]

fig, ax = plt.subplots(figsize=(14, 6))
bp = ax.boxplot(
    top10_latency_data,
    vert=False,
    patch_artist=True,
    showfliers=True,
    widths=0.7,
    whis=[0,100],
    medianprops=dict(color='#393939'),
    boxprops=dict(facecolor='#525252', color='#393939', alpha=0.5)
)
ax.set_yticks(np.arange(1, len(top10_latency_names)+1))
ax.set_yticklabels(top10_latency_names)
plt.title('Top 10 Schools by Median Latency (min-max box)')
plt.xlabel('Latency (ms)')
plt.ylabel('School')
plt.tight_layout()
plt.show()

bottom10_latency = school_speed_stats.nsmallest(10, 'latency_median')
bottom10_latency_sorted = bottom10_latency.sort_values('latency_median', ascending=True)
bottom10_latency_ids = bottom10_latency_sorted['school_id_giga'].tolist()
bottom10_latency_names = bottom10_latency_sorted['school_name'].tolist()
bottom10_latency_data = [
    m.loc[m['school_id_giga'] == sid, 'latency'].dropna().values
    for sid in bottom10_latency_ids
]

fig, ax = plt.subplots(figsize=(14, 6))
bp = ax.boxplot(
    bottom10_latency_data,
    vert=False,
    patch_artist=True,
    showfliers=True,
    widths=0.7,
    whis=[0,100],
    medianprops=dict(color='#393939'),
    boxprops=dict(facecolor='#525252', color='#393939', alpha=0.5)
)
ax.set_yticks(np.arange(1, len(bottom10_latency_names)+1))
ax.set_yticklabels(bottom10_latency_names)
plt.title('Bottom 10 Schools by Median Latency (min-max box)')
plt.xlabel('Latency (ms)')
plt.ylabel('School')
plt.tight_layout()
plt.show()

### Speed consistency (CV)

In [ ]:
m.columns

In [ ]:
# =============================================================================
# SPEED CONSISTENCY & RELIABILITY — Coefficient of Variation Analysis
# =============================================================================

# Calculate per-school variability
school_consistency = m.groupby('school_id_giga').agg({
    'download_speed': ['median', 'std', 'mean', 'min', 'max', 'count'],
    'upload_speed': ['median', 'std', 'mean'],
    'packet_loss_rate': 'median',
    'school_name': 'first',
    'admin1': 'first',
    'admin2': 'first'
}).reset_index()

school_consistency.columns = ['school_id_giga', 'dl_median', 'dl_std', 'dl_mean', 'dl_min', 'dl_max', 'n_measurements',
                              'ul_median', 'ul_std', 'ul_mean', 'loss_rate_median', 'school_name', 'admin1', 'admin2']

# Coefficient of Variation (std / mean) — measures relative variability
school_consistency['dl_cv'] = (school_consistency['dl_std'] / school_consistency['dl_mean']).fillna(0).replace([np.inf, -np.inf], 0)
school_consistency['ul_cv'] = (school_consistency['ul_std'] / school_consistency['ul_mean']).fillna(0).replace([np.inf, -np.inf], 0)

# Range-based variability
school_consistency['dl_range'] = school_consistency['dl_max'] - school_consistency['dl_min']
school_consistency['dl_range_pct'] = (school_consistency['dl_range'] / school_consistency['dl_median']) * 100

# Robust variability + eligibility (added 2026-07-30): plain CV = std/mean lets a
# single outlier test crown a school "variable" and 1-2-test schools score CV=0
# ("stable"). Rank instead by robust CV (IQR/median) over eligible schools only.
_q = m.groupby('school_id_giga')['download_speed'].quantile([0.25, 0.75]).unstack()
school_consistency = school_consistency.merge(
    (_q[0.75] - _q[0.25]).rename('dl_iqr').reset_index(), on='school_id_giga', how='left')
school_consistency = school_consistency.merge(
    m.groupby('school_id_giga')['timestamplocal'].agg(lambda s: s.dt.normalize().nunique())
      .rename('days_measured').reset_index(), on='school_id_giga', how='left')
school_consistency['dl_rcv'] = school_consistency['dl_iqr'] / school_consistency['dl_median']
eligible = school_consistency[(school_consistency['n_measurements'] >= 30)
                              & (school_consistency['days_measured'] >= 10)
                              & (school_consistency['dl_median'] > 0)].copy()
print(f"Eligible for stability ranking (>=30 tests & >=10 days): {len(eligible)} of {len(school_consistency)} schools")

# Sort by robust variability (eligible schools only)
most_variable = eligible.nlargest(15, 'dl_rcv')
most_stable = eligible.nsmallest(15, 'dl_rcv')

print(f"\n{'='*80}")
print("SPEED CONSISTENCY & RELIABILITY (Coefficient of Variation)")
print(f"{'='*80}")

print(f"\nMost VARIABLE Schools (unstable connectivity):")
print(most_variable[['school_name', 'admin2', 'n_measurements', 'dl_median', 'dl_rcv', 'dl_cv', 'dl_range', 'loss_rate_median']].round(4).to_string(index=False))

print(f"\n\nMost STABLE Schools (consistent connectivity):")
print(most_stable[['school_name', 'admin2', 'n_measurements', 'dl_median', 'dl_rcv', 'dl_cv', 'dl_range', 'loss_rate_median']].round(4).to_string(index=False))

# Statistics
print(f"\n\nOverall Consistency Statistics:")
print(f"  Mean Coefficient of Variation: {school_consistency['dl_cv'].mean():.2f}")
print(f"  Median Coefficient of Variation: {school_consistency['dl_cv'].median():.2f}")
print(f"  Schools with CV < 0.5 (stable): {(school_consistency['dl_cv'] < 0.5).sum()} ({(school_consistency['dl_cv'] < 0.5).sum()/len(school_consistency)*100:.1f}%)")
print(f"  Schools with CV > 1.0 (highly variable): {(school_consistency['dl_cv'] > 1.0).sum()} ({(school_consistency['dl_cv'] > 1.0).sum()/len(school_consistency)*100:.1f}%)")

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Distribution of CV
valid_cv = school_consistency[np.isfinite(school_consistency['dl_cv'])]['dl_cv']
axes[0, 0].hist(valid_cv, bins=50, color='#0050e6', alpha=0.7, edgecolor='black')
axes[0, 0].axvline(valid_cv.median(), color='#525252', linestyle='--', linewidth=2, label=f'Median: {valid_cv.median():.2f}')
axes[0, 0].set_xlabel('Coefficient of Variation (Download Speed)', fontweight='bold')
axes[0, 0].set_ylabel('Number of Schools', fontweight='bold')
axes[0, 0].set_title('Speed Variability Distribution', fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# Median speed vs CV
axes[0, 1].scatter(school_consistency['dl_median'], school_consistency['dl_cv'], 
                   s=school_consistency['n_measurements']*2, alpha=0.6, c='#277aff', edgecolor='black')
axes[0, 1].set_xlabel('Median Download Speed (Mbps)', fontweight='bold')
axes[0, 1].set_ylabel('Coefficient of Variation', fontweight='bold')
axes[0, 1].set_title('Speed vs Stability (bubble size = measurements)', fontweight='bold')
axes[0, 1].grid(alpha=0.3)

# Top variable schools
top_var = eligible.nlargest(10, 'dl_rcv')
axes[1, 0].barh(range(len(top_var)), top_var['dl_rcv'], color='#989898', alpha=0.8, edgecolor='black')
axes[1, 0].set_yticks(range(len(top_var)))
axes[1, 0].set_yticklabels([name[:25] for name in top_var['school_name']], fontsize=9)
axes[1, 0].set_xlabel('Robust CV (IQR / median)', fontweight='bold')
axes[1, 0].set_title('Top 10 Most Variable Schools', fontweight='bold')
axes[1, 0].grid(axis='x', alpha=0.3)

# Speed range distribution
axes[1, 1].scatter(school_consistency['dl_min'], school_consistency['dl_max'], 
                   s=50, alpha=0.6, c='#7eb0ff', edgecolor='black')
# Add diagonal line for no variability
max_speed = school_consistency['dl_max'].max()
axes[1, 1].plot([0, max_speed], [0, max_speed], 'r--', linewidth=2, alpha=0.5, label='No variability')
axes[1, 1].set_xlabel('Minimum Download Speed (Mbps)', fontweight='bold')
axes[1, 1].set_ylabel('Maximum Download Speed (Mbps)', fontweight='bold')
axes[1, 1].set_title('Speed Range: Min vs Max per School', fontweight='bold')
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n\n💡 INSIGHT:")
print(f"   CV < 0.5 = Reliable connectivity (good for education)")
print(f"   CV 0.5-1.0 = Moderately variable")
print(f"   CV > 1.0 = Highly unstable (problematic for real-time learning)")

In [ ]:
# =============================================================================
# SPEED TIME SERIES — MOST VARIABLE vs MOST STABLE (well-defined; added 2026-07-30)
# =============================================================================
# Uses `eligible` + robust CV from the consistency cell above, so 1-2-test schools
# can't rank "stable" and single outliers can't rank a school "variable".
# Dots are display-capped at each school's p99.
pick_var = eligible.nlargest(3, 'dl_rcv')
pick_sta = eligible.nsmallest(3, 'dl_rcv')
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
panels = [(axes[0][i], row, '#ed1c24', 'VARIABLE') for i, (_, row) in enumerate(pick_var.iterrows())] + \
         [(axes[1][i], row, '#00d661', 'STABLE') for i, (_, row) in enumerate(pick_sta.iterrows())]
for ax, row, color, tag in panels:
    d = m[m['school_id_giga'] == row['school_id_giga']].sort_values('timestamplocal')
    cap = d['download_speed'].quantile(0.99)
    ax.plot(d['timestamplocal'], d['download_speed'].clip(upper=cap), '.', color=color, alpha=0.4, ms=4)
    roll = d.set_index('timestamplocal')['download_speed'].clip(upper=cap).rolling('7D').median()
    ax.plot(roll.index, roll.values, color=color, lw=1.4)
    ax.axhline(row['dl_median'], color='#525252', ls='--', lw=1)
    ax.set_title(f"{tag}: {str(row['school_name'])[:28]}\n"
                 f"robust CV={row['dl_rcv']:.2f}  med={row['dl_median']:.0f} Mbps  (n={int(row['n_measurements'])})",
                 fontsize=9, color=color, fontweight='bold')
    ax.tick_params(axis='x', rotation=45, labelsize=7)
axes[0][0].set_ylabel('Download (Mbps)')
axes[1][0].set_ylabel('Download (Mbps)')
fig.suptitle('Speed time series - most variable vs most stable schools\n'
             '(eligible: >=30 tests & >=10 measured days; robust CV = IQR/median; display capped at p99)',
             fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()


### ISP performance analysis

In [ ]:
# =============================================================================
# ISP PERFORMANCE ANALYSIS
# =============================================================================
MIN_MEASUREMENTS_PER_ISP = 10

isp_performance = m.groupby('isp_mapped').agg({
    'download_speed': ['mean', 'median', 'std'],
    'upload_speed': ['mean', 'median', 'std'],
    'latency': ['mean', 'median', 'std'],
    'packet_loss_rate': 'median',
    'measurement_id': 'count',
    'school_id_giga': 'nunique'
}).reset_index()

isp_performance.columns = [
    'isp', 'avg_download', 'median_download', 'std_download',
    'avg_upload', 'median_upload', 'std_upload',
    'avg_latency', 'median_latency', 'std_latency',
    'median_loss_rate',
    'measurement_count', 'schools_served'
]

isp_perf = (
    isp_performance
    [isp_performance['measurement_count'] >= MIN_MEASUREMENTS_PER_ISP]
    [isp_performance['schools_served'] > 2]
    .sort_values('median_download', ascending=False)
    .copy()
)

# Presentation table
display_df = isp_perf[[
    'isp', 'schools_served', 'measurement_count',
    'median_download', 'median_upload', 'median_latency', 'std_download', 'median_loss_rate'
]].copy()
display_df['median_loss_rate'] = display_df['median_loss_rate'] * 100  # convert to %

display_df.columns = [
    'ISP', 'Schools', 'Measurements',
    'Download median (Mbps)', 'Upload median (Mbps)', 'Latency median (ms)', 'Download std dev',
    'Loss Rate median (%)'
]
display_df = display_df.reset_index(drop=True)

(
    display_df.style
    .format({
        'Download median (Mbps)': '{:.1f}',
        'Upload median (Mbps)':   '{:.1f}',
        'Latency median (ms)':    '{:.0f}',
        'Download std dev':       '{:.1f}',
        'Loss Rate median (%)':   '{:.2f}',
        'Measurements':           '{:,}',
    })
    .background_gradient(subset=['Download median (Mbps)'], cmap='RdYlGn', vmin=0, vmax=100)
    .background_gradient(subset=['Upload median (Mbps)'],   cmap='RdYlGn', vmin=0, vmax=50)
    .background_gradient(subset=['Latency median (ms)'],    cmap='RdYlGn_r', vmin=5, vmax=80)
    .background_gradient(subset=['Download std dev'],       cmap='RdYlGn_r', vmin=0, vmax=100)
    .background_gradient(subset=['Loss Rate median (%)'],    cmap='RdYlGn_r', vmin=0, vmax=5)
    .set_properties(**{'text-align': 'center', 'font-size': '13px'})
    .set_properties(subset=['ISP'], **{'text-align': 'left', 'font-weight': 'bold'})
    .set_table_styles([
        {'selector': 'th', 'props': [('font-size', '12px'), ('text-align', 'center'),
                                     ('background-color', '#161616'), ('color', 'white'),
                                     ('padding', '8px 12px')]},
        {'selector': 'td', 'props': [('padding', '6px 12px')]},
        {'selector': 'tr:hover td', 'props': [('background-color', '#eaf2ff !important')]},
        {'selector': '', 'props': [('border-collapse', 'collapse'), ('width', '100%')]},
    ])
    .set_caption(
        f'ISP Performance — {COUNTRY_ISO3} | '
        f'sorted by median download | min {MIN_MEASUREMENTS_PER_ISP} measurements, >2 schools'
    )
    .hide(axis='index')
)


### ISP differences — statistical significance


In [ ]:
# =============================================================================
# ARE THE ISP DIFFERENCES REAL? — omnibus + BH-corrected pairwise (per-school)
# =============================================================================
# The table above ranks ISPs by median download, but a ranking is not a test:
# with hundreds of measurements per school, trivial gaps look certain. So we
# aggregate to one median per SCHOOL per ISP (no pseudoreplication) and then:
#   1. Kruskal-Wallis omnibus - do the ISPs differ at all on this metric?
#   2. Only if they do, every pairwise Mann-Whitney shift with a Benjamini-
#      Hochberg FDR correction, each carrying a bootstrap CI and Cliff's delta.
# A school needs >= MIN_MEAS_PER_ISP_SCHOOL measurements on an ISP to count, and
# an ISP needs >= MIN_SCHOOLS_PER_ISP such schools.
MIN_SCHOOLS_PER_ISP = 3
MIN_MEAS_PER_ISP_SCHOOL = 3

_isp_src = m.dropna(subset=['isp_mapped'])
for _metric, _better, _lbl in [('download_speed', 'up', 'download'),
                               ('upload_speed', 'up', 'upload'),
                               ('latency', 'down', 'latency')]:
    if _metric not in _isp_src.columns:
        continue
    _om = kruskal_omnibus(_isp_src, unit_col='school_id_giga', group_col='isp_mapped',
                          value_col=_metric, min_units=MIN_SCHOOLS_PER_ISP,
                          min_per_cell=MIN_MEAS_PER_ISP_SCHOOL)
    print(f"\n{'='*72}")
    print(f"ISP {_lbl.upper()} - Kruskal-Wallis across {_om['n_groups']} ISPs "
          f"({_om['n_units']} school-ISP medians)")
    if _om['n_groups'] < 2:
        print("  too few qualifying ISPs to compare"); continue
    print(f"  H = {_om['statistic']:.1f}   p = {_om['p_value']:.4g}   "
          f"epsilon^2 = {_om['effect']:.3f}  (0 = no separation, 1 = total)")
    if not (_om['p_value'] < 0.05):
        print("  -> ISPs do NOT differ overall on this metric; the ranking is within noise.")
        continue
    _pw = pairwise_shift_tests(_isp_src, unit_col='school_id_giga', group_col='isp_mapped',
                               value_col=_metric, groups=_om['groups'],
                               min_units=MIN_SCHOOLS_PER_ISP, min_per_cell=MIN_MEAS_PER_ISP_SCHOOL)
    _nsig = int((_pw['p_adj'] < 0.05).sum())
    print(f"  -> {_nsig}/{len(_pw)} ISP pairs differ after BH correction. "
          f"'d_median (A-B)' is A minus B; a CI excluding 0 means the direction is trustworthy.")
    if _lbl == 'latency':
        print("     (latency: LOWER is better, so a positive A-B means ISP A is worse.)")
    display(_pw.head(12))


In [ ]:
# =============================================================================
# ISP PERFORMANCE ANALYSIS - by education level
# =============================================================================

In [ ]:
# =============================================================================
# ISP PERFORMANCE ANALYSIS - by education level and per month
# =============================================================================

### ISP distribution

In [ ]:
# =============================================================================
# ISP DISTRIBUTION VISUALIZATION
# =============================================================================

top_n = 5
top_isps = m['isp_mapped'].value_counts().head(top_n)

# top_isps = isp_performance_filtered['isp'].value_counts().head()

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.bar(range(len(top_isps)), top_isps.values, color='#277aff')
ax.set_xticks(range(len(top_isps)))
ax.set_xticklabels(top_isps.index, rotation=45, ha='right')
ax.set_xlabel('ISP')
ax.set_ylabel('Number of Measurements')
ax.set_title(f'Top {top_n} ISPs by Measurement Count - {COUNTRY_NAME}')
ax.grid(axis='y', alpha=0.3)

for i, val in enumerate(top_isps.values):
    ax.text(i, val + max(top_isps.values)*0.01, f'{val:,}', 
            ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

print(f"\nMeasurements from top {top_n}: {top_isps.sum():,} ({top_isps.sum()/len(m)*100:.1f}%)")

### ISP download performance

In [ ]:
# =============================================================================
# ISP DOWNLOAD SPEED PERFORMANCE
# =============================================================================

top_n_isps = 15
top_isp_download = isp_perf.head(top_n_isps)

fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(range(len(top_isp_download)), top_isp_download['avg_download'], 
       color='#277aff', alpha=0.8, label='Mean')
ax.errorbar(range(len(top_isp_download)), top_isp_download['avg_download'],
            yerr=top_isp_download['std_download'],
            fmt='none', ecolor='#002d9c', capsize=4, alpha=0.6)

ax.set_xticks(range(len(top_isp_download)))
ax.set_xticklabels(top_isp_download['isp'], rotation=45, ha='right')
ax.set_ylabel('Avg Download Speed (Mbps)')
ax.set_title(f'ISP Download Performance - {COUNTRY_NAME}')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

### ISP upload performance

In [ ]:
# =============================================================================
# ISP UPLOAD SPEED PERFORMANCE
# =============================================================================

isp_upload = isp_perf.sort_values('avg_upload', ascending=False).head(top_n_isps)

fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(range(len(isp_upload)), isp_upload['avg_upload'], 
       color='#7eb0ff', alpha=0.8)
ax.errorbar(range(len(isp_upload)), isp_upload['avg_upload'],
            yerr=isp_upload['std_upload'],
            fmt='none', ecolor='#0050e6', capsize=4, alpha=0.6)

ax.set_xticks(range(len(isp_upload)))
ax.set_xticklabels(isp_upload['isp'], rotation=45, ha='right')
ax.set_ylabel('Avg Upload Speed (Mbps)')
ax.set_title(f'ISP Upload Performance - {COUNTRY_NAME}')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

### ISP latency performance

In [ ]:
# =============================================================================
# ISP LATENCY PERFORMANCE
# =============================================================================

isp_latency = isp_perf.sort_values('avg_latency').head(top_n_isps)

fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(range(len(isp_latency)), isp_latency['avg_latency'], 
       color='#002d9c', alpha=0.8)
ax.errorbar(range(len(isp_latency)), isp_latency['avg_latency'],
            yerr=isp_latency['std_latency'],
            fmt='none', ecolor='#6f6f6f', capsize=4, alpha=0.6)

ax.set_xticks(range(len(isp_latency)))
ax.set_xticklabels(isp_latency['isp'], rotation=45, ha='right')
ax.set_ylabel('Avg Latency (ms)')
ax.set_title(f'ISP Latency Performance (Lower is Better) - {COUNTRY_NAME}')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

### Per-school ISP assignment

In [ ]:
# =============================================================================
# PER-SCHOOL ISP ASSIGNMENT
# =============================================================================

# Assign most common ISP detected at each school
school_isp_assignment = m.groupby('school_id_giga').agg({
    'isp_mapped': lambda x: x.value_counts().index[0] if len(x.dropna()) > 0 else 'Unknown',
    'school_name': 'first',
    'admin2': 'first',
    'connectivity_type_govt': 'first',
}).reset_index()

school_isp_assignment.columns = ['school_id_giga', 'school_isp', 'school_name', 'admin2', 'connectivity_type_govt']

print(f"✓ Assigned ISP to {len(school_isp_assignment):,} schools")
print(f"  Unique ISPs assigned: {school_isp_assignment['school_isp'].nunique()}")

# Merge back to measurements for per-school analysis
m = m.merge(school_isp_assignment[['school_id_giga', 'school_isp']], on='school_id_giga', how='left')

print(f"\nSchool-ISP Distribution:")
print(school_isp_assignment['school_isp'].value_counts().head(10))


### Bandwidth throttling & concurrent providers

In [ ]:
m.columns

In [ ]:
# =============================================================================
# BANDWIDTH THROTTLING & CONCURRENT PROVIDERS
# =============================================================================

throttle_analysis = m.copy()

# 1. BANDWIDTH THROTTLING DETECTION
# Look for clustering at common speed ceilings
throttle_thresholds = [10, 20, 50, 100]  # Common throttling points (Mbps)

print(f"\n{'='*80}")
print("BANDWIDTH THROTTLING EVIDENCE")
print(f"{'='*80}")

# Per-school: detect if speeds cluster near a ceiling
school_throttle = throttle_analysis.groupby('school_id_giga').agg({
    'download_speed': ['median', 'max', 'count'],
    'school_name': 'first',
    'isp_mapped': 'first'
}).reset_index()

school_throttle.columns = ['school_id_giga', 'dl_median', 'dl_max', 'n', 'school_name', 'provider']

# Detect hard caps: if max is consistently near a threshold
throttled_schools = []
for threshold in throttle_thresholds:
    near_threshold = school_throttle[
        (school_throttle['dl_max'] >= threshold * 0.95) & (school_throttle['dl_max'] <= threshold * 1.05)
    ]
    throttled_schools.append({
        'threshold': threshold,
        'count': len(near_threshold),
        'examples': near_threshold.head(3)
    })

print(f"\nSchools with Potential Hard Caps:")
for item in throttled_schools:
    if item['count'] > 0:
        print(f"\n  ~{item['threshold']} Mbps threshold: {item['count']} schools")
        for _, row in item['examples'].iterrows():
            print(f"    - {row['school_name'][:40]}: median={row['dl_median']:.1f}M, max={row['dl_max']:.1f}M ({row['provider']})")

# 2. CONCURRENT PROVIDERS (Multiple ISPs per school)
print(f"\n\n{'='*80}")
print("CONCURRENT PROVIDERS — Schools with Multiple ISPs")
print(f"{'='*80}")

# Get detected ISP from connectivity data
CONCURRENT_MIN_MEASUREMENTS = 10   # minimum measurements from secondary ISP
CONCURRENT_MIN_SHARE = 0.05        # minimum share of school's total measurements

if 'isp_mapped' in throttle_analysis.columns:
    # Per school, per ISP: count measurements and share
    isp_counts = (
        throttle_analysis.groupby(['school_id_giga', 'isp_mapped'])
        .size()
        .reset_index(name='isp_n')
    )
    school_totals = throttle_analysis.groupby('school_id_giga').size().reset_index(name='total_n')
    isp_counts = isp_counts.merge(school_totals, on='school_id_giga')
    isp_counts['isp_share'] = isp_counts['isp_n'] / isp_counts['total_n']

    # Keep only ISPs that meet both thresholds
    qualified = isp_counts[
        (isp_counts['isp_n'] >= CONCURRENT_MIN_MEASUREMENTS) &
        (isp_counts['isp_share'] >= CONCURRENT_MIN_SHARE)
    ]

    # Schools with ≥2 qualifying ISPs are genuinely multi-provider
    qualifying_isps_per_school = (
        qualified.sort_values('isp_n', ascending=False)
        .groupby('school_id_giga')['isp_mapped']
        .apply(list)
        .reset_index(name='isp_mapped')
    )
    qualifying_isps_per_school['n_providers'] = qualifying_isps_per_school['isp_mapped'].apply(len)

    school_meta = throttle_analysis.groupby('school_id_giga').agg(
        school_name=('school_name', 'first'),
        connectivity_provider=('isp_mapped', 'first')
    ).reset_index()

    isp_diversity = qualifying_isps_per_school.merge(school_meta, on='school_id_giga')
    multi_provider = isp_diversity[isp_diversity['n_providers'] > 1].sort_values('n_providers', ascending=False)
    
    print(f"\nSchools with detected multiple ISPs (redundancy/failover):")
    print(f"Total schools with concurrent providers: {len(multi_provider)}")
    
    if len(multi_provider) > 0:
        print(f"\nTop examples:")
        for _, row in multi_provider.head(10).iterrows():
            isps = ', '.join([str(x)[:20] for x in row['isp_mapped']])
            print(f"  {row['school_name'][:40]}: {isps}")
    else:
        print(f"\nNo schools with detected concurrent providers.")
        print(f"(Single provider per school suggests no built-in redundancy)")
else:
    print(f"\nISP data not available for concurrent provider analysis.")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of max speeds showing potential throttle points
axes[0].hist(school_throttle['dl_max'], bins=100, color='#0050e6', alpha=0.7, edgecolor='black')
for threshold in throttle_thresholds:
    axes[0].axvline(threshold, color='#525252', linestyle='--', linewidth=2, alpha=0.7, label=f'{threshold}M')
axes[0].set_xlabel('Maximum Download Speed per School (Mbps)', fontweight='bold')
axes[0].set_ylabel('Number of Schools', fontweight='bold')
axes[0].set_title('Speed Ceiling Distribution (Red lines = Throttle thresholds)', fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Provider distribution
if 'isp_mapped' in throttle_analysis.columns:
    isp_dist = throttle_analysis['isp_mapped'].value_counts().head(10)
    axes[1].barh(range(len(isp_dist)), isp_dist.values, color='#277aff', alpha=0.8, edgecolor='black')
    axes[1].set_yticks(range(len(isp_dist)))
    axes[1].set_yticklabels([str(x)[:30] for x in isp_dist.index], fontsize=10)
    axes[1].set_xlabel('Number of Measurements', fontweight='bold')
    axes[1].set_title('Top 10 Detected ISPs', fontweight='bold')
    axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

### Time-of-day bottleneck severity

### Data plan / hard-cap detection

In [ ]:
# =============================================================================
# DATA PLAN / HARD-CAP DETECTION
# =============================================================================

# Detect if schools are hitting bandwidth caps (10, 20, 50 Mbps are common)
COMMON_CAPS = [10, 20, 50]
CAP_TOLERANCE = 1.5  # p95 within this Mbps below a round cap → suspected cap

# For each school, get most common ISP (in case school_isp wasn't already merged)
school_isp_most_common = m.groupby('school_id_giga')['isp_mapped'].apply(
    lambda x: x.value_counts().index[0] if len(x.dropna()) > 0 else 'Unknown'
).reset_index()
school_isp_most_common.columns = ['school_id_giga', 'primary_isp']

# Calculate per-school speed percentiles
school_speed_stats = m.groupby('school_id_giga').agg({
    'download_speed': ['median', lambda x: x.quantile(0.95), 'max', 'count'],
    'upload_speed': ['median', lambda x: x.quantile(0.95)],
    'school_name': 'first',
}).reset_index()

school_speed_stats.columns = ['school_id_giga', 'dl_median', 'dl_p95', 'dl_max', 'n_measurements',
                              'ul_median', 'ul_p95', 'school_name']

# Merge ISP info
school_speed_stats = school_speed_stats.merge(school_isp_most_common, on='school_id_giga', how='left')

# Detect suspected caps
def detect_cap(dl_p95):
    """Check if p95 download speed suggests a bandwidth cap."""
    for cap in COMMON_CAPS:
        if cap - CAP_TOLERANCE <= dl_p95 <= cap:
            return f"{cap} Mbps"
    return "None detected"

school_speed_stats['suspected_cap'] = school_speed_stats['dl_p95'].apply(detect_cap)

# Filter schools with suspected caps
suspected_caps = school_speed_stats[school_speed_stats['suspected_cap'] != 'None detected'].copy()

print(f"{'='*80}")
print(f"DATA PLAN / BANDWIDTH CAP DETECTION")
print(f"{'='*80}")
print(f"\nSchools with suspected bandwidth caps: {len(suspected_caps)}")

if len(suspected_caps) > 0:
    print(f"\nTop 15 schools showing cap signatures:")
    for idx, school in suspected_caps.nlargest(15, 'dl_p95').iterrows():
        print(f"  {school['school_name'][:30]:30} | p95={school['dl_p95']:6.1f}M | Cap: {school['suspected_cap']:10} | ISP: {school['primary_isp']}")
    
    print(f"\nCap Distribution:")
    print(suspected_caps['suspected_cap'].value_counts())
else:
    print(f"\n✓ No schools showing strong cap signatures")


### Hourly speed profile

In [ ]:
# =============================================================================
# HOURLY SPEED PROFILE — When are speeds best/worst?
# =============================================================================

# Use ORIGINAL unfiltered data for fair comparison
m_for_hourly = m_original.copy()

# Extract hour from timestamp
m_for_hourly['hour'] = pd.to_datetime(m_for_hourly['timestamplocal']).dt.hour

# Calculate hourly statistics
hourly_stats = m_for_hourly.groupby('hour').agg({
    'download_speed': ['median', 'mean', lambda x: x.quantile(0.25), lambda x: x.quantile(0.75), 'count'],
    'upload_speed': ['median', 'mean'],
    'latency': ['median', 'mean'],
}).reset_index()

hourly_stats.columns = ['hour', 'dl_median', 'dl_mean', 'dl_q25', 'dl_q75', 'n_measurements',
                        'ul_median', 'ul_mean', 'latency_median', 'latency_mean']

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Download speed by hour
axes[0, 0].plot(hourly_stats['hour'], hourly_stats['dl_median'], marker='o', linewidth=2.5, label='Median', color='#277aff')
axes[0, 0].fill_between(hourly_stats['hour'], hourly_stats['dl_q25'], hourly_stats['dl_q75'], alpha=0.3, color='#277aff', label='Q25-Q75')
axes[0, 0].axvspan(SCHOOL_HOURS_START, SCHOOL_HOURS_END, alpha=0.1, color='#277aff', label='School hours')
axes[0, 0].set_title('Download Speed by Hour of Day', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Hour (24h)')
axes[0, 0].set_ylabel('Speed (Mbps)')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# Upload speed by hour
axes[0, 1].plot(hourly_stats['hour'], hourly_stats['ul_median'], marker='s', linewidth=2.5, color='#0050e6')
axes[0, 1].axvspan(SCHOOL_HOURS_START, SCHOOL_HOURS_END, alpha=0.1, color='#277aff')
axes[0, 1].set_title('Upload Speed by Hour of Day', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Hour (24h)')
axes[0, 1].set_ylabel('Speed (Mbps)')
axes[0, 1].grid(alpha=0.3)

# Latency by hour
axes[1, 0].plot(hourly_stats['hour'], hourly_stats['latency_median'], marker='d', linewidth=2.5, color='#989898')
axes[1, 0].axvspan(SCHOOL_HOURS_START, SCHOOL_HOURS_END, alpha=0.1, color='#277aff')
axes[1, 0].set_title('Latency by Hour of Day', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Hour (24h)')
axes[1, 0].set_ylabel('Latency (ms)')
axes[1, 0].grid(alpha=0.3)

# School hours vs off-hours comparison (using full unfiltered data)
school_hours_data = m_for_hourly[(m_for_hourly['hour'] >= SCHOOL_HOURS_START) & (m_for_hourly['hour'] <= SCHOOL_HOURS_END)]
off_hours_data = m_for_hourly[~((m_for_hourly['hour'] >= SCHOOL_HOURS_START) & (m_for_hourly['hour'] <= SCHOOL_HOURS_END))]

comparison = pd.DataFrame({
    'Period': ['School Hours', 'Off-Hours'],
    'Measurements': [len(school_hours_data), len(off_hours_data)],
    'DL (Mbps)': [school_hours_data['download_speed'].median(), off_hours_data['download_speed'].median()],
    'UL (Mbps)': [school_hours_data['upload_speed'].median(), off_hours_data['upload_speed'].median()],
    'Latency (ms)': [school_hours_data['latency'].median(), off_hours_data['latency'].median()]
})

axes[1, 1].axis('tight')
axes[1, 1].axis('off')
table = axes[1, 1].table(cellText=comparison.round(1).values, colLabels=comparison.columns,
                         cellLoc='center', loc='center', bbox=[0, 0, 1, 1])
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 2)
axes[1, 1].set_title('School Hours vs Off-Hours Performance', fontsize=12, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

print(f"\n{'='*70}")
print("SCHOOL HOURS vs OFF-HOURS COMPARISON")
print(f"{'='*70}")
print(comparison.to_string(index=False))

# ── Statistical significance: per-school PAIRED shift (robust) ────────────────
# The old version ran an independent t-test on raw measurement rows. That is
# doubly wrong here: (a) rows are not independent — one school contributes many,
# so the t-test sees thousands of "observations" that are really a few dozen
# schools, and reports a p-value that is always tiny; (b) speeds are right-skewed,
# so a mean-based test is the wrong tool. Instead we pair each school with itself
# (school-hours vs off-hours), aggregate to one median per school per period, and
# test the paired shift nonparametrically with a bootstrap CI on the effect size.
# This same paired_shift_test template is reused for ISP and WiFi/Ethernet.
if 'school_id_giga' in m_for_hourly.columns:
    m_for_hourly['period'] = np.where(
        (m_for_hourly['hour'] >= SCHOOL_HOURS_START) & (m_for_hourly['hour'] <= SCHOOL_HOURS_END),
        'School Hours', 'Off-Hours')

    print(f"\nPer-school paired shift, School Hours vs Off-Hours "
          f"(each school its own control; medians, Wilcoxon signed-rank, bootstrap CI):")
    for _metric, _better in [('download_speed', 'up'), ('upload_speed', 'up'), ('latency', 'down')]:
        if _metric not in m_for_hourly.columns:
            continue
        _res = paired_shift_test(
            m_for_hourly, unit_col='school_id_giga', group_col='period', value_col=_metric,
            group_a='School Hours', group_b='Off-Hours', min_per_cell=3)
        print(format_shift_result(_res, label=f'  {_metric}', better=_better))
else:
    print("\n(No school_id_giga column — skipping per-school paired significance test.)")


### School-level performance by ISP

In [ ]:
# =============================================================================
# SCHOOL-LEVEL PERFORMANCE BY ISP
# =============================================================================

# Per-school statistics with ISP
# NOTE (fixed 2026-07-30): connectivity_type_govt is 100% null for FJI; as a groupby
# key pandas drops every row -> empty table. Group without it, merge it back after.
school_perf = m.groupby(['school_id_giga', 'school_name', 'school_isp']).agg({
    'download_speed': ['median', 'mean', lambda x: x.quantile(0.95), 'count'],
    'upload_speed': ['median', 'mean'],
    'latency': ['median', 'mean'],
}).reset_index()

school_perf.columns = ['school_id_giga', 'school_name', 'school_isp',
                       'dl_median', 'dl_mean', 'dl_p95', 'n',
                       'ul_median', 'ul_mean', 'latency_median', 'latency_mean']
school_perf = school_perf.merge(
    m[['school_id_giga', 'connectivity_type_govt']].drop_duplicates('school_id_giga'),
    on='school_id_giga', how='left')

# Apply service tier classification
school_perf['service_tier'] = school_perf.apply(
    lambda row: classify_service_level(row['dl_median'], row['ul_median'], row['latency_median']),
    axis=1
)

# Merge in suspected cap info
school_perf = school_perf.merge(
    school_speed_stats[['school_id_giga', 'suspected_cap']],
    on='school_id_giga',
    how='left'
)

# Filter to schools with sufficient data
school_perf_reliable = school_perf[school_perf['n'] >= 10]

print(f"\n{'='*70}")
print("SCHOOL-LEVEL PERFORMANCE BY ISP")
print(f"{'='*70}")
print(f"\nTotal schools with 10+ measurements: {len(school_perf_reliable):,}")

# Summary by ISP
print(f"\nPerformance by ISP (median download speed):")
isp_perf = school_perf_reliable.groupby('school_isp').agg({
    'school_id_giga': 'count',
    'dl_median': ['median', 'mean', 'min', 'max'],
    'dl_p95': ['median', 'mean'],
    'latency_median': 'median'
}).round(1)

isp_perf.columns = ['Schools', 'DL_p50', 'DL_mean', 'DL_min', 'DL_max', 'DL_p95_median', 'DL_p95_mean', 'Lat_median']
isp_perf = isp_perf.sort_values('DL_p50', ascending=False)

print(isp_perf.head(10).to_string())

# Service tier distribution by ISP
print(f"\n\nService Tier Distribution by ISP:")
tier_by_isp = pd.crosstab(school_perf_reliable['school_isp'], school_perf_reliable['service_tier'])
print(tier_by_isp.head(10).to_string())


### Weekly IQB heatmap

In [ ]:
# =============================================================================
# WEEKLY IQB HEATMAP — per-school IQB score over time (p50 & p95)
# =============================================================================
# One row per school, one column per ISO week; cell = IQB score that week.
# Same methodology as the per-school scores: school-hours, complete-case,
# polarity-correct percentiles. Weeks with < WEEKLY_MIN_N measurements are blank.
from matplotlib.colors import LinearSegmentedColormap

WEEKLY_MIN_N = 5            # min measurements per school-week to score
WEEKLY_TOP_SCHOOLS = 40     # show the N schools with most measurements (readability)

_GIGA_RAG = LinearSegmentedColormap.from_list("giga_rag", [GIGA_BAD, GIGA_MODERATE, GIGA_GOOD])
_GIGA_RAG.set_bad("#f4f4f4")   # weeks with no/insufficient data -> light grey

wk = m.copy()
if "measurement_time_window" in wk.columns:
    wk = wk[wk["measurement_time_window"] == "school_hours"]
_need = ["download_speed", "upload_speed", "latency", "loss_rate"]
wk = wk.dropna(subset=_need)
wk["week"] = pd.to_datetime(wk["date"]).dt.to_period("W").apply(lambda p: p.start_time.date())

_names = m.drop_duplicates("school_id_giga").set_index("school_id_giga")["school_name"]
_top = wk["school_id_giga"].value_counts().head(WEEKLY_TOP_SCHOOLS).index
wk = wk[wk["school_id_giga"].isin(_top)]
_g = wk.groupby(["school_id_giga", "week"])

for pct in IQB_DISPLAY_PCTS:
    q_hi, q_lo = pct / 100.0, 1 - pct / 100.0
    agg = pd.DataFrame({
        "n":   _g.size(),
        "dl":  _g["download_speed"].quantile(q_hi),
        "ul":  _g["upload_speed"].quantile(q_hi),
        "lat": _g["latency"].quantile(q_lo),
        "loss": _g["loss_rate"].quantile(q_lo),
    }).reset_index()
    agg = agg[agg["n"] >= WEEKLY_MIN_N].copy()
    agg["iqb"] = agg.apply(lambda r: calculate_iqb_score({"m-lab": {
        "download_throughput_mbps": r["dl"], "upload_throughput_mbps": r["ul"],
        "latency_ms": r["lat"], "packet_loss": r["loss"]}}), axis=1)

    pivot = agg.pivot(index="school_id_giga", columns="week", values="iqb")
    pivot = pivot.loc[pivot.mean(axis=1).sort_values(ascending=False).index]   # best schools on top

    fig, ax = plt.subplots(figsize=(min(22, 2 + 0.5 * pivot.shape[1]), 2 + 0.32 * len(pivot)))
    im = ax.imshow(pivot.values, aspect="auto", cmap=_GIGA_RAG, vmin=0, vmax=1)
    ax.set_xticks(range(pivot.shape[1]))
    ax.set_xticklabels([str(c) for c in pivot.columns], rotation=90, fontsize=7)
    ax.set_yticks(range(len(pivot)))
    ax.set_yticklabels([str(_names.get(s, s))[:34] for s in pivot.index], fontsize=7)
    ax.set_title(f"Weekly IQB score per school — p{pct} — {COUNTRY_NAME}")
    cbar = plt.colorbar(im, ax=ax, fraction=0.015, pad=0.01)
    cbar.set_label(f"IQB score (p{pct})")
    plt.tight_layout(); plt.show()

print(f"Weekly IQB heatmap: top {min(WEEKLY_TOP_SCHOOLS, len(_top))} schools by volume, "
      f"weeks with >= {WEEKLY_MIN_N} school-hours measurements; percentiles {IQB_DISPLAY_PCTS}.")


### Provider switching

In [ ]:
# =============================================================================
# PROVIDER SWITCHING & CONCURRENT PROVIDERS — ISP Transitions Over Time
# =============================================================================

# Identify schools with multiple providers (excluding noise)
MIN_MEASUREMENTS_PER_PROVIDER = 5  # Filter out 1-2 stray measurements

# Count providers per school
school_provider_counts = m.groupby(['school_id_giga', 'isp_mapped']).agg({
    'measurement_id': 'count',
    'school_name': 'first',
    'admin2': 'first',
    'connectivity_type_govt': 'first'
}).reset_index()

school_provider_counts.columns = ['school_id_giga', 'isp_mapped', 'n_measurements', 'school_name', 'admin2', 'connectivity_type']

# Find schools with multiple providers (both with >= MIN_MEASUREMENTS)
schools_with_multiple_isps = school_provider_counts.groupby('school_id_giga').agg({
    'isp_mapped': 'nunique',
    'n_measurements': lambda x: (x >= MIN_MEASUREMENTS_PER_PROVIDER).sum(),  # Count providers with sufficient data
    'school_name': 'first',
    'admin2': 'first'
}).reset_index()

schools_with_multiple_isps.columns = ['school_id_giga', 'total_providers', 'providers_with_substance', 'school_name', 'admin2']

# Filter: schools with 2+ providers, both with >= MIN_MEASUREMENTS (meaningful overlap or switching)
meaningful_switches = schools_with_multiple_isps[schools_with_multiple_isps['providers_with_substance'] >= 2].copy()

print(f"{'='*80}")
print(f"PROVIDER SWITCHING & CONCURRENT PROVIDERS (>= {MIN_MEASUREMENTS_PER_PROVIDER} measurements per provider)")
print(f"{'='*80}")
print(f"\nSchools with multiple ISPs (meaningful overlap/switching): {len(meaningful_switches)}")

if len(meaningful_switches) > 0:
    print(f"\nExamples of Multi-ISP Schools:")
    print(meaningful_switches.sort_values('providers_with_substance', ascending=False)[['school_name', 'admin2', 'total_providers', 'providers_with_substance']].head(10).to_string(index=False))
    
    # Get details for top examples
    top_switching_schools = meaningful_switches.sort_values('providers_with_substance', ascending=False).head(5)['school_id_giga'].values
    
    # Create timeseries for each
    fig, axes = plt.subplots(len(top_switching_schools), 1, figsize=(14, 4*len(top_switching_schools)))
    if len(top_switching_schools) == 1:
        axes = [axes]
    
    # Define color map for providers - use distinct colors
    providers_in_data = m['isp_mapped'].unique()
    # Use tab10 or Set1 for more contrast, or define explicit colors
    if len(providers_in_data) <= 10:
        colors = plt.cm.tab10(np.linspace(0, 1, len(providers_in_data)))
    else:
        colors = plt.cm.tab20(np.linspace(0, 1, len(providers_in_data)))
    provider_colors = {prov: colors[i] for i, prov in enumerate(providers_in_data)}
    
    for idx, school_id in enumerate(top_switching_schools):
        school_data = m[m['school_id_giga'] == school_id].copy()
        school_data = school_data.sort_values('timestamplocal')
        school_name = school_data['school_name'].iloc[0]
        
        # Get provider breakdown for this school
        provider_breakdown = school_provider_counts[school_provider_counts['school_id_giga'] == school_id]
        active_providers = provider_breakdown[provider_breakdown['n_measurements'] >= MIN_MEASUREMENTS_PER_PROVIDER]['isp_mapped'].tolist()
        
        ax = axes[idx]
        
        # Plot download speed over time, colored by provider with lines and points
        for provider in active_providers:
            provider_data = school_data[school_data['isp_mapped'] == provider].sort_values('timestamplocal')
            ax.plot(provider_data['timestamplocal'], provider_data['download_speed'],
                   label=f"{provider} (n={len(provider_data)})",
                   color=provider_colors.get(provider, '#gray'),
                   linewidth=2, marker='o', markersize=6, alpha=0.8, zorder=2)
        
        ax.set_ylabel('Download Speed (Mbps)', fontweight='bold')
        ax.set_title(f"{school_name} — Provider Transitions Over Time\n({', '.join(active_providers)})", 
                    fontweight='bold', fontsize=11)
        ax.legend(loc='best', fontsize=9)
        ax.grid(alpha=0.3)
        ax.tick_params(axis='x', rotation=45)
    
    if len(top_switching_schools) > 1:
        axes[-1].set_xlabel('Date', fontweight='bold')
    else:
        axes[0].set_xlabel('Date', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n💡 INSIGHT:")
    print(f"   Schools switching providers may indicate:")
    print(f"   • Network upgrades (old provider → new provider)")
    print(f"   • Redundancy/failover (concurrent for resilience)")
    print(f"   • Performance issues (switching to alternative)")
else:
    print(f"\n⚠️  No schools with meaningful multi-ISP overlap (2+ providers with {MIN_MEASUREMENTS_PER_PROVIDER}+ measurements each)")


### Traceroute analysis

In [ ]:
# Opt-in: traceroute needs `gcloud auth application-default login` + M-Lab BigQuery (costs $).
RUN_TRACEROUTE = False   # set True to run
if RUN_TRACEROUTE:
    try:
        # =============================================================================
        # TRACEROUTE ANALYSIS — M-Lab Scamper via BigQuery
        #
        # Source: M-Lab `measurement-lab.base_tables.scamper1` (BigQuery)
        # Join:   UUIDs from m['uuid'] (Giga Meter) → BQ scamper traceroute records
        # Tools:  fetch_mlab_data.fetch_mlab_traceroutes_bq()
        #         mlab_traceroute_analyzer.run_analysis()
        #
        # PREREQUISITE — run once: gcloud auth application-default login
        # NETWORK_SEGMENTS in mlab_traceroute_analyzer.py configured for Moldova.
        # =============================================================================

        import sys, importlib
        from pathlib import Path

        _monitoring_dir = Path.home() / 'Documents/Giga/Delivery/Monitoring'
        if str(_monitoring_dir) not in sys.path:
            sys.path.insert(0, str(_monitoring_dir))

        import fetch_mlab_data as fmd
        import mlab_traceroute_analyzer as mta
        importlib.reload(fmd); importlib.reload(mta)

        # ── Parameters ───────────────────────────────────────────────────────────────
        BQ_PROJECT     = 'measurement-lab'
        TR_SAMPLE_DAYS = 7      # days back from latest measurement to pull UUIDs for
        TR_MAX_UUIDS   = 500    # BQ cost cap — raise for full runs
        TR_ISP_FILTER  = ['Moldtelecom', 'StarNet']  # None = all ISPs
        TR_SOURCE      = 'standard'  # 'standard' = ndt.scamper1 (most countries incl. Moldova)
                                      # 'autojoin' = autojoin_autoload_v2_ndt.scamper2_union (BYOS, e.g. Mongolia)

        # ── Date window from loaded measurements ─────────────────────────────────────
        tr_end   = pd.to_datetime(m['timestamplocal']).dt.date.max()
        tr_start = tr_end - pd.Timedelta(days=TR_SAMPLE_DAYS)

        mask = (
            (pd.to_datetime(m['timestamplocal']).dt.date >= tr_start) &
            (pd.to_datetime(m['timestamplocal']).dt.date <= tr_end)
        )

        # ── ISP filter ───────────────────────────────────────────────────────────────
        if TR_ISP_FILTER and 'isp_mapped' in m.columns:
            mask = mask & m['isp_mapped'].isin(TR_ISP_FILTER)
            print(f'ISP filter: {TR_ISP_FILTER}')

        uuids = m.loc[mask, 'uuid'].dropna().unique().tolist()[:TR_MAX_UUIDS]
        print(f'Date window : {tr_start} → {tr_end}')
        print(f'UUIDs       : {len(uuids):,} (cap {TR_MAX_UUIDS})')

        if not uuids:
            print('No UUIDs — widen TR_SAMPLE_DAYS, check m["uuid"], or relax TR_ISP_FILTER.')
        else:
            traces_df = fmd.fetch_mlab_traceroutes_bq(
                uuids=uuids,
                start_date=str(tr_start),
                end_date=str(tr_end),
                bq_project=BQ_PROJECT,
                source=TR_SOURCE,
            )
            print(f'Traceroute records: {len(traces_df):,}')

            if traces_df.empty:
                print('No records returned. Check: UUIDs are download UUIDs; date range matches BQ partition.')
            else:
                ndt_df = (
                    m.loc[mask, ['uuid', 'school_id_giga', 'school_name', 'isp_mapped',
                                 'download_speed', 'upload_speed', 'latency']]
                    .drop_duplicates(subset='uuid')
                    .rename(columns={'school_id_giga': 'giga_id_school'})
                )

                results = mta.run_analysis(
                    input_df=traces_df,
                    ndt_df=ndt_df,
                    skip_config_prompt=True,
                    charts=True,
                    export_csv=False,
                )

                if results:
                    summary = results.get('summary')
                    if summary is not None:
                        print('\nPer-school traceroute summary:')
                        display(summary)
                    hops_df = results.get('hops_df')
                    if hops_df is not None and not hops_df.empty:
                        print('\nHop segment distribution:')
                        print(hops_df['segment'].value_counts().to_string())
                        if TR_ISP_FILTER:
                            print('\nSegment distribution by ISP:')
                            isp_seg = hops_df.groupby(['isp_mapped', 'segment']).size().unstack(fill_value=0) \
                                if 'isp_mapped' in hops_df.columns else None
                            if isp_seg is not None:
                                print(isp_seg.to_string())
    except Exception as _e:
        print(f'Traceroute analysis failed (needs gcloud auth + BigQuery): {_e}')
else:
    print('⏭  Traceroute skipped — set RUN_TRACEROUTE=True (needs gcloud auth + M-Lab BigQuery).')


### Service tier by admin2

In [ ]:
# =============================================================================
# SERVICE TIER ANALYSIS BY ADMIN2 (District)
# =============================================================================

# Per-school statistics grouped by admin2
admin2_perf = m_geo.groupby(['school_id_giga', 'admin2']).agg({
    'download_speed': 'median',
    'upload_speed': 'median',
    'latency': 'median',
    'loss_rate': 'median'
}).reset_index()

# Classify service tier
admin2_perf['service_tier'] = admin2_perf.apply(
    lambda row: classify_service_level(row['download_speed'], row['upload_speed'], row['latency']),
    axis=1
)

# Aggregate to admin2 level
admin2_summary = admin2_perf.groupby('admin2').agg({
    'school_id_giga': 'count',
    'download_speed': ['median', 'mean'],
    'upload_speed': ['median', 'mean'],
    'latency': ['median', 'mean'],
    'loss_rate': 'median'
}).round(4)

admin2_summary.columns = ['Schools', 'DL_Median', 'DL_Mean', 'UL_Median', 'UL_Mean', 'Lat_Median', 'Lat_Mean', 'Loss_Median']
admin2_summary['Loss_Median'] = (admin2_summary['Loss_Median'] * 100).round(2)
admin2_summary = admin2_summary.sort_values('Schools', ascending=False).head(15)

# Count schools per tier
admin2_tier_counts = pd.crosstab(admin2_perf['admin2'], admin2_perf['service_tier'])
admin2_tier_pct = admin2_tier_counts.div(admin2_tier_counts.sum(axis=1), axis=0).round(3) * 100

print(f"\n{'='*80}")
print("SERVICE TIER DISTRIBUTION BY ADMIN2 (DISTRICT) — TOP 15")
print(f"{'='*80}")
print(f"\n{admin2_summary.to_string()}")

print(f"\n\nTier distribution by district (% of schools):")
print(f"{admin2_tier_pct.head(15).to_string()}")

# Visualization: Top 12 districts by school count
top_districts = admin2_summary.head(12).index.tolist()
admin2_for_plot = admin2_perf[admin2_perf['admin2'].isin(top_districts)]

fig, ax = plt.subplots(figsize=(12, 8))

# Tier distribution stacked bar for top districts
admin2_tier_plot = pd.crosstab(admin2_for_plot['admin2'], admin2_for_plot['service_tier'])
admin2_tier_plot = admin2_tier_plot[[col for col in tier_order if col in admin2_tier_plot.columns]]
admin2_tier_plot = admin2_tier_plot.loc[top_districts]  # Reorder by school count

admin2_tier_plot.plot(
    kind='barh', stacked=True, ax=ax,
    color=['#ed1c24', '#ffc93d', '#33ff8f', '#00d661'],
    edgecolor='black', alpha=0.85
)
ax.set_xlabel('Number of Schools', fontweight='bold')
ax.set_title('Service Tier Distribution by District (Top 12 by School Count)', fontweight='bold', fontsize=12)
ax.legend(title='Service Tier', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

### Service tier by education

In [ ]:
# =============================================================================
# SERVICE TIER ANALYSIS BY EDUCATION LEVEL
# =============================================================================

# Per-school statistics grouped by education_level
edu_perf = m_geo.groupby(['school_id_giga', 'education_level']).agg({
    'download_speed': 'median',
    'upload_speed': 'median',
    'latency': 'median',
    'loss_rate': 'median'
}).reset_index()

# Classify service tier
edu_perf['service_tier'] = edu_perf.apply(
    lambda row: classify_service_level(row['download_speed'], row['upload_speed'], row['latency']),
    axis=1
)

# Aggregate to education level
edu_summary = edu_perf.groupby('education_level').agg({
    'school_id_giga': 'count',
    'download_speed': ['median', 'mean'],
    'upload_speed': ['median', 'mean'],
    'latency': ['median', 'mean'],
    'loss_rate': 'median'
}).round(4)

edu_summary.columns = ['Schools', 'DL_Median', 'DL_Mean', 'UL_Median', 'UL_Mean', 'Lat_Median', 'Lat_Mean', 'Loss_Median']
edu_summary['Loss_Median'] = (edu_summary['Loss_Median'] * 100).round(2)

# Reorder education levels logically
edu_order = ['Pre-Primary', 'Primary', 'Secondary', 'Unknown']
_edu_ordered = [e for e in edu_order if e in edu_summary.index]
edu_summary = edu_summary.reindex(_edu_ordered if _edu_ordered else edu_summary.index)

# Count schools per tier
edu_tier_counts = pd.crosstab(edu_perf['education_level'], edu_perf['service_tier'])
edu_tier_pct = edu_tier_counts.div(edu_tier_counts.sum(axis=1), axis=0).round(3) * 100
_edu_tier_ordered = [e for e in edu_order if e in edu_tier_pct.index]
edu_tier_pct = edu_tier_pct.reindex(_edu_tier_ordered if _edu_tier_ordered else edu_tier_pct.index)

print(f"\n{'='*80}")
print("SERVICE TIER DISTRIBUTION BY EDUCATION LEVEL")
print(f"{'='*80}")
print(f"\n{edu_summary.to_string()}")

print(f"\n\nTier distribution (% of schools per education level):")
print(f"{edu_tier_pct.to_string()}")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Download speed by education level
_edu_colors = ['#7eb0ff', '#0050e6', '#277aff', '#989898', '#6f6f6f', '#0050e6']
edu_summary['DL_Median'].plot(
    kind='bar', ax=axes[0], color=_edu_colors[:len(edu_summary)],
    edgecolor='black', alpha=0.8
)
axes[0].axhline(TIER_THRESHOLD_2, color='#525252', linestyle='--', linewidth=2, label=f'TIER 2 threshold ({TIER_THRESHOLD_2}M)')
axes[0].axhline(TIER_THRESHOLD_3, color='#277aff', linestyle='--', linewidth=2, label=f'TIER 3 threshold ({TIER_THRESHOLD_3}M)')
axes[0].set_ylabel('Median Download Speed (Mbps)', fontweight='bold')
axes[0].set_title('Median Download Speed by Education Level', fontweight='bold')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Tier distribution
edu_tier_pct_ordered = edu_tier_pct[[col for col in tier_order if col in edu_tier_pct.columns]]
edu_tier_pct_ordered.plot(
    kind='bar', stacked=True, ax=axes[1],
    color=['#ed1c24', '#ffc93d', '#33ff8f', '#00d661'],
    edgecolor='black', alpha=0.85
)
axes[1].set_ylabel('Percentage of Schools (%)', fontweight='bold')
axes[1].set_title('Service Tier Distribution by Education Level', fontweight='bold')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right')
axes[1].legend(title='Service Tier', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

### Manual-measurement bias

In [ ]:
# =============================================================================
# MANUAL MEASUREMENTS — Timing & Bias Analysis
# =============================================================================

# Check if manual measurement data exists
if 'notes' in m_original.columns:
    m_manual = m_original.copy()
    m_manual['measurement_type'] = m_manual['notes'].fillna('Unknown')
    manual_keywords = ['manual', 'test', 'field', 'visitor', 'user']
    m_manual['is_manual'] = m_manual['measurement_type'].str.lower().str.contains(
        '|'.join(manual_keywords), na=False
    )
    manual_count = m_manual['is_manual'].sum()
    
    print(f"\n{'='*80}")
    print("MANUAL MEASUREMENTS — TIMING & BIAS ANALYSIS")
    print(f"{'='*80}")
    print(f"Manual measurements detected: {manual_count}")
    
    if manual_count > 0:
        manual_data = m_manual[m_manual['is_manual']].copy()
        manual_data['hour'] = pd.to_datetime(manual_data['timestamplocal']).dt.hour
        hour_counts = manual_data['hour'].value_counts().sort_index()
        
        print(f"Peak testing hour: {hour_counts.idxmax()}:00 ({hour_counts.max()} measurements)")

        # The point of this section is bias: do manual tests read differently from
        # automated ones? That is a within-school contrast, so pair each school with
        # itself (manual vs automated) and test the shift nonparametrically.
        if 'school_id_giga' in m_manual.columns:
            m_manual['meas_method'] = np.where(m_manual['is_manual'], 'Manual', 'Automated')
            print(f"\nPer-school Manual vs Automated shift (paired; each school its own control):")
            for _metric, _better in [('download_speed', 'up'), ('upload_speed', 'up'), ('latency', 'down')]:
                if _metric not in m_manual.columns:
                    continue
                _res = paired_shift_test(
                    m_manual, unit_col='school_id_giga', group_col='meas_method',
                    value_col=_metric, group_a='Manual', group_b='Automated', min_per_cell=3)
                print(format_shift_result(_res, label=f'  {_metric}', better=_better))
else:
    print(f"\n{'='*80}")
    print("MANUAL MEASUREMENTS — Skipped")
    print(f"{'='*80}")
    print("⚠️  No measurement notes available for this dataset")


### WiFi vs Ethernet

In [ ]:
# =============================================================================
# WiFi vs ETHERNET CONNECTION TYPE
# =============================================================================

# Proxy: measurement has WiFi data (ssid or quality not null) → WiFi; else → Ethernet/unknown
wifi_indicator = m_original['detected_wifi_ssid'].notna() | m_original['detected_wifi_quality'].notna()
m_original['connection_type'] = wifi_indicator.map({True: 'WiFi', False: 'Ethernet / Unknown'})

conn_counts = m_original['connection_type'].value_counts()
total = len(m_original)

print(f"\n{'='*80}")
print('CONNECTION TYPE — WiFi vs Ethernet')
print(f"{'='*80}")
for ctype, n in conn_counts.items():
    print(f'  {ctype:<25} {n:>6,}  ({n/total*100:.1f}% of measurements)')

# Per-school: majority connection type
school_conn = (
    m_original.groupby('school_id_giga')['connection_type']
    .agg(lambda x: x.value_counts().index[0])
    .reset_index(name='dominant_connection')
)
school_conn_counts = school_conn['dominant_connection'].value_counts()

print(f"\nSchools by dominant connection type:")
for ctype, n in school_conn_counts.items():
    print(f'  {ctype:<25} {n:>4} schools')

# Speed / latency / loss by connection type
print(f"\nMedian metrics by connection type:")
for ctype, grp in m_original.groupby('connection_type'):
    lr = grp['loss_rate'].median() * 100 if 'loss_rate' in grp.columns else float('nan')
    print(f'  {ctype:<25} DL {grp["download_speed"].median():.1f} Mbps  '
          f'Lat {grp["latency"].median():.0f} ms  Loss {lr:.2f}%  (n={len(grp):,})')

# Per-school unpaired significance: does connection type really shift performance,
# or is the gap noise? Aggregate to one median per school within each type (a school
# may appear in both), then Mann-Whitney + Cliff's delta + bootstrap CI — the
# per-measurement gap above is not a valid test (thousands of rows, few schools).
if 'school_id_giga' in m_original.columns:
    print(f"\nPer-school WiFi vs Ethernet shift (Mann-Whitney, bootstrap CI):")
    for _metric, _better in [('download_speed', 'up'), ('upload_speed', 'up'), ('latency', 'down')]:
        if _metric not in m_original.columns:
            continue
        _res = two_group_shift_test(
            m_original, unit_col='school_id_giga', group_col='connection_type',
            value_col=_metric, group_a='WiFi', group_b='Ethernet / Unknown', min_per_cell=3)
        print(format_shift_result(_res, label=f'  {_metric}', better=_better))


# =============================================================================
# WiFi SIGNAL STRENGTH & SPEED CORRELATION ANALYSIS
# =============================================================================

wifi_strength_cols = ['detected_wifi_quality', 'detected_wifi_signal', 'wifi_quality', 'wifi_signal', 'wifi_rssi', 'signal', 'quality']
available_strength_col = next((c for c in wifi_strength_cols if c in m_original.columns), None)

if available_strength_col:
    print(f"\n{'='*80}")
    print("WiFi SIGNAL STRENGTH — CORRELATION WITH SPEED, LATENCY & PACKET LOSS")
    print(f"{'='*80}")
    print(f"\nAnalyzing WiFi strength field: {available_strength_col}")

    _extra_cols = [c for c in ['detected_wifi_model', 'connectivity_type_govt', 'loss_rate'] if c in m_original.columns]
    wifi_analysis = m_original[
        ['download_speed', 'upload_speed', 'latency', available_strength_col] + _extra_cols
    ].dropna(subset=['download_speed', available_strength_col])
    # Filter latency to valid range (same as preprocessing)
    wifi_analysis = wifi_analysis[(wifi_analysis['latency'] > 0) & (wifi_analysis['latency'] < LATENCY_OUTLIER_THRESHOLD)]

    print(f"Measurements with WiFi signal data: {len(wifi_analysis):,}")

    corr_dl  = wifi_analysis['download_speed'].corr(wifi_analysis[available_strength_col])
    corr_ul  = wifi_analysis['upload_speed'].corr(wifi_analysis[available_strength_col])
    corr_lat = wifi_analysis['latency'].corr(wifi_analysis[available_strength_col])
    corr_loss = wifi_analysis['loss_rate'].corr(wifi_analysis[available_strength_col]) if 'loss_rate' in wifi_analysis.columns else float('nan')

    print(f"\nOverall Correlation with {available_strength_col}:")
    print(f"  Download Speed: {corr_dl:.3f}")
    print(f"  Upload Speed:   {corr_ul:.3f}")
    print(f"  Latency:        {corr_lat:.3f}")
    print(f"  Packet Loss:    {corr_loss:.3f}")

    if 'connectivity_type_govt' in wifi_analysis.columns:
        print(f"\nCorrelation by Connectivity Type:")
        for conn_type in wifi_analysis['connectivity_type_govt'].dropna().unique():
            subset = wifi_analysis[wifi_analysis['connectivity_type_govt'] == conn_type]
            if len(subset) > 10:
                print(f"\n  {conn_type} (n={len(subset):,}):")
                print(f"    Download: {subset['download_speed'].corr(subset[available_strength_col]):.3f}")
                print(f"    Latency:  {subset['latency'].corr(subset[available_strength_col]):.3f}")
                if 'loss_rate' in subset.columns:
                    print(f"    Loss:     {subset['loss_rate'].corr(subset[available_strength_col]):.3f}")

    # Normalise signal to 0-100
    signal_vals = wifi_analysis[available_strength_col]
    signal_min, signal_max = signal_vals.min(), signal_vals.max()
    if signal_max > 100:
        wifi_analysis = wifi_analysis.copy()
        wifi_analysis['signal_normalized'] = ((signal_vals - signal_min) / (signal_max - signal_min) * 100).clip(0, 100)
    else:
        wifi_analysis = wifi_analysis.copy()
        wifi_analysis['signal_normalized'] = signal_vals.clip(0, 100)

    wifi_analysis['signal_bin'] = pd.cut(wifi_analysis['signal_normalized'],
                                          bins=[0, 25, 50, 75, 100],
                                          labels=['Weak (0-25%)', 'Fair (25-50%)', 'Good (50-75%)', 'Excellent (75-100%)'])

    _agg = {
        'download_speed': ['median', 'mean', 'count'],
        'upload_speed': ['median', 'mean'],
        'latency': ['median', 'mean'],
    }
    if 'loss_rate' in wifi_analysis.columns:
        _agg['loss_rate'] = 'median'

    signal_perf = wifi_analysis.groupby('signal_bin').agg(_agg).round(4)

    if 'loss_rate' in wifi_analysis.columns:
        signal_perf.columns = ['DL_Median', 'DL_Mean', 'Samples', 'UL_Median', 'UL_Mean', 'Lat_Median', 'Lat_Mean', 'Loss_Median']
        signal_perf['Loss_Median'] = (signal_perf['Loss_Median'] * 100).round(3)
    else:
        signal_perf.columns = ['DL_Median', 'DL_Mean', 'Samples', 'UL_Median', 'UL_Mean', 'Lat_Median', 'Lat_Mean']

    print(f"\n\nPerformance by WiFi Signal Strength:")
    print(signal_perf.to_string())

    # Visualization — 2x3: DL scatter, UL scatter, Latency scatter, Loss scatter, DL by bin, sample count
    _has_loss = 'loss_rate' in wifi_analysis.columns
    fig, axes = plt.subplots(2, 3 if _has_loss else 2, figsize=(18 if _has_loss else 14, 10))

    x_line = np.linspace(wifi_analysis['signal_normalized'].min(), wifi_analysis['signal_normalized'].max(), 100)

    def _scatter_with_trend(ax, y_col, color, ylabel, title, corr_val):
        valid = wifi_analysis.dropna(subset=[y_col, 'signal_normalized'])
        ax.scatter(valid['signal_normalized'], valid[y_col], alpha=0.3, s=15, color=color, edgecolor='none')
        z = np.polyfit(valid['signal_normalized'], valid[y_col], 1)
        ax.plot(x_line, np.poly1d(z)(x_line), 'r-', linewidth=2, label=f'r={corr_val:.2f}')
        ax.set_xlabel(f'{available_strength_col} (Normalized %)', fontweight='bold')
        ax.set_ylabel(ylabel, fontweight='bold')
        ax.set_title(title, fontweight='bold')
        ax.legend(); ax.grid(alpha=0.3)

    _scatter_with_trend(axes[0, 0], 'download_speed', '#0050e6', 'Download Speed (Mbps)', 'Signal vs Download', corr_dl)
    _scatter_with_trend(axes[0, 1], 'upload_speed',   '#277aff', 'Upload Speed (Mbps)',   'Signal vs Upload',   corr_ul)
    _scatter_with_trend(axes[0, 2 if _has_loss else 0], 'latency', '#6f6f6f', 'Latency (ms)', 'Signal vs Latency', corr_lat)

    if _has_loss:
        wifi_analysis['loss_pct'] = wifi_analysis['loss_rate'] * 100
        _scatter_with_trend(axes[1, 0], 'loss_pct', '#989898', 'Loss Rate (%)', 'Signal vs Packet Loss', corr_loss)
        _bar_ax_dl  = axes[1, 1]
        _bar_ax_n   = axes[1, 2]
    else:
        _bar_ax_dl  = axes[1, 0]
        _bar_ax_n   = axes[1, 1]

    _bin_colors = ['#ed1c24', '#ffc93d', '#33ff8f', '#00d661']
    signal_perf['DL_Median'].plot(kind='bar', ax=_bar_ax_dl,
                                   color=_bin_colors[:len(signal_perf)], edgecolor='black', alpha=0.8)
    _bar_ax_dl.set_ylabel('Median Download Speed (Mbps)', fontweight='bold')
    _bar_ax_dl.set_title('Median Download Speed by WiFi Signal Bin', fontweight='bold')
    _bar_ax_dl.set_xticklabels(_bar_ax_dl.get_xticklabels(), rotation=45, ha='right')
    _bar_ax_dl.grid(axis='y', alpha=0.3)

    signal_perf['Samples'].plot(kind='bar', ax=_bar_ax_n, color='#7eb0ff', alpha=0.8, edgecolor='black')
    _bar_ax_n.set_ylabel('Number of Measurements', fontweight='bold')
    _bar_ax_n.set_title('Measurement Count by WiFi Signal Bin', fontweight='bold')
    _bar_ax_n.set_xticklabels(_bar_ax_n.get_xticklabels(), rotation=45, ha='right')
    _bar_ax_n.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.show()

    print(f"\n\nKEY INSIGHT:")
    if abs(corr_dl) > 0.5:
        print(f"  Strong correlation (r={corr_dl:.2f}) between WiFi signal and download speed")
    elif abs(corr_dl) > 0.3:
        print(f"  Moderate correlation (r={corr_dl:.2f}) between WiFi signal and download speed")
    else:
        print(f"  Weak correlation (r={corr_dl:.2f}) between WiFi signal and download speed")
        print(f"  → ISP capacity, congestion, or device limitations likely dominate")
    if _has_loss and abs(corr_loss) > 0.2:
        direction = "higher signal → lower loss" if corr_loss < 0 else "higher signal → higher loss (unusual)"
        print(f"  Loss rate correlation (r={corr_loss:.2f}): {direction}")

else:
    print(f"\n  No WiFi signal strength data available in measurements")
    print(f"  Expected one of: {wifi_strength_cols}")


### Per-school time series

In [ ]:
# =============================================================================
# PER-SCHOOL TIME SERIES — Performance Over Time with Metadata
# =============================================================================

# Prepare time series data
ts_data = m_original.copy()
ts_data['timestamp'] = pd.to_datetime(ts_data['timestamplocal'])
ts_data = ts_data.sort_values('timestamp')

# Select top schools by measurement count
top_schools = ts_data.groupby('school_id_giga').size().nlargest(12).index.tolist()
ts_subset = ts_data[ts_data['school_id_giga'].isin(top_schools)]

print(f"\n{'='*80}")
print("PER-SCHOOL TIME SERIES ANALYSIS")
print(f"{'='*80}")
print(f"\nShowing top {len(top_schools)} schools by measurement count")
print(f"Total measurements displayed: {len(ts_subset):,}")

# Create time series plots (3x4 grid for 12 schools)
n_schools = len(top_schools)
n_cols = 4
n_rows = (n_schools + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 5 * n_rows))
axes = axes.flatten() if n_schools > 1 else [axes]

for idx, school_id in enumerate(top_schools):
    school_data = ts_subset[ts_subset['school_id_giga'] == school_id].sort_values('timestamp')
    ax = axes[idx]
    
    if len(school_data) == 0:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center')
        continue
    
    # Get metadata
    school_name = school_data['school_name_master'].iloc[0] if 'school_name_master' in school_data.columns else school_data['school_name_measurement'].iloc[0] if 'school_name_measurement' in school_data.columns else school_data['school_name'].iloc[0]
    connectivity = school_data['connectivity_type_govt_master'] if 'connectivity_type_govt_master' in school_data.columns else school_data['connectivity_type_govt'].iloc[0]
    def _first_of(df, *cols):
        for _c in cols:
            if _c in df.columns:
                v = df[_c].iloc[0]
                return v
        return np.nan
    provider = _first_of(school_data, 'connectivity_provider_master', 'connectivity_provider')
    education = school_data['education_level'].iloc[0]
    admin1 = school_data['admin1_master'] if 'admin1_master' in school_data.columns else school_data['admin1'].iloc[0]
    admin2 = school_data['admin2_master'] if 'admin2_master' in school_data.columns else school_data['admin2'].iloc[0]
    n_measurements = len(school_data)
    
    # Plot time series
    ax.plot(school_data['timestamp'], school_data['download_speed'], 
            marker='o', linestyle='-', linewidth=2, markersize=4, 
            label='Download', color='#0050e6', alpha=0.7)
    ax.plot(school_data['timestamp'], school_data['upload_speed'], 
            marker='s', linestyle='-', linewidth=2, markersize=4, 
            label='Upload', color='#277aff', alpha=0.7)
    
    
    # Format title with metadata
    title = f"{school_name[:25]}\n"
    dl_med = school_data['download_speed'].median()
    ul_med = school_data['upload_speed'].median()
    lat_med = school_data['latency'].median()
    school_tier = classify_service_level(dl_med, ul_med, lat_med)
    
    # Debug: print Kalukanya metrics
    if 'KALUKANYA' in school_name.upper():
        print(f"\n🔍 DEBUG {school_name}: DL={dl_med:.2f}, UL={ul_med:.2f}, Lat={lat_med:.2f} → {school_tier}")
    title += f"{connectivity} / {provider if pd.notna(provider) else 'Unknown'} | {education} | {admin2}\n"
    title += f"{school_tier}\n"
    title += f"(n={n_measurements}, {admin1})"
    
    ax.set_title(title, fontsize=9, fontweight='bold')
    ax.set_ylabel('Speed (Mbps)', fontsize=8)
    ax.set_xlabel('Time', fontsize=8)
    ax.tick_params(axis='both', labelsize=7)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=7, loc='upper left')
    
    # Rotate x-axis labels
    for label in ax.get_xticklabels():
        label.set_rotation(45)
        label.set_ha('right')

# Hide unused subplots
for idx in range(len(top_schools), len(axes)):
    axes[idx].axis('off')

plt.suptitle('Per-School Performance Time Series (Top 12 by Measurement Count)', 
             fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

# Summary table of top schools
print(f"\n\nTop Schools Summary:")
school_summary = []
for school_id in top_schools:
    school_data = ts_subset[ts_subset['school_id_giga'] == school_id]
    school_name = school_data['school_name_master'].iloc[0] if 'school_name_master' in school_data.columns else school_data['school_name_measurement'].iloc[0] if 'school_name_measurement' in school_data.columns else school_data['school_name'].iloc[0]
    school_summary.append({
        'School': school_name[:30],
        'Connectivity': school_data['connectivity_type_govt_master'] if 'connectivity_type_govt_master' in school_data.columns else school_data['connectivity_type_govt'].iloc[0],
        'Provider': _first_of(school_data, 'connectivity_provider_master', 'connectivity_provider'),
        'Education': school_data['education_level'].iloc[0],
        'Admin2': school_data['admin2_master'] if 'admin2_master' in school_data.columns else school_data['admin2'].iloc[0],
        'Measurements': len(school_data),
        'DL_Median': school_data['download_speed'].median(),
        'UL_Median': school_data['upload_speed'].median(),
        'Lat_Median': school_data['latency'].median()
    })

school_summary_df = pd.DataFrame(school_summary).round(2)
print(school_summary_df.to_string(index=False))

# Performance trend analysis
print(f"\n\nPerformance Trends (first vs last measurement):")
trends = []
for school_id in top_schools:
    school_data = ts_subset[ts_subset['school_id_giga'] == school_id].sort_values('timestamp')
    if len(school_data) > 1:
        first_dl = school_data['download_speed'].iloc[0]
        last_dl = school_data['download_speed'].iloc[-1]
        change = last_dl - first_dl
        school_name = school_data['school_name_master'].iloc[0] if 'school_name_master' in school_data.columns else school_data['school_name_measurement'].iloc[0] if 'school_name_measurement' in school_data.columns else school_data['school_name'].iloc[0]
        
        trends.append({
            'School': school_name[:30],
            'First_DL': first_dl,
            'Last_DL': last_dl,
            'Change_Mbps': change,
            'Change_%': (change / first_dl * 100) if first_dl > 0 else 0,
            'Days_Span': (school_data['timestamp'].max() - school_data['timestamp'].min()).days
        })

trends_df = pd.DataFrame(trends).sort_values('Change_Mbps', ascending=False).round(2)
print(trends_df.to_string(index=False))

improving = len(trends_df[trends_df['Change_Mbps'] > 0])
degrading = len(trends_df[trends_df['Change_Mbps'] < 0])
print(f"\n  {improving} schools improving, {degrading} schools degrading")

### Anomaly & risk flags

In [ ]:
# =============================================================================
# ANOMALY & RISK FLAGS — Identify Schools with Unusual Patterns
# =============================================================================

anomalies = m.copy()
# Master schema varies by pull — guard columns the agg below expects.
for _col in ('connectivity_type_govt', 'connectivity_provider'):
    if _col not in anomalies.columns:
        anomalies[_col] = np.nan

# Calculate per-school metrics
school_metrics = anomalies.groupby('school_id_giga').agg({
    'download_speed': ['median', 'std', 'count', 'min', 'max'],
    'upload_speed': ['median', 'std'],
    'latency': ['median', 'std', 'min', 'max'],
    'loss_rate': 'median',
    'school_name': 'first',
    'connectivity_type_govt': 'first',
    'connectivity_provider': 'first',
    'admin1': 'first',
    'admin2': 'first'
}).reset_index()

school_metrics.columns = ['school_id_giga', 'dl_median', 'dl_std', 'n_measurements', 'dl_min', 'dl_max',
                          'ul_median', 'ul_std', 'lat_median', 'lat_std', 'lat_min', 'lat_max',
                          'loss_rate_median', 'school_name', 'connectivity', 'provider', 'admin1', 'admin2']

# Calculate coefficient of variation (CV) for speed variability
school_metrics['dl_cv'] = (school_metrics['dl_std'] / school_metrics['dl_median']).fillna(0)
school_metrics['ul_cv'] = (school_metrics['ul_std'] / school_metrics['ul_median']).fillna(0)
school_metrics['lat_cv'] = (school_metrics['lat_std'] / school_metrics['lat_median']).fillna(0)

# Define anomaly thresholds
HIGH_VARIABILITY_CV = 1.0  # CV > 1 = very unstable
EXTREME_VOLATILITY_CV = 2.0  # CV > 2 = extremely unstable
HIGH_LATENCY_STD = 200  # ms
POOR_PERFORMANCE_DL = 1  # Mbps (TIER 0)
INTERMITTENT_THRESHOLD = 50  # % coefficient of variation in speed range
HIGH_LOSS_THRESHOLD = 0.01  # >1% packet loss = flagged

# Flag anomalies
school_metrics['flag_extreme_variability'] = school_metrics['dl_cv'] > EXTREME_VOLATILITY_CV
school_metrics['flag_high_variability'] = (school_metrics['dl_cv'] > HIGH_VARIABILITY_CV) & (~school_metrics['flag_extreme_variability'])
school_metrics['flag_persistent_poor'] = school_metrics['dl_median'] < POOR_PERFORMANCE_DL
school_metrics['flag_high_latency_variability'] = school_metrics['lat_std'] > HIGH_LATENCY_STD
school_metrics['flag_intermittent'] = school_metrics['dl_max'] > (school_metrics['dl_min'] * 3)  # 3x difference suggests intermittent
school_metrics['flag_high_loss'] = school_metrics['loss_rate_median'] > HIGH_LOSS_THRESHOLD

# Count flagged schools
flagged = school_metrics[
    school_metrics['flag_extreme_variability'] | 
    school_metrics['flag_high_variability'] | 
    school_metrics['flag_persistent_poor'] | 
    school_metrics['flag_high_latency_variability'] | 
    school_metrics['flag_intermittent'] |
    school_metrics['flag_high_loss']
]

print(f"\n{'='*80}")
print("ANOMALY & RISK FLAGS")
print(f"{'='*80}")
print(f"\nTotal schools analyzed: {len(school_metrics)}")
print(f"Schools with anomalies: {len(flagged)} ({len(flagged)/len(school_metrics)*100:.1f}%)")

print(f"\n\nAnomaly Breakdown:")
print(f"  🔴 Extreme variability (CV > {EXTREME_VOLATILITY_CV}): {school_metrics['flag_extreme_variability'].sum()}")
print(f"  🟠 High variability (CV > {HIGH_VARIABILITY_CV}): {school_metrics['flag_high_variability'].sum()}")
print(f"  ⬇️  Persistent poor performance (<{POOR_PERFORMANCE_DL}M): {school_metrics['flag_persistent_poor'].sum()}")
print(f"  ⚡ High latency jitter (std>{HIGH_LATENCY_STD}ms): {school_metrics['flag_high_latency_variability'].sum()}")
print(f"  🔌 Intermittent access (max/min ratio >3x): {school_metrics['flag_intermittent'].sum()}")
print(f"  📉 High packet loss (>{HIGH_LOSS_THRESHOLD*100:.0f}%): {school_metrics['flag_high_loss'].sum()}")

# Show worst offenders
if len(flagged) > 0:
    print(f"\n\nTop Risk Schools (ordered by severity):")
    
    # Create risk score
    flagged['risk_score'] = (
        flagged['flag_extreme_variability'].astype(int) * 5 +
        flagged['flag_high_variability'].astype(int) * 3 +
        flagged['flag_persistent_poor'].astype(int) * 4 +
        flagged['flag_high_latency_variability'].astype(int) * 2 +
        flagged['flag_intermittent'].astype(int) * 3 +
        flagged['flag_high_loss'].astype(int) * 3
    )
    
    flagged_sorted = flagged.sort_values('risk_score', ascending=False).head(15)
    
    display_cols = ['school_name', 'connectivity', 'admin2', 'n_measurements', 
                    'dl_median', 'dl_cv', 'lat_std', 'loss_rate_median',
                    'flag_extreme_variability', 'flag_persistent_poor',
                    'flag_intermittent', 'flag_high_loss', 'risk_score']
    print(flagged_sorted[display_cols].to_string(index=False))

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Speed variability distribution
valid_cv = school_metrics[np.isfinite(school_metrics['dl_cv'])]['dl_cv']
axes[0, 0].hist(valid_cv, bins=50, color='#989898', alpha=0.7, edgecolor='black')
axes[0, 0].axvline(HIGH_VARIABILITY_CV, color='#ffc93d', linestyle='--', linewidth=2, label='High threshold')
axes[0, 0].axvline(EXTREME_VOLATILITY_CV, color='#525252', linestyle='--', linewidth=2, label='Extreme threshold')
axes[0, 0].set_xlabel('Coefficient of Variation (Speed)', fontweight='bold')
axes[0, 0].set_ylabel('Number of Schools', fontweight='bold')
axes[0, 0].set_title('Speed Variability Distribution', fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# Latency jitter distribution
valid_lat_std = school_metrics[np.isfinite(school_metrics['lat_std'])]['lat_std']
axes[0, 1].hist(valid_lat_std, bins=50, color='#7eb0ff', alpha=0.7, edgecolor='black')
axes[0, 1].axvline(HIGH_LATENCY_STD, color='#525252', linestyle='--', linewidth=2, label=f'Risk threshold ({HIGH_LATENCY_STD}ms)')
axes[0, 1].set_xlabel('Latency Standard Deviation (ms)', fontweight='bold')
axes[0, 1].set_ylabel('Number of Schools', fontweight='bold')
axes[0, 1].set_title('Latency Jitter Distribution', fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# Anomaly type counts
anomaly_counts = pd.Series({
    'Extreme\nVariability': school_metrics['flag_extreme_variability'].sum(),
    'High\nVariability': school_metrics['flag_high_variability'].sum(),
    'Poor\nPerformance': school_metrics['flag_persistent_poor'].sum(),
    'Latency\nJitter': school_metrics['flag_high_latency_variability'].sum(),
    'Intermittent\nAccess': school_metrics['flag_intermittent'].sum(),
    'High\nLoss': school_metrics['flag_high_loss'].sum()
})
anomaly_counts.plot(kind='bar', ax=axes[1, 0], color=['#525252', '#989898', '#002d9c', '#7eb0ff', '#0050e6', '#0050e6'],
                     edgecolor='black', alpha=0.8)
axes[1, 0].set_ylabel('Number of Schools', fontweight='bold')
axes[1, 0].set_title('Anomaly Type Breakdown', fontweight='bold')
axes[1, 0].grid(axis='y', alpha=0.3)
axes[1, 0].set_xticklabels(axes[1, 0].get_xticklabels(), rotation=45, ha='right')

# Scatter: Median speed vs Variability
colors = ['red' if x else 'gray' for x in school_metrics['flag_extreme_variability']]
axes[1, 1].scatter(school_metrics['dl_median'], school_metrics['dl_cv'], 
                   c=colors, s=50, alpha=0.6, edgecolor='black')
axes[1, 1].axhline(HIGH_VARIABILITY_CV, color='#ffc93d', linestyle='--', linewidth=1, alpha=0.5)
axes[1, 1].axhline(EXTREME_VOLATILITY_CV, color='#525252', linestyle='--', linewidth=1, alpha=0.5)
axes[1, 1].set_xlabel('Median Download Speed (Mbps)', fontweight='bold')
axes[1, 1].set_ylabel('Coefficient of Variation', fontweight='bold')
axes[1, 1].set_title('Speed vs Stability (Red = Extreme Variability)', fontweight='bold')
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# FOREIGN / ROAMING ISP FLAGS — measurements egressing via out-of-country ISPs
# =============================================================================
# Roaming SIM hotspots tunnel traffic to the SIM's home carrier, so the detected
# ISP is the home network — e.g. Uzbek carriers (Uzbektelekom/UNITEL/COSCOM) at
# two Fiji schools via an Uzbek roaming hotspot (found Aug 2026). Heuristic:
# ISPs serving <= RARE_ISP_MAX_SCHOOLS schools nationally AND a minority share
# at the school. Review the list — rare legitimate local ISPs can appear too.
RARE_ISP_MAX_SCHOOLS = 3

_isp_schools = m.groupby('isp_mapped')['school_id_giga'].nunique()
_rare = set(_isp_schools[_isp_schools <= RARE_ISP_MAX_SCHOOLS].index)
_school_share = (m.groupby(['school_id_giga', 'isp_mapped']).size()
                 / m.groupby('school_id_giga').size()).rename('share')
flag = (m.groupby(['school_id_giga', 'school_name', 'isp_mapped'])
          .agg(n=('isp_mapped', 'size'), lat_med=('latency', 'median'),
               first=('date', 'min'), last=('date', 'max')).reset_index()
          .merge(_school_share.reset_index(), on=['school_id_giga', 'isp_mapped']))
flag = flag[flag['isp_mapped'].isin(_rare) & (flag['share'] < 0.5)]
flag['school_lat_med'] = flag['school_id_giga'].map(m.groupby('school_id_giga')['latency'].median())
flag = flag.sort_values('n', ascending=False)

if len(flag):
    print(f"⚠️ Possible roaming/foreign or ad-hoc ISPs: {len(flag)} school×ISP combos, "
          f"{int(flag['n'].sum()):,} measurements — verify before trusting these schools' baselines")
    display(flag[['school_name', 'isp_mapped', 'n', 'share', 'lat_med', 'school_lat_med', 'first', 'last']]
            .round({'share': 2, 'lat_med': 0, 'school_lat_med': 0}).head(20))
else:
    print("✓ No rare-ISP anomalies flagged")

---
# Part C — Outputs

In [ ]:
# =============================================================================
# EXPORT IQB-EDU PER-SCHOOL VERDICTS
# =============================================================================
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
_verdict_path = Path(OUTPUT_DIR) / f"{COUNTRY_ISO3}_iqb_school_verdicts.csv"
school_iqb.to_csv(_verdict_path, index=False)
print(f"\u2713 Per-school IQB-Edu verdicts \u2192 {_verdict_path}")
display(school_iqb.head())


In [ ]:
# =============================================================================
# SAVE ENRICHED DATASET
# =============================================================================
# Saves m with all derived columns (ISP mapping, loss_rate, WiFi unpacked,
# admin metadata, time windows) for use in downstream notebooks.
# File: {COUNTRY_ISO3}_measurements_enriched.parquet in CACHE_DIR

_enriched_path = Path(CACHE_DIR) / f"{COUNTRY_ISO3.lower()}_measurements_enriched.parquet"
os.makedirs(CACHE_DIR, exist_ok=True)
m.to_parquet(_enriched_path, index=False)
print(f"Saved enriched dataset: {_enriched_path}")
print(f"  Shape: {m.shape[0]:,} rows x {m.shape[1]} columns")


In [ ]:
# =============================================================================
# COLUMN DICTIONARY - m DataFrame
# =============================================================================
# Origin:
#   Physical = present in the consolidated physical table
#              default.all_gigameter_measurement_data (pre-extracted by the pipeline)
#   Alias    = physical column re-exposed under a legacy name in the load cell
#   Master   = merged from the school master register (merge-master cell)
#   Derived  = computed in this notebook
# =============================================================================

_col_dict = [
    # -- Core identifiers (Physical) ------------------------------------------
    ("measurement_id",             "Physical", "Unique measurement record ID"),
    ("measurement_uuid",           "Physical", "M-Lab test UUID - links to NDT / BigQuery records"),
    ("school_id_giga",             "Physical", "Giga canonical school identifier"),
    ("school_id_govt",             "Physical", "Government school identifier"),
    ("school_name",                "Physical", "School display name"),
    ("iso3_code",                  "Physical", "ISO3 country code"),
    ("country",                    "Physical", "Country name"),

    # -- Timestamps -----------------------------------------------------------
    ("date",                       "Physical", "Measurement date (UTC, date only)"),
    ("created_timestamp",          "Physical", "Measurement creation timestamp (UTC)"),
    ("local_created_timestamp",    "Physical", "created_timestamp in school local timezone (pipeline)"),
    ("local_hour_of_measurement",  "Physical", "Hour of day in local time"),
    ("local_day_of_week",          "Physical", "Day name in local time"),
    ("is_weekday",                 "Physical", "True if Mon-Fri (local)"),
    ("timestamp",                  "Alias",    "UTC timestamp; aliased from created_timestamp (load cell)"),
    ("timestamplocal",             "Derived",  "timestamp converted to TIMEZONE, tz-aware (load cell)"),

    # -- Speed / quality metrics ----------------------------------------------
    ("download_speed",             "Physical", "Download throughput (Mbps)"),
    ("upload_speed",               "Physical", "Upload throughput (Mbps)"),
    ("latency",                    "Physical", "Round-trip latency (ms)"),
    ("s2c_bytes_retrans",          "Physical", "NDT S2C bytes retransmitted (server->client)"),
    ("s2c_bytes_sent",             "Physical", "NDT S2C bytes sent"),
    ("packet_loss_rate",           "Physical", "s2c_bytes_retrans / s2c_bytes_sent (pipeline)"),
    ("loss_rate",                  "Derived",  "Numeric copy of packet_loss_rate (field-prep cell)"),
    ("pass_fail_overall",          "Physical", "Measurement-validity flag (data-quality, NOT a use-case verdict)"),
    ("reasons_failed_overall",     "Physical", "Reasons a measurement failed validity"),

    # -- Data volume (Physical) -----------------------------------------------
    ("data_downloaded_gb",         "Physical", "Data downloaded in test (GB)"),
    ("data_uploaded_gb",           "Physical", "Data uploaded in test (GB)"),
    ("data_usage_gb",              "Physical", "Total data used in test (GB)"),
    ("avg_download_speed",         "Physical", "Per-school average download speed (pipeline aggregate)"),
    ("avg_upload_speed",           "Physical", "Per-school average upload speed (pipeline aggregate)"),
    ("avg_latency",                "Physical", "Per-school average latency (pipeline aggregate)"),
    ("total_data_usage_gb",        "Physical", "Per-school total data used (pipeline aggregate)"),

    # -- WiFi (Physical - pre-extracted by pipeline) --------------------------
    ("detected_wifi_ssid",         "Physical", "Primary WiFi network name"),
    ("detected_wifi_model",        "Physical", "WiFi adapter model"),
    ("detected_wifi_quality",      "Physical", "WiFi link quality score 0-100"),
    ("detected_wifi_signal",       "Physical", "WiFi signal level (dBm)"),
    ("detected_wifi_tx_rate",      "Physical", "WiFi transmit rate (Mbps)"),
    ("detected_wifi_channel",      "Physical", "WiFi channel number"),
    ("detected_wifi_frequency",    "Physical", "WiFi frequency (MHz)"),

    # -- Server / ISP ---------------------------------------------------------
    ("detected_server",            "Physical", "M-Lab server city (pipeline: server_info.City)"),
    ("isp_name",                   "Physical", "Detected ISP name"),
    ("isp_asn",                    "Physical", "Detected ISP ASN"),
    ("detected_isp",               "Alias",    "Aliased from isp_name (load cell)"),
    ("detected_isp_asn",           "Alias",    "Aliased from isp_asn (load cell)"),
    ("isp_mapped",                 "Derived",  "Canonical ISP name via isp_mappings.json (ISP mapping cell)"),
    ("school_isp",                 "Derived",  "Dominant isp_mapped per school (ISP mapping cell)"),

    # -- Device / client (Physical) -------------------------------------------
    ("device_id",                  "Physical", "Device identifier"),
    ("browser_id",                 "Physical", "Browser identifier"),
    ("ip_address",                 "Physical", "Client IP address at time of test"),
    ("app_version",                "Physical", "GigaMeter app version"),
    ("notes",                      "Physical", "Measurement type label (e.g. daily, manual)"),
    ("installed_path",             "Physical", "App install path on device"),
    ("windows_username",           "Physical", "Windows username on device (if applicable)"),
    ("rt_source",                  "Physical", "Data source: GigaMeter / DailyCheckApp / Mlab"),

    # -- Geography / school metadata (Physical - pre-joined) ------------------
    ("admin1",                     "Physical", "Province / region"),
    ("admin2",                     "Physical", "District"),
    ("connectivity",               "Physical", "Connectivity flag"),
    ("connectivity_type_govt",     "Physical", "Government-reported connectivity type"),
    ("cellular_coverage_type",     "Physical", "Cellular coverage type"),
    ("education_level",            "Physical", "Government education level (physical table)"),
    ("electricity_availability",   "Physical", "Electricity availability"),
    ("fiber_node_distance",        "Physical", "Distance to nearest fiber node"),
    ("school_area_type",           "Physical", "Urban / rural classification"),
    ("school_funding_type",        "Physical", "School funding type"),
    ("latitude",                   "Physical", "School latitude (decimal degrees)"),
    ("longitude",                  "Physical", "School longitude (decimal degrees)"),

    # -- Master register merge (columns not already in the physical table) ----
    ("education_level", "Master",   "Normalised education level (merge-master cell)"),
    ("education_level_govt",       "Master",   "Government education level from master (merge-master cell)"),
    ("connectivity_provider",      "Master",   "Government-reported provider (merge-master cell)"),

    # -- Analysis-derived time columns ----------------------------------------
    ("measurement_date",           "Derived",  "Date of timestamplocal (preprocessing cell)"),
    ("measurement_weekday",        "Derived",  "Weekday integer 0=Mon (preprocessing cell)"),
    ("measurement_time_window",    "Derived",  "school_hours / off_hours; overwrites the pipeline window (preprocessing cell)"),
    ("measurement_hour",           "Derived",  "Hour of day 0-23 from timestamplocal (distributions cell)"),
    ("measurement_dayofweek",      "Derived",  "Day name e.g. Monday (distributions cell)"),
    ("measurement_date_only",      "Derived",  "Date object from timestamplocal (distributions cell)"),
]

import pandas as pd
col_df = pd.DataFrame(_col_dict, columns=["Column", "Origin", "Definition"])

_colors = {
    "Physical": "",
    "Alias":    "background-color: #eaf2ff",
    "Master":   "background-color: #d6ffe9",
    "Derived":  "background-color: #f4f4f4",
}

def _row_style(row):
    bg = _colors.get(row["Origin"], "")
    return [bg] * len(row)

counts = col_df["Origin"].value_counts()
caption = (f"m column dictionary - {len(col_df)} columns  |  "
           + "  ".join(f"{o}: {counts.get(o,0)}" for o in ["Physical", "Alias", "Master", "Derived"]))

display(
    col_df.style
    .apply(_row_style, axis=1)
    .set_properties(**{"text-align": "left", "font-size": "12px", "padding": "4px 10px"})
    .set_properties(subset=["Column"], **{"font-family": "monospace", "font-weight": "bold"})
    .set_properties(subset=["Origin"], **{"text-align": "center"})
    .set_table_styles([
        {"selector": "th", "props": [("background-color", "#161616"), ("color", "white"),
                                      ("font-size", "12px"), ("padding", "6px 10px"), ("text-align", "left")]},
    ])
    .hide(axis="index")
    .set_caption(caption)
)


### Export the report (HTML / PDF)

Run **Kernel → Restart & Run All**, **Save**, then run this cell. It renders a self-contained,
code-hidden report you can send to other analysts. PDF is optional (needs a one-time
`pip install playwright && playwright install chromium`).

In [ ]:
# =============================================================================
# EXPORT SHAREABLE REPORT (self-contained HTML; optional PDF)
# =============================================================================
import os, subprocess
os.makedirs(OUTPUT_DIR, exist_ok=True)

NB_PATH = str(Path.home() / "Documents/Giga/Delivery/Monitoring/Analytics/meter_explorer_02.ipynb")
html_out = Path(OUTPUT_DIR) / f"{COUNTRY_ISO3}_eda_explorer.html"

# Self-contained HTML, code cells hidden -> a clean report (embeds all charts/CSS in one file).
# NB: save the notebook (with outputs) first so the export captures the latest run.
_cmd = ["jupyter", "nbconvert", "--to", "html", "--no-input", "--embed-images",
        "--output", html_out.stem, "--output-dir", str(html_out.parent), NB_PATH]
_res = subprocess.run(_cmd, capture_output=True, text=True)
if _res.returncode == 0:
    print(f"\u2713 Shareable HTML \u2192 {html_out}")
else:
    print("\u26a0\ufe0f HTML export failed:\n", _res.stderr[-800:])

# PDF (optional) - uncomment after: pip install playwright && playwright install chromium
# pdf_cmd = ["jupyter", "nbconvert", "--to", "webpdf", "--no-input", "--allow-chromium-download",
#            "--output", html_out.stem, "--output-dir", str(html_out.parent), NB_PATH]
# subprocess.run(pdf_cmd)
print("\nTo also keep the full version (with code + appendix), re-run without --no-input.")
